# Full-GPU UAV training

This notebook is the full-GPU counterpart of `test.ipynb`.

The batched environment state, reset/step logic, sensing, belief map, communication, energy, reward, observations, replay buffer, actor/critic inference, and gradient updates are designed to remain on the CUDA device. `test.ipynb` remains the CPU-reference implementation used for semantic/parity checking.

Run the cells top-to-bottom, then call `benchmark_env(...)` or `train_full_gpu(...)` explicitly.

In [ ]:
"""End-to-end CUDA benchmark/training path for the UAV search project.

The environment state, reset, step, sensing, belief, communication, energy,
observations, replay buffer, actor/critic inference and gradient updates live
on one CUDA device. Python only orchestrates kernel launches and logging.

This is intentionally separate from test.ipynb so it can be benchmarked
against the existing CPU-env + CUDA-learner path before replacing it.
"""

from __future__ import annotations

import ast
import random
import sys
import types
import json
import math
import time
from pathlib import Path

import torch
import torch.nn.functional as F

from config import CONFIG


def _require_cuda(device: torch.device) -> None:
    if device.type != "cuda" or not torch.cuda.is_available():
        raise RuntimeError("test_gpu.ipynb requires a CUDA device for full-GPU mode")


## Full-GPU batched environment

In [ ]:
class FullGpuUAVBatchEnv:
    """Fixed-shape batched Simple backend implemented with CUDA tensors."""

    observation_dim = 333
    state_dim = 2875
    continuous_dim = 4

    def __init__(self, num_envs: int, device: str = "cuda:0", seed: int = 44, strict_cuda: bool = True):
        self.device = torch.device(device)
        if strict_cuda:
            _require_cuda(self.device)
        self.num_envs = int(num_envs)
        self.num_uavs = int(CONFIG["num_uavs"])
        self.num_targets = int(CONFIG["num_targets"])
        self.num_obstacles = int(CONFIG["num_obstacles"])
        self.discrete_dim = self.num_uavs + 1
        self.grid_n = math.ceil(float(CONFIG["map_size"]) / float(CONFIG["grid_cell_m"]))
        nearest_obstacles = int(CONFIG["observation_nearest_obstacles"])
        patch_cells = int(CONFIG["belief_patch_cells"])
        coarse_cells = int(CONFIG["belief_coarse_cells"])
        critic_cells = int(CONFIG["critic_belief_grid_cells"])
        buffer_slots = int(
            float(CONFIG["buffer_bytes"])
            // float(CONFIG["report_bytes"])
        )
        pending_slots = int(
            float(CONFIG["pending_buffer_bytes"])
            // float(CONFIG["report_bytes"])
        )
        self.observation_dim = (
            9
            + 3
            + 5 * (self.num_uavs - 1)
            + 5 * nearest_obstacles
            + patch_cells * patch_cells
            + coarse_cells * coarse_cells
            + 5
            + self.discrete_dim
        )
        per_uav_state_dim = (
            self.observation_dim
            + 4 * buffer_slots
            + 4 * pending_slots
            + 2
            + critic_cells * critic_cells
        )
        self.state_dim = (
            self.num_uavs * per_uav_state_dim
            + 4 * self.num_targets
            + 4 * self.num_obstacles
            + 1
        )
        self.seed = int(seed)
        self.generator = torch.Generator(device=self.device)
        self.generator.manual_seed(self.seed)

        self.map_size = float(CONFIG["map_size"])
        self.altitude_min = float(CONFIG["altitude_min"])
        self.altitude_max = float(CONFIG["altitude_max"])
        self.dt = float(CONFIG["dt"])
        self.max_speed = float(CONFIG["max_speed"])
        self.max_accel = float(CONFIG["max_accel"])
        self.cell_size = float(CONFIG["grid_cell_m"])
        self.max_steps = int(CONFIG["max_steps"])
        self.gcs = torch.tensor(CONFIG["gcs_position"], device=self.device, dtype=torch.float32)
        # Optional external mission-network adapter. The default Simple backend
        # remains fully tensorized; UavNetSim is attached explicitly by the
        # benchmark/training helpers because UavNetSim itself is CPU/SimPy.
        self.external_network = None
        self.network_model = str(
            CONFIG.get("full_gpu_network_model", "simple")
        ).strip().lower()
        if self.network_model not in {"simple", "packet_approx", "uavnetsim_gpu"}:
            raise ValueError(
                "full_gpu_network_model must be simple, packet_approx, or uavnetsim_gpu"
            )

        # At altitude_max with 90-degree full FOV, radius <= altitude.
        max_radius_cells = math.ceil(self.altitude_max / self.cell_size) + 1
        offsets = torch.arange(-max_radius_cells, max_radius_cells + 1, device=self.device)
        oy, ox = torch.meshgrid(offsets, offsets, indexing="ij")
        self.cell_offsets = torch.stack((ox.reshape(-1), oy.reshape(-1)), dim=-1).long()

        # Communication choice table: 0=silent, last=GCS, middle entries are peers.
        choices = torch.full(
            (self.num_uavs, self.discrete_dim),
            -2,
            device=self.device,
            dtype=torch.long,
        )
        for sender in range(self.num_uavs):
            peers = [p for p in range(self.num_uavs) if p != sender]
            choices[sender, 1 : self.num_uavs] = torch.tensor(
                peers, device=self.device, dtype=torch.long
            )
            choices[sender, self.num_uavs] = -1
        self.choices = choices

        # Immutable tensors reused by every observation/network step.
        self.xyz_scale = torch.tensor(
            [
                self.map_size,
                self.map_size,
                max(self.altitude_max, 1.0),
            ],
            device=self.device,
            dtype=torch.float32,
        )
        peer_rows = [
            [peer for peer in range(self.num_uavs) if peer != sender]
            for sender in range(self.num_uavs)
        ]
        self.peer_index = torch.tensor(
            peer_rows,
            device=self.device,
            dtype=torch.long,
        )
        patch_n = int(CONFIG["belief_patch_cells"])
        patch_radius = patch_n // 2
        patch_axis = torch.arange(
            -patch_radius,
            patch_radius + 1,
            device=self.device,
            dtype=torch.long,
        )
        patch_y, patch_x = torch.meshgrid(
            patch_axis,
            patch_axis,
            indexing="ij",
        )
        self.belief_patch_offsets = torch.stack(
            (
                patch_x.reshape(-1),
                patch_y.reshape(-1),
            ),
            dim=-1,
        )
        self.target_ids = torch.arange(
            self.num_targets,
            device=self.device,
            dtype=torch.long,
        ).view(1, self.num_targets)

        self.reset()

    def attach_external_network(self, adapter):
        if self.num_envs != 1:
            raise ValueError(
                "external packet-level networking currently requires num_envs=1"
            )
        if self.external_network is not None:
            self.external_network.close()
        self.external_network = adapter
        self.external_network.reset_from_env()
        return self

    def detach_external_network(self):
        if self.external_network is not None:
            self.external_network.close()
        self.external_network = None

    def close(self):
        self.detach_external_network()

    def _rand(self, *shape):
        return torch.rand(*shape, device=self.device, generator=self.generator)


    def _sample_world(self):
        E, U, T, O = (
            self.num_envs,
            self.num_uavs,
            self.num_targets,
            self.num_obstacles,
        )
        pool = max(
            64,
            int(CONFIG.get("full_gpu_world_candidates", 256))
            * int(CONFIG.get("full_gpu_world_sampling_rounds", 4)),
        )

        # GPU rejection sampling that mirrors create_uav(): random positions
        # inside the launch disk, clipped only by map bounds and minimum
        # pairwise launch spacing.
        launch_radius = float(CONFIG["launch_radius_m"])
        launch_spacing = float(CONFIG["launch_min_spacing_m"])
        positions = self.gcs.view(1, 1, 3).repeat(E, U, 1)
        positions[..., 2] = self.altitude_min
        for uav_index in range(U):
            offsets = (self._rand(E, pool, 2) * 2.0 - 1.0) * launch_radius
            candidates = self.gcs[:2].view(1, 1, 2) + offsets
            base_valid = (
                (offsets.square().sum(-1) <= launch_radius * launch_radius)
                & (candidates[..., 0] >= 0.0)
                & (candidates[..., 0] <= self.map_size)
                & (candidates[..., 1] >= 0.0)
                & (candidates[..., 1] <= self.map_size)
            )
            if uav_index > 0:
                distances = torch.linalg.vector_norm(
                    candidates[:, :, None, :]
                    - positions[:, None, :uav_index, :2],
                    dim=-1,
                )
                spacing_score = distances.amin(-1)
                valid = base_valid & (spacing_score >= launch_spacing)
            else:
                spacing_score = torch.full(
                    (E, pool),
                    launch_spacing,
                    device=self.device,
                )
                valid = base_valid

            first_valid = valid.float().argmax(-1)
            # The fallback is still fully GPU-side and chooses the
            # best-separated in-disk candidate. With the project geometry the
            # configured pool has abundant valid samples; this only prevents a
            # silent all-false argmax from selecting an arbitrary invalid point.
            fallback_score = torch.where(
                base_valid,
                spacing_score,
                torch.full_like(spacing_score, -1.0),
            )
            fallback = fallback_score.argmax(-1)
            chosen_index = torch.where(valid.any(-1), first_valid, fallback)
            chosen = torch.gather(
                candidates,
                1,
                chosen_index[:, None, None].expand(-1, 1, 2),
            ).squeeze(1)
            positions[:, uav_index, :2] = chosen

        velocities = torch.zeros(E, U, 3, device=self.device)
        battery = torch.full(
            (E, U),
            float(CONFIG["battery_j"]),
            device=self.device,
        )
        active = torch.ones(E, U, device=self.device, dtype=torch.bool)

        rmin = float(CONFIG["obstacle_radius_min_m"])
        rmax = float(CONFIG["obstacle_radius_max_m"])
        hmin = float(CONFIG["obstacle_height_min_m"])
        hmax = float(CONFIG["obstacle_height_max_m"])
        obstacle_radius = rmin + (rmax - rmin) * self._rand(E, O)
        obstacle_height = hmin + (hmax - hmin) * self._rand(E, O)
        obstacle_xy = torch.zeros(E, O, 2, device=self.device)
        gcs_exclusion = float(CONFIG["gcs_exclusion_radius_m"])
        launch_margin = float(CONFIG["safety_distance"])

        for obstacle_index in range(O):
            radius = obstacle_radius[:, obstacle_index]
            candidates = (
                radius[:, None, None]
                + self._rand(E, pool, 2)
                * (self.map_size - 2.0 * radius).clamp_min(1.0)[:, None, None]
            )
            gcs_distance = torch.linalg.vector_norm(
                candidates - self.gcs[:2].view(1, 1, 2),
                dim=-1,
            )
            launch_distance = torch.linalg.vector_norm(
                candidates[:, :, None, :]
                - positions[:, None, :, :2],
                dim=-1,
            )
            valid = (
                (gcs_distance > (gcs_exclusion + radius)[:, None])
                & (
                    launch_distance
                    > (radius[:, None, None] + launch_margin)
                ).all(-1)
            )
            margin_score = torch.minimum(
                gcs_distance - (gcs_exclusion + radius)[:, None],
                (
                    launch_distance
                    - (radius[:, None, None] + launch_margin)
                ).amin(-1),
            )
            if obstacle_index > 0:
                previous_distance = torch.linalg.vector_norm(
                    candidates[:, :, None, :]
                    - obstacle_xy[:, None, :obstacle_index, :],
                    dim=-1,
                )
                previous_margin = (
                    previous_distance
                    - radius[:, None, None]
                    - obstacle_radius[:, None, :obstacle_index]
                )
                valid &= (previous_margin > 0.0).all(-1)
                margin_score = torch.minimum(
                    margin_score,
                    previous_margin.amin(-1),
                )

            first_valid = valid.float().argmax(-1)
            fallback = margin_score.argmax(-1)
            chosen_index = torch.where(valid.any(-1), first_valid, fallback)
            obstacle_xy[:, obstacle_index] = torch.gather(
                candidates,
                1,
                chosen_index[:, None, None].expand(-1, 1, 2),
            ).squeeze(1)

        target_xy = torch.zeros(E, T, 2, device=self.device)
        target_exclusion = float(CONFIG["target_exclusion_radius_m"])
        for target_index in range(T):
            candidates = self._rand(E, pool, 2) * self.map_size
            gcs_distance = torch.linalg.vector_norm(
                candidates - self.gcs[:2].view(1, 1, 2),
                dim=-1,
            )
            obstacle_distance = torch.linalg.vector_norm(
                candidates[:, :, None, :] - obstacle_xy[:, None, :, :],
                dim=-1,
            )
            valid = (
                (gcs_distance > target_exclusion)
                & (
                    obstacle_distance
                    > obstacle_radius[:, None, :]
                ).all(-1)
            )
            margin_score = torch.minimum(
                gcs_distance - target_exclusion,
                (
                    obstacle_distance - obstacle_radius[:, None, :]
                ).amin(-1),
            )
            candidate_cells = torch.floor(candidates / self.cell_size).long()
            if target_index > 0:
                previous_cells = torch.floor(
                    target_xy[:, :target_index] / self.cell_size
                ).long()
                duplicate_cell = (
                    candidate_cells[:, :, None, :]
                    == previous_cells[:, None, :, :]
                ).all(-1).any(-1)
                valid &= ~duplicate_cell
                margin_score = torch.where(
                    duplicate_cell,
                    torch.full_like(margin_score, -1.0),
                    margin_score,
                )

            first_valid = valid.float().argmax(-1)
            fallback = margin_score.argmax(-1)
            chosen_index = torch.where(valid.any(-1), first_valid, fallback)
            target_xy[:, target_index] = torch.gather(
                candidates,
                1,
                chosen_index[:, None, None].expand(-1, 1, 2),
            ).squeeze(1)

        return (
            positions,
            velocities,
            battery,
            active,
            target_xy,
            obstacle_xy,
            obstacle_radius,
            obstacle_height,
        )

    def reset(self, mask: torch.Tensor | None = None):
        sampled = self._sample_world()
        E, U, T = self.num_envs, self.num_uavs, self.num_targets

        if mask is None:
            self.positions, self.velocities, self.battery, self.active = sampled[:4]
            self.target_xy, self.obstacle_xy, self.obstacle_radius, self.obstacle_height = sampled[4:]
            self.confirmed = torch.zeros(E, T, device=self.device, dtype=torch.bool)
            self.delivered = torch.zeros(E, T, device=self.device, dtype=torch.bool)
            self.belief = torch.full(
                (E, U, self.grid_n, self.grid_n),
                float(CONFIG["belief_prior"]),
                device=self.device,
            )
            self.report_valid = torch.zeros(E, U, T, device=self.device, dtype=torch.bool)
            self.report_remaining = torch.zeros(E, U, T, device=self.device)
            self.report_created = torch.full(
                (E, U, T), self.max_steps + int(CONFIG["report_ttl"]) + 1,
                device=self.device, dtype=torch.long
            )
            self.report_source = torch.full(
                (E, U, T), -1, device=self.device, dtype=torch.long
            )
            self.peer_received = torch.zeros(
                E, U, U, T, device=self.device
            )
            self.peer_source = torch.full(
                (E, U, U, T), -1, device=self.device, dtype=torch.long
            )
            self.peer_created = torch.full(
                (E, U, U, T),
                self.max_steps + int(CONFIG["report_ttl"]) + 1,
                device=self.device,
                dtype=torch.long,
            )
            self.pending_valid = torch.zeros(E, U, T, device=self.device, dtype=torch.bool)
            self.pending_remaining = torch.zeros(E, U, T, device=self.device)
            self.pending_created = torch.full_like(
                self.report_created,
                self.max_steps + int(CONFIG["report_ttl"]) + 1,
            )
            self.step_count = torch.zeros(E, device=self.device, dtype=torch.long)
        else:
            mask = mask.to(device=self.device, dtype=torch.bool)
            if mask.ndim != 1 or mask.shape[0] != E:
                raise ValueError("reset mask must have shape (num_envs,)")
            m3 = mask[:, None, None]
            self.positions = torch.where(m3, sampled[0], self.positions)
            self.velocities = torch.where(m3, sampled[1], self.velocities)
            self.battery = torch.where(mask[:, None], sampled[2], self.battery)
            self.active = torch.where(mask[:, None], sampled[3], self.active)
            self.target_xy = torch.where(m3[:, :1].expand(-1, T, 2), sampled[4], self.target_xy)
            mo2 = mask[:, None, None]
            self.obstacle_xy = torch.where(mo2, sampled[5], self.obstacle_xy)
            self.obstacle_radius = torch.where(mask[:, None], sampled[6], self.obstacle_radius)
            self.obstacle_height = torch.where(mask[:, None], sampled[7], self.obstacle_height)
            self.confirmed = torch.where(mask[:, None], torch.zeros_like(self.confirmed), self.confirmed)
            self.delivered = torch.where(mask[:, None], torch.zeros_like(self.delivered), self.delivered)
            self.belief = torch.where(
                mask[:, None, None, None],
                torch.full_like(self.belief, float(CONFIG["belief_prior"])),
                self.belief,
            )
            self.report_valid = torch.where(
                mask[:, None, None], torch.zeros_like(self.report_valid), self.report_valid
            )
            self.report_remaining = torch.where(
                mask[:, None, None], torch.zeros_like(self.report_remaining), self.report_remaining
            )
            self.report_created = torch.where(
                mask[:, None, None],
                torch.full_like(self.report_created, self.max_steps + int(CONFIG["report_ttl"]) + 1),
                self.report_created,
            )
            self.report_source = torch.where(
                mask[:, None, None],
                torch.full_like(self.report_source, -1),
                self.report_source,
            )
            self.peer_received = torch.where(
                mask[:, None, None, None],
                torch.zeros_like(self.peer_received),
                self.peer_received,
            )
            self.peer_source = torch.where(
                mask[:, None, None, None],
                torch.full_like(self.peer_source, -1),
                self.peer_source,
            )
            self.peer_created = torch.where(
                mask[:, None, None, None],
                torch.full_like(
                    self.peer_created,
                    self.max_steps + int(CONFIG["report_ttl"]) + 1,
                ),
                self.peer_created,
            )
            self.pending_valid = torch.where(
                mask[:, None, None], torch.zeros_like(self.pending_valid), self.pending_valid
            )
            self.pending_remaining = torch.where(
                mask[:, None, None], torch.zeros_like(self.pending_remaining), self.pending_remaining
            )
            self.pending_created = torch.where(
                mask[:, None, None],
                torch.full_like(self.pending_created, self.max_steps + int(CONFIG["report_ttl"]) + 1),
                self.pending_created,
            )
            self.step_count = torch.where(mask, torch.zeros_like(self.step_count), self.step_count)

        entropy_map = self._belief_entropy()
        self.belief_potential = self._belief_potential(
            entropy_map
        )
        if self.external_network is not None:
            self.external_network.reset_from_env()
        obs, mask_out = self._observation(
            entropy_map=entropy_map
        )
        state = self._state(
            obs,
            entropy_map=entropy_map,
        )
        return obs, state, mask_out

    def _belief_entropy(self):
        p = self.belief.clamp(1e-6, 1.0 - 1e-6)
        return (
            -p * torch.log2(p)
            - (1.0 - p) * torch.log2(1.0 - p)
        )

    def _belief_potential(self, entropy_map=None):
        if entropy_map is None:
            entropy_map = self._belief_entropy()
        return (
            1.0
            - entropy_map.mean(dim=(1, 2, 3))
        ).clamp(0.0, 1.0)

    @staticmethod
    def _project_unit_ball(x):
        x = x.clamp(-1.0, 1.0)
        norm = torch.linalg.vector_norm(x, dim=-1, keepdim=True).clamp_min(1.0)
        return x / norm

    def _segment_cylinder_blocked(
        self,
        start,
        end,
        obstacle_xy=None,
        obstacle_radius=None,
        obstacle_height=None,
    ):
        """Exact vectorized counterpart of CPU segment_intersects_obstacle().

        start/end have shape [E, ..., 3]. Obstacles are the environment's
        vertical finite cylinders [E, O]. The returned tensor has shape
        [E, ...] and is True when the segment intersects any cylinder.
        """
        if obstacle_xy is None:
            obstacle_xy = self.obstacle_xy
        if obstacle_radius is None:
            obstacle_radius = self.obstacle_radius
        if obstacle_height is None:
            obstacle_height = self.obstacle_height

        start = start.to(dtype=self.positions.dtype)
        end = end.to(dtype=self.positions.dtype)
        extra = start.ndim - 2
        E = start.shape[0]
        O = obstacle_xy.shape[1]
        obs_xy = obstacle_xy.reshape(
            E, *([1] * extra), O, 2
        )
        obs_r = obstacle_radius.reshape(
            E, *([1] * extra), O
        )
        obs_h = obstacle_height.reshape(
            E, *([1] * extra), O
        )
        s = start.unsqueeze(-2)
        e = end.unsqueeze(-2)
        d = e - s
        dxy = d[..., :2]
        dz = d[..., 2]
        rel_xy = s[..., :2] - obs_xy
        eps = torch.finfo(start.dtype).eps

        dz_small = dz.abs() <= eps
        horizontal_z_ok = (s[..., 2] >= 0.0) & (s[..., 2] <= obs_h)
        safe_dz = torch.where(dz_small, torch.ones_like(dz), dz)
        t_ground = (0.0 - s[..., 2]) / safe_dz
        t_top = (obs_h - s[..., 2]) / safe_dz
        z_enter = torch.maximum(
            torch.zeros_like(t_ground),
            torch.minimum(t_ground, t_top),
        )
        z_exit = torch.minimum(
            torch.ones_like(t_ground),
            torch.maximum(t_ground, t_top),
        )
        z_enter = torch.where(dz_small, torch.zeros_like(z_enter), z_enter)
        z_exit = torch.where(dz_small, torch.ones_like(z_exit), z_exit)
        z_valid = torch.where(
            dz_small,
            horizontal_z_ok,
            z_enter <= z_exit,
        )

        a = (dxy * dxy).sum(-1)
        b = 2.0 * (rel_xy * dxy).sum(-1)
        c = (rel_xy * rel_xy).sum(-1) - obs_r * obs_r
        a_small = a <= eps
        discriminant = b * b - 4.0 * a * c
        root = discriminant.clamp_min(0.0).sqrt()
        safe_two_a = torch.where(
            a_small, torch.ones_like(a), 2.0 * a
        )
        t1 = (-b - root) / safe_two_a
        t2 = (-b + root) / safe_two_a
        xy_enter = torch.maximum(
            torch.zeros_like(t1),
            torch.minimum(t1, t2),
        )
        xy_exit = torch.minimum(
            torch.ones_like(t1),
            torch.maximum(t1, t2),
        )
        xy_enter = torch.where(a_small, torch.zeros_like(xy_enter), xy_enter)
        xy_exit = torch.where(a_small, torch.ones_like(xy_exit), xy_exit)
        xy_valid = torch.where(
            a_small,
            c <= 0.0,
            (discriminant >= 0.0) & (xy_enter <= xy_exit),
        )

        enter = torch.maximum(z_enter, xy_enter)
        exit_ = torch.minimum(z_exit, xy_exit)
        intersects = z_valid & xy_valid & (enter <= exit_)
        return intersects.any(-1)


    def _motion(self, motion):
        E, U, T = self.num_envs, self.num_uavs, self.num_targets
        old_pos = self.positions
        old_vel = self.velocities
        projected = self._project_unit_ball(motion)
        accel = projected * self.max_accel
        candidate_velocity = old_vel + accel * self.dt
        speed = torch.linalg.vector_norm(
            candidate_velocity,
            dim=-1,
            keepdim=True,
        )
        candidate_velocity = candidate_velocity * torch.minimum(
            torch.ones_like(speed),
            torch.tensor(self.max_speed, device=self.device)
            / speed.clamp_min(1e-8),
        )
        raw_pos = old_pos + candidate_velocity * self.dt
        candidate_pos = raw_pos.clone()
        candidate_pos[..., 0].clamp_(0.0, self.map_size)
        candidate_pos[..., 1].clamp_(0.0, self.map_size)
        candidate_pos[..., 2].clamp_(
            self.altitude_min,
            self.altitude_max,
        )
        hclip = (
            (raw_pos[..., :2] - candidate_pos[..., :2])
            .abs()
            .amax(-1)
            > 1e-6
        )
        zclip = (
            (raw_pos[..., 2] - candidate_pos[..., 2]).abs()
            > 1e-6
        )
        # CPU reference uses the realized velocity after boundary clipping.
        candidate_velocity = (
            candidate_pos - old_pos
        ) / self.dt

        # Exact CPU-equivalent segment/finite-cylinder collision, including
        # horizontal and vertical clearance.
        clearance = float(CONFIG["obstacle_clearance_m"])
        blocked_by_obstacle = self._segment_cylinder_blocked(
            old_pos,
            candidate_pos,
            obstacle_radius=self.obstacle_radius + clearance,
            obstacle_height=self.obstacle_height + clearance,
        ) & self.active

        # Match apply_swarm_motion(): collision is checked over the entire
        # simultaneous segment, not only at the candidate endpoints. Blocking
        # is monotone, so U fixed iterations are sufficient for convergence
        # without a CPU-side convergence check.
        blocked = blocked_by_obstacle.clone()
        blocked_by_peer = torch.zeros_like(blocked)
        eye = torch.eye(
            U,
            device=self.device,
            dtype=torch.bool,
        ).unsqueeze(0)
        active_pair = (
            self.active.unsqueeze(2)
            & self.active.unsqueeze(1)
            & ~eye
        )
        safety = float(CONFIG["safety_distance"])

        for _ in range(U):
            effective = torch.where(
                blocked.unsqueeze(-1),
                old_pos,
                candidate_pos,
            )
            relative_start = (
                old_pos.unsqueeze(2) - old_pos.unsqueeze(1)
            )
            displacement = effective - old_pos
            relative_motion = (
                displacement.unsqueeze(2)
                - displacement.unsqueeze(1)
            )
            denominator = (
                relative_motion * relative_motion
            ).sum(-1)
            t_closest = torch.where(
                denominator > 1e-12,
                -(
                    relative_start * relative_motion
                ).sum(-1)
                / denominator.clamp_min(1e-12),
                torch.zeros_like(denominator),
            ).clamp(0.0, 1.0)
            relative_at_closest = (
                relative_start
                + t_closest.unsqueeze(-1) * relative_motion
            )
            min_distance = torch.linalg.vector_norm(
                relative_at_closest,
                dim=-1,
            )
            pair_bad = (
                (min_distance < safety)
                & active_pair
            )
            peer_now = pair_bad.any(-1)
            blocked_by_peer |= peer_now
            blocked |= peer_now

        new_pos = torch.where(
            blocked.unsqueeze(-1),
            old_pos,
            candidate_pos,
        )
        new_vel = torch.where(
            blocked.unsqueeze(-1),
            torch.zeros_like(candidate_velocity),
            candidate_velocity,
        )
        new_vel = torch.where(
            self.active.unsqueeze(-1),
            new_vel,
            torch.zeros_like(new_vel),
        )
        new_pos = torch.where(
            self.active.unsqueeze(-1),
            new_pos,
            old_pos,
        )
        realized_accel = (
            new_vel - old_vel
        ) / self.dt

        self.positions = new_pos
        self.velocities = new_vel
        return {
            "blocked": blocked,
            "blocked_by_obstacle": blocked_by_obstacle,
            "blocked_by_peer": blocked_by_peer,
            "boundary": hclip | zclip,
            "horizontal_boundary": hclip,
            "altitude_boundary": zclip,
            "realized_accel": realized_accel,
            "commanded_motion": projected,
        }

    def _sensing_profile(self):
        z = self.positions[..., 2]
        anchors = torch.tensor(CONFIG["altitude_anchors"], device=self.device)
        pdv = torch.tensor(CONFIG["pd"], device=self.device)
        pfv = torch.tensor(CONFIG["pf"], device=self.device)
        # Piecewise linear interpolation over four anchors.
        idx = torch.bucketize(z.contiguous(), anchors[1:-1])
        z0, z1 = anchors[idx], anchors[idx + 1]
        w = ((z - z0) / (z1 - z0).clamp_min(1e-6)).clamp(0.0, 1.0)
        pd = pdv[idx] + w * (pdv[idx + 1] - pdv[idx])
        pf = pfv[idx] + w * (pfv[idx + 1] - pfv[idx])
        half = math.radians(float(CONFIG["camera_full_fov_deg"]) / 2.0)
        radius = z * math.tan(half)
        return pd, pf, radius

    @staticmethod
    def _circle_sqrt_integral(x, radius):
        radius_safe = radius.clamp_min(torch.finfo(x.dtype).eps)
        x = torch.maximum(torch.minimum(x, radius), -radius)
        root = (radius * radius - x * x).clamp_min(0.0).sqrt()
        ratio = (x / radius_safe).clamp(-1.0, 1.0)
        return 0.5 * (x * root + radius * radius * torch.asin(ratio))

    def _cell_coverage_fraction(self, circle_x, circle_y, radius, gx, gy):
        original_shape = gx.shape
        circle_x = circle_x.reshape(-1)
        circle_y = circle_y.reshape(-1)
        radius = radius.reshape(-1)
        gx = gx.reshape(-1).float()
        gy = gy.reshape(-1).float()

        x_min = gx * self.cell_size - circle_x
        x_max = ((gx + 1.0) * self.cell_size).clamp(max=self.map_size) - circle_x
        y_min = gy * self.cell_size - circle_y
        y_max = ((gy + 1.0) * self.cell_size).clamp(max=self.map_size) - circle_y
        left = torch.maximum(x_min, -radius)
        right = torch.minimum(x_max, radius)
        base_valid = (
            (radius > 0.0)
            & (right > left)
            & (y_max > -radius)
            & (y_min < radius)
        )

        def crossings(y_edge):
            has = y_edge.abs() < radius
            x_cross = (radius * radius - y_edge * y_edge).clamp_min(0.0).sqrt()
            negative = torch.where(has, -x_cross, left)
            positive = torch.where(has, x_cross, left)
            negative = torch.maximum(left, torch.minimum(right, negative))
            positive = torch.maximum(left, torch.minimum(right, positive))
            return negative, positive

        y_min_neg, y_min_pos = crossings(y_min)
        y_max_neg, y_max_pos = crossings(y_max)
        cuts = torch.stack(
            (left, right, y_min_neg, y_min_pos, y_max_neg, y_max_pos), dim=-1
        ).sort(dim=-1).values
        a, b = cuts[:, :-1], cuts[:, 1:]
        width = b - a
        midpoint = 0.5 * (a + b)
        radius_2d = radius[:, None]
        y_min_2d = y_min[:, None]
        y_max_2d = y_max[:, None]
        half_height = (
            radius_2d * radius_2d - midpoint * midpoint
        ).clamp_min(0.0).sqrt()
        upper = torch.minimum(y_max_2d, half_height)
        lower = torch.maximum(y_min_2d, -half_height)
        arc = self._circle_sqrt_integral(b, radius_2d) - self._circle_sqrt_integral(
            a, radius_2d
        )
        upper_integral = torch.where(
            y_max_2d < half_height, y_max_2d * width, arc
        )
        lower_integral = torch.where(
            y_min_2d > -half_height, y_min_2d * width, -arc
        )
        area = torch.where(
            (width > 0.0) & (upper > lower),
            upper_integral - lower_integral,
            torch.zeros_like(width),
        ).sum(-1).clamp_min(0.0)
        cell_area = ((x_max - x_min) * (y_max - y_min)).clamp_min(
            torch.finfo(area.dtype).eps
        )
        fraction = (area / cell_area).clamp(0.0, 1.0)
        return torch.where(base_valid, fraction, torch.zeros_like(fraction)).reshape(
            original_shape
        )

    def _sense(self):
        E, U, T = self.num_envs, self.num_uavs, self.num_targets
        pd, pf, radius = self._sensing_profile()
        base = torch.floor(self.positions[..., :2] / self.cell_size).long()
        cand = base.unsqueeze(2) + self.cell_offsets.view(1, 1, -1, 2)
        gx, gy = cand[..., 0], cand[..., 1]
        valid = (gx >= 0) & (gx < self.grid_n) & (gy >= 0) & (gy < self.grid_n)
        gx_c = gx.clamp(0, self.grid_n - 1)
        gy_c = gy.clamp(0, self.grid_n - 1)
        centers = (torch.stack((gx_c, gy_c), dim=-1).float() + 0.5) * self.cell_size

        coverage = self._cell_coverage_fraction(
            self.positions[..., 0].unsqueeze(-1).expand_as(gx_c),
            self.positions[..., 1].unsqueeze(-1).expand_as(gy_c),
            radius.unsqueeze(-1).expand_as(gx_c),
            gx_c,
            gy_c,
        )
        visible = valid & (coverage > 0.0)
        visible &= (self.positions[..., 2] > 0.0).unsqueeze(-1) & self.active.unsqueeze(-1)

        if bool(CONFIG["sensing_obstacle_occlusion"]):
            start = self.positions.unsqueeze(2).expand(-1, -1, centers.shape[2], -1)
            end = torch.cat(
                (centers, torch.zeros(*centers.shape[:-1], 1, device=self.device)), dim=-1
            )
            blocked = self._segment_cylinder_blocked(start, end)
            visible &= ~blocked

        target_cell = torch.floor(self.target_xy / self.cell_size).long().clamp(0, self.grid_n - 1)
        cell_match = (
            (gx_c.unsqueeze(-1) == target_cell[:, None, None, :, 0])
            & (gy_c.unsqueeze(-1) == target_cell[:, None, None, :, 1])
        )
        target_distance = torch.linalg.vector_norm(
            self.target_xy[:, None, :, :] - self.positions[:, :, None, :2], dim=-1
        )
        target_in_fov = target_distance <= radius[:, :, None]

        if bool(CONFIG["sensing_obstacle_occlusion"]):
            target_end = torch.cat(
                (
                    self.target_xy[:, None, :, :].expand(-1, U, -1, -1),
                    torch.zeros(E, U, self.num_targets, 1, device=self.device),
                ),
                dim=-1,
            )
            target_start = self.positions[:, :, None, :].expand(
                -1, -1, self.num_targets, -1
            )
            target_blocked = self._segment_cylinder_blocked(
                target_start, target_end
            )
            target_in_fov &= ~target_blocked

        target_visible_per_cell = (
            cell_match
            & target_in_fov[:, :, None, :]
        )
        has_target = target_visible_per_cell.any(-1) & visible

        effective_pf = (
            1.0 - torch.pow(1.0 - pf.unsqueeze(-1), coverage)
        ).clamp(0.0, 1.0)
        effective_pd = (
            coverage * pd.unsqueeze(-1)
            + (1.0 - coverage) * effective_pf
        ).clamp(0.0, 1.0)
        positive_prob = torch.where(
            has_target, pd.unsqueeze(-1).expand_as(effective_pf), effective_pf
        )
        observation = self._rand(*positive_prob.shape) < positive_prob
        observation &= visible

        flat_idx = gy_c * self.grid_n + gx_c
        belief_flat = self.belief.view(E, U, -1)
        sentinel_index = self.grid_n * self.grid_n
        padded_belief = torch.cat(
            (
                belief_flat,
                torch.full(
                    (E, U, 1),
                    float(CONFIG["belief_prior"]),
                    device=self.device,
                    dtype=belief_flat.dtype,
                ),
            ),
            dim=2,
        )
        scatter_idx = torch.where(
            valid,
            flat_idx,
            torch.full_like(flat_idx, sentinel_index),
        )
        prior = torch.gather(
            padded_belief,
            2,
            scatter_idx,
        )
        eps = 1e-6
        prior_c = prior.clamp(eps, 1.0 - eps)
        pos_num = effective_pd * prior_c
        pos_den = pos_num + effective_pf * (1.0 - prior_c)
        neg_num = (1.0 - effective_pd) * prior_c
        neg_den = neg_num + (1.0 - effective_pf) * (1.0 - prior_c)
        posterior = torch.where(
            observation,
            pos_num / pos_den.clamp_min(eps),
            neg_num / neg_den.clamp_min(eps),
        ).clamp(eps, 1.0 - eps)
        posterior = torch.where(visible, posterior, prior)
        padded_belief.scatter_(
            2,
            scatter_idx,
            posterior,
        )
        belief_flat.copy_(
            padded_belief[..., :sentinel_index]
        )

        fine = self.positions[..., 2] <= float(CONFIG["fine_altitude"])
        confirm_cell = (
            visible
            & observation
            & (posterior >= float(CONFIG["confirmation_threshold"]))
            & fine.unsqueeze(-1)
            & has_target
        )
        confirm_target = (
            confirm_cell.unsqueeze(-1) & target_visible_per_cell
        ).any(2)  # [E,U,T]
        any_confirm = confirm_target.any(1)
        newly_confirmed = any_confirm & ~self.confirmed
        self.confirmed |= any_confirm

        # CPU semantics permit each confirming source UAV to create its own
        # report copy. A later confirmation can also create a new source copy
        # if that UAV no longer has that target buffered/pending.
        already_local = self.report_valid | self.pending_valid
        create_mask = (
            confirm_target
            & ~self.delivered[:, None, :]
            & ~already_local
        )
        capacity = int(
            float(CONFIG["buffer_bytes"])
            // float(CONFIG["report_bytes"])
        )
        pending_capacity = int(
            float(CONFIG["pending_buffer_bytes"])
            // float(CONFIG["report_bytes"])
        )
        free_slots = (
            capacity - self.report_valid.sum(-1)
        ).clamp_min(0)
        direct_rank = create_mask.long().cumsum(-1)
        direct = (
            create_mask
            & (direct_rank <= free_slots.unsqueeze(-1))
        )
        remaining_create = create_mask & ~direct
        pending_free_slots = (
            pending_capacity - self.pending_valid.sum(-1)
        ).clamp_min(0)
        pending_rank = remaining_create.long().cumsum(-1)
        pending = (
            remaining_create
            & (
                pending_rank
                <= pending_free_slots.unsqueeze(-1)
            )
        )
        dropped = create_mask & ~direct & ~pending

        self.report_valid |= direct
        self.report_remaining = torch.where(
            direct,
            torch.full_like(
                self.report_remaining,
                float(CONFIG["report_bytes"]),
            ),
            self.report_remaining,
        )
        self.report_created = torch.where(
            direct,
            self.step_count[:, None, None].expand_as(
                self.report_created
            ),
            self.report_created,
        )
        holder_ids = torch.arange(
            U,
            device=self.device,
            dtype=torch.long,
        ).view(1, U, 1).expand(E, U, T)
        self.report_source = torch.where(
            direct,
            holder_ids,
            self.report_source,
        )
        self.pending_valid |= pending
        self.pending_remaining = torch.where(
            pending,
            torch.full_like(
                self.pending_remaining,
                float(CONFIG["report_bytes"]),
            ),
            self.pending_remaining,
        )
        self.pending_created = torch.where(
            pending,
            self.step_count[:, None, None].expand_as(
                self.pending_created
            ),
            self.pending_created,
        )

        false_mask = (
            visible
            & observation
            & (
                posterior
                >= float(CONFIG["confirmation_threshold"])
            )
            & fine.unsqueeze(-1)
            & ~has_target
        )
        false_confirmation = false_mask.sum(dim=(1, 2))
        corrected = torch.where(
            false_mask,
            torch.full_like(
                posterior,
                float(CONFIG["verified_empty_belief"]),
            ),
            torch.gather(
                padded_belief,
                2,
                scatter_idx,
            ),
        )
        padded_belief.scatter_(
            2,
            scatter_idx,
            corrected,
        )
        belief_flat.copy_(
            padded_belief[..., :sentinel_index]
        )

        return {
            "new_confirmed": newly_confirmed.sum(-1),
            "false_confirmation": false_confirmation,
            "dropped": dropped.sum(dim=(1, 2)),
        }


    def _flush_pending(self):
        capacity = int(
            float(CONFIG["buffer_bytes"])
            // float(CONFIG["report_bytes"])
        )
        sentinel = (
            self.max_steps
            + int(CONFIG["report_ttl"])
            + 1
        )
        holder_ids = torch.arange(
            self.num_uavs,
            device=self.device,
            dtype=torch.long,
        ).view(1, self.num_uavs, 1).expand(
            self.num_envs,
            self.num_uavs,
            self.num_targets,
        )
        for _ in range(capacity):
            free = (
                (self.report_valid.sum(-1) < capacity)
                & self.active
            )
            created = torch.where(
                self.pending_valid,
                self.pending_created,
                torch.full_like(
                    self.pending_created,
                    sentinel,
                ),
            )
            target = created.argmin(-1)
            selected = torch.gather(
                self.pending_valid,
                2,
                target.unsqueeze(-1),
            ).squeeze(-1)
            move = free & selected
            target_oh = F.one_hot(
                target,
                self.num_targets,
            ).bool()
            move_mask = move.unsqueeze(-1) & target_oh
            self.report_valid |= move_mask
            self.report_remaining = torch.where(
                move_mask,
                self.pending_remaining,
                self.report_remaining,
            )
            self.report_created = torch.where(
                move_mask,
                self.pending_created,
                self.report_created,
            )
            self.report_source = torch.where(
                move_mask,
                holder_ids,
                self.report_source,
            )
            self.pending_valid &= ~move_mask
            self.pending_remaining = torch.where(
                self.pending_valid,
                self.pending_remaining,
                torch.zeros_like(
                    self.pending_remaining
                ),
            )

    def _destination_mask(self):
        E, U = self.num_envs, self.num_uavs
        has_report = self.report_valid.any(-1) & self.active
        mask = torch.zeros(E, U, self.discrete_dim, device=self.device)
        mask[..., 0] = 1.0

        diff = self.positions.unsqueeze(2) - self.positions.unsqueeze(1)
        peer_dist = torch.linalg.vector_norm(diff, dim=-1)
        for sender in range(U):
            peers = self.choices[sender, 1:U]
            ok = (
                peer_dist[:, sender, peers] <= float(CONFIG["peer_contact_range_m"])
            ) & self.active[:, peers] & has_report[:, sender, None]
            mask[:, sender, 1:U] = ok.float()
        gcs_dist = torch.linalg.vector_norm(self.positions - self.gcs.view(1, 1, 3), dim=-1)
        mask[..., U] = (
            (gcs_dist <= float(CONFIG["gcs_contact_range_m"])) & has_report
        ).float()
        return mask


    def _commit_completed_peer_receipts(self):
        """Retry complete peer transfers when receiver buffer space exists."""
        E, U, T = (
            self.num_envs,
            self.num_uavs,
            self.num_targets,
        )
        report_size = float(CONFIG["report_bytes"])
        capacity = int(
            float(CONFIG["buffer_bytes"]) // report_size
        )
        for sender in range(U):
            for receiver in range(U):
                if sender == receiver:
                    continue
                for target in range(T):
                    complete = (
                        self.peer_received[
                            :, sender, receiver, target
                        ]
                        >= report_size
                    )
                    source = self.peer_source[
                        :, sender, receiver, target
                    ]
                    created = self.peer_created[
                        :, sender, receiver, target
                    ]
                    generation_valid = (
                        source >= 0
                    ) & (
                        created
                        <= self.step_count
                    )
                    free = (
                        self.report_valid[:, receiver, :]
                        .sum(-1)
                        < capacity
                    )
                    add = (
                        complete
                        & generation_valid
                        & ~self.delivered[:, target]
                        & ~self.report_valid[
                            :, receiver, target
                        ]
                        & free
                    )
                    # known GCS progress is the best progress among all
                    # copies of the same report generation.
                    same_generation = (
                        self.report_valid[:, :, target]
                        & (
                            self.report_source[
                                :, :, target
                            ]
                            == source[:, None]
                        )
                        & (
                            self.report_created[
                                :, :, target
                            ]
                            == created[:, None]
                        )
                    )
                    known_remaining = torch.where(
                        same_generation,
                        self.report_remaining[
                            :, :, target
                        ],
                        torch.full_like(
                            self.report_remaining[
                                :, :, target
                            ],
                            report_size,
                        ),
                    ).amin(-1)
                    self.report_valid[
                        :, receiver, target
                    ] |= add
                    self.report_remaining[
                        :, receiver, target
                    ] = torch.where(
                        add,
                        known_remaining,
                        self.report_remaining[
                            :, receiver, target
                        ],
                    )
                    self.report_created[
                        :, receiver, target
                    ] = torch.where(
                        add,
                        created,
                        self.report_created[
                            :, receiver, target
                        ],
                    )
                    self.report_source[
                        :, receiver, target
                    ] = torch.where(
                        add,
                        source,
                        self.report_source[
                            :, receiver, target
                        ],
                    )

    def _uavnetsim_gpu_radio(
        self,
        selected_dest,
        tx_power,
        attempted_mask,
    ):
        """Vectorized UavNetSim-v2 A2A PHY/SINR surrogate on one CUDA device.

        The external simulator remains a SimPy discrete-event engine. This
        routine mirrors the configured A2A path-gain law and simultaneous-link
        SINR calculation in tensor form so hundreds/thousands of training
        environments do not leave CUDA for packet/radio calculations.
        """
        E, U = self.num_envs, self.num_uavs
        is_gcs = selected_dest == -1
        is_peer = selected_dest >= 0
        receiver = selected_dest.clamp(0, max(0, U - 1))
        peer_pos = torch.gather(
            self.positions,
            1,
            receiver.unsqueeze(-1).expand(-1, -1, 3),
        )
        dest_pos = torch.where(
            is_gcs.unsqueeze(-1),
            self.gcs.view(1, 1, 3),
            peer_pos,
        )
        min_distance = 1.0
        distance = torch.linalg.vector_norm(
            dest_pos - self.positions,
            dim=-1,
        ).clamp_min(min_distance)
        receiver_active = torch.gather(
            self.active,
            1,
            receiver,
        )
        contact = torch.where(
            is_gcs,
            distance <= float(CONFIG["gcs_contact_range_m"]),
            torch.where(
                is_peer,
                distance <= float(CONFIG["peer_contact_range_m"]),
                torch.zeros_like(is_peer),
            ),
        )
        contact &= torch.where(
            is_peer,
            receiver_active,
            torch.ones_like(receiver_active),
        )

        frequency = float(CONFIG["uavnetsim_carrier_frequency_hz"])
        wavelength = 299_792_458.0 / frequency
        reference_gain = (wavelength / (4.0 * math.pi)) ** 2
        los_model = str(CONFIG["uavnetsim_los_model"]).lower()
        nlos_model = str(CONFIG["uavnetsim_nlos_model"]).lower()
        los_params = {
            "free_space": (2.0, 0.0),
            "log_distance": (2.1, 1.0),
        }
        nlos_params = {
            "urban": (3.0, 12.0),
            "suburban": (2.6, 8.0),
        }
        if los_model not in los_params:
            raise ValueError(f"unsupported UavNetSim LoS model: {los_model}")
        if nlos_model not in nlos_params:
            raise ValueError(f"unsupported UavNetSim NLoS model: {nlos_model}")
        los_exp, los_excess = los_params[los_model]
        nlos_exp, nlos_excess = nlos_params[nlos_model]

        # CSMA/CA normally serializes contenders. Applying every attempted
        # sender as simultaneous interference would badly under-estimate its
        # throughput; the expected backoff/contention model below accounts for
        # sharing. Explicit simultaneous interference is reserved for ALOHA-like
        # access where overlapping transmissions are part of the MAC semantics.
        mac_name = str(CONFIG["uavnetsim_mac_protocol"]).strip().upper()
        interference_enabled = (
            bool(CONFIG.get("full_gpu_network_interference", True))
            and mac_name in {"PURE_ALOHA", "ALOHA"}
        )
        if interference_enabled:
            tx_start = self.positions[:, :, None, :].expand(
                -1, -1, U, -1
            )
            rx_end = dest_pos[:, None, :, :].expand(
                -1, U, -1, -1
            )
            pair_distance = torch.linalg.vector_norm(
                rx_end - tx_start,
                dim=-1,
            ).clamp_min(min_distance)
            pair_nlos = self._segment_cylinder_blocked(
                tx_start,
                rx_end,
            )
            exponent = torch.where(
                pair_nlos,
                torch.full_like(pair_distance, nlos_exp),
                torch.full_like(pair_distance, los_exp),
            )
            excess_db = torch.where(
                pair_nlos,
                torch.full_like(pair_distance, nlos_excess),
                torch.full_like(pair_distance, los_excess),
            )
            gain = (
                reference_gain
                * torch.pow(pair_distance, -exponent)
                * torch.pow(
                    torch.full_like(pair_distance, 10.0),
                    -excess_db / 10.0,
                )
            )
            # UavNetSim treats a transmitter that is itself the selected
            # receiver as unit-gain self-interference (half-duplex conflict).
            tx_ids = torch.arange(U, device=self.device).view(1, U, 1)
            selected_receivers = receiver[:, None, :]
            self_interference = is_peer[:, None, :] & (
                tx_ids == selected_receivers
            )
            gain = torch.where(
                self_interference,
                torch.ones_like(gain),
                gain,
            )
            rx_matrix = tx_power[:, :, None] * gain
            desired = torch.diagonal(
                rx_matrix, dim1=1, dim2=2
            )
            active_tx = attempted_mask[:, :, None].to(rx_matrix.dtype)
            total_received = (rx_matrix * active_tx).sum(dim=1)
            interference = (
                total_received
                - desired * attempted_mask.to(desired.dtype)
            ).clamp_min(0.0)
            selected_nlos = torch.diagonal(
                pair_nlos, dim1=1, dim2=2
            )
        else:
            selected_nlos = self._segment_cylinder_blocked(
                self.positions,
                dest_pos,
            )
            exponent = torch.where(
                selected_nlos,
                torch.full_like(distance, nlos_exp),
                torch.full_like(distance, los_exp),
            )
            excess_db = torch.where(
                selected_nlos,
                torch.full_like(distance, nlos_excess),
                torch.full_like(distance, los_excess),
            )
            gain = (
                reference_gain
                * torch.pow(distance, -exponent)
                * torch.pow(
                    torch.full_like(distance, 10.0),
                    -excess_db / 10.0,
                )
            )
            desired = tx_power * gain
            interference = torch.zeros_like(desired)

        thermal_dbm = (
            float(CONFIG["uavnetsim_thermal_noise_density_dbm_hz"])
            + 10.0 * math.log10(float(CONFIG["uavnetsim_bandwidth_hz"]))
            + float(CONFIG["uavnetsim_receiver_noise_figure_db"])
        )
        noise_w = 10.0 ** ((thermal_dbm - 30.0) / 10.0)
        sinr = desired / (noise_w + interference).clamp_min(1e-30)
        sinr_db = 10.0 * torch.log10(sinr.clamp_min(1e-20))
        link_ok = (
            attempted_mask
            & contact
            & (
                sinr_db
                >= float(CONFIG["uavnetsim_sinr_threshold_db"])
            )
        )
        rate = torch.where(
            link_ok,
            torch.full_like(
                sinr_db,
                float(CONFIG["uavnetsim_bit_rate_bps"]),
            ),
            torch.zeros_like(sinr_db),
        )
        return {
            "receiver": receiver,
            "dest_pos": dest_pos,
            "distance": distance,
            "contact": contact,
            "nlos": selected_nlos,
            "signal_w": desired,
            "interference_w": interference,
            "sinr_db": sinr_db,
            "link_ok": link_ok,
            "rate_bps": rate,
        }


    def _network(self, destination_idx, power_action):
        E, U, T = (
            self.num_envs,
            self.num_uavs,
            self.num_targets,
        )
        report_size = float(CONFIG["report_bytes"])
        selected_dest = torch.gather(
            self.choices.unsqueeze(0).expand(
                E, -1, -1
            ),
            2,
            destination_idx.unsqueeze(-1),
        ).squeeze(-1)
        tx_power = float(CONFIG["tx_power_min_w"]) + (
            power_action.squeeze(-1)
            .clamp(-1.0, 1.0)
            + 1.0
        ) * 0.5 * (
            float(CONFIG["tx_power_max_w"])
            - float(CONFIG["tx_power_min_w"])
        )

        # CPU backend retries complete custody transfers before new radio work.
        self._commit_completed_peer_receipts()

        attempted_all = torch.zeros(
            E, U, device=self.device, dtype=torch.bool
        )
        actual_all = torch.zeros(
            E, U, device=self.device
        )
        comm_energy_all = torch.zeros(
            E, U, device=self.device
        )
        new_delivery_total = torch.zeros(
            E, device=self.device, dtype=torch.long
        )
        sentinel = (
            self.max_steps
            + int(CONFIG["report_ttl"])
            + 1
        )
        target_ids = self.target_ids

        network_model = self.network_model

        if network_model in {"packet_approx", "uavnetsim_gpu"}:
            all_is_gcs = selected_dest == -1
            all_is_peer = selected_dest >= 0
            all_gcs_has_report = (
                self.report_valid
                & (self.report_remaining > 0.0)
            ).any(-1)
            all_peer_has_report = (
                self.report_valid.any(-1)
            )
            pre_attempt = (
                self.active
                & (
                    (
                        all_is_gcs
                        & all_gcs_has_report
                    )
                    | (
                        all_is_peer
                        & all_peer_has_report
                    )
                )
            )
            channel_contenders = (
                pre_attempt.sum(-1)
                .clamp_min(1)
                .float()
            )
            packet_payload_bytes = float(
                CONFIG[
                    "uavnetsim_payload_bytes"
                ]
            )
            queue_budget_bytes = (
                packet_payload_bytes
                * float(
                    CONFIG[
                        "uavnetsim_max_queue_size"
                    ]
                )
            )
            bit_rate = float(
                CONFIG[
                    "comm_max_link_rate_bps"
                ]
            )
            data_airtime_s = (
                packet_payload_bytes
                * 8.0
                / bit_rate
            )
            ack_airtime_s = (
                float(
                    CONFIG[
                        "full_gpu_packet_ack_bits"
                    ]
                )
                / bit_rate
            )
            avg_backoff_s = (
                (
                    float(
                        CONFIG[
                            "full_gpu_packet_cw_min"
                        ]
                    )
                    - 1.0
                )
                * 0.5
                * float(
                    CONFIG[
                        "full_gpu_packet_slot_us"
                    ]
                )
                * 1e-6
            )
            fixed_mac_overhead_s = (
                (
                    float(
                        CONFIG[
                            "full_gpu_packet_difs_us"
                        ]
                    )
                    + float(
                        CONFIG[
                            "full_gpu_packet_sifs_us"
                        ]
                    )
                )
                * 1e-6
                + ack_airtime_s
                + avg_backoff_s
            )
            packet_service_time_s = (
                data_airtime_s
                + fixed_mac_overhead_s
            )
            mac_service_budget_bytes = (
                math.floor(
                    self.dt
                    / max(
                        packet_service_time_s,
                        1e-12,
                    )
                )
                * packet_payload_bytes
            )
        else:
            channel_contenders = torch.ones(
                E,
                device=self.device,
            )
            packet_payload_bytes = float(
                CONFIG[
                    "uavnetsim_payload_bytes"
                ]
            )
            queue_budget_bytes = float("inf")
            mac_service_budget_bytes = float(
                "inf"
            )

        uavnetsim_radio = None
        if network_model == "uavnetsim_gpu":
            # Pre-compute all selected-link radio quantities in one batched
            # CUDA pass. Report mutation below remains ordered per sender so
            # custody/delivery semantics stay identical and deterministic.
            uavnetsim_radio = self._uavnetsim_gpu_radio(
                selected_dest,
                tx_power,
                pre_attempt,
            )

        for sender in range(U):
            dest = selected_dest[:, sender]
            is_gcs = dest == -1
            is_peer = dest >= 0
            receiver = dest.clamp(
                0,
                max(0, U - 1),
            )

            report_valid = self.report_valid[
                :, sender, :
            ]
            gcs_candidate = (
                report_valid
                & (
                    self.report_remaining[
                        :, sender, :
                    ]
                    > 0.0
                )
            )

            peer_received = torch.gather(
                self.peer_received[:, sender, :, :],
                1,
                receiver[:, None, None].expand(
                    -1, 1, T
                ),
            ).squeeze(1)
            peer_source = torch.gather(
                self.peer_source[:, sender, :, :],
                1,
                receiver[:, None, None].expand(
                    -1, 1, T
                ),
            ).squeeze(1)
            peer_created = torch.gather(
                self.peer_created[:, sender, :, :],
                1,
                receiver[:, None, None].expand(
                    -1, 1, T
                ),
            ).squeeze(1)
            same_generation = (
                (
                    peer_source
                    == self.report_source[
                        :, sender, :
                    ]
                )
                & (
                    peer_created
                    == self.report_created[
                        :, sender, :
                    ]
                )
            )
            peer_complete = (
                same_generation
                & (peer_received >= report_size)
            )
            peer_candidate = (
                report_valid
                & ~peer_complete
            )
            candidate = torch.where(
                is_gcs[:, None],
                gcs_candidate,
                torch.where(
                    is_peer[:, None],
                    peer_candidate,
                    torch.zeros_like(
                        peer_candidate
                    ),
                ),
            )
            key = torch.where(
                candidate,
                self.report_created[
                    :, sender, :
                ] * (T + 1)
                + target_ids,
                torch.full_like(
                    self.report_created[
                        :, sender, :
                    ],
                    sentinel * (T + 1) + T,
                ),
            )
            target = key.argmin(-1)
            selected_valid = candidate.any(-1)
            attempted = (
                selected_valid
                & self.active[:, sender]
                & (is_gcs | is_peer)
            )
            attempted_all[:, sender] = attempted

            if uavnetsim_radio is not None:
                receiver = uavnetsim_radio["receiver"][:, sender]
                dest_pos = uavnetsim_radio["dest_pos"][:, sender, :]
                distance = uavnetsim_radio["distance"][:, sender]
                contact = uavnetsim_radio["contact"][:, sender]
                nlos = uavnetsim_radio["nlos"][:, sender]
                snr_db = uavnetsim_radio["sinr_db"][:, sender]
                rate = uavnetsim_radio["rate_bps"][:, sender]
                link_ok = (
                    uavnetsim_radio["link_ok"][:, sender]
                    & attempted
                )
                capacity_bytes = torch.where(
                    link_ok,
                    torch.floor(rate * self.dt / 8.0),
                    torch.zeros_like(rate),
                )
            else:
                peer_pos = torch.gather(
                    self.positions,
                    1,
                    receiver[:, None, None].expand(
                        -1, 1, 3
                    ),
                ).squeeze(1)
                dest_pos = torch.where(
                    is_gcs[:, None],
                    self.gcs.view(1, 3),
                    peer_pos,
                )
                distance = torch.linalg.vector_norm(
                    dest_pos - self.positions[:, sender, :],
                    dim=-1,
                ).clamp_min(
                    float(CONFIG["comm_reference_distance_m"])
                )
                contact = torch.where(
                    is_gcs,
                    distance <= float(CONFIG["gcs_contact_range_m"]),
                    torch.where(
                        is_peer,
                        distance <= float(CONFIG["peer_contact_range_m"]),
                        torch.zeros_like(is_peer),
                    ),
                )
                receiver_active = torch.gather(
                    self.active,
                    1,
                    receiver[:, None],
                ).squeeze(1)
                contact &= torch.where(
                    is_peer,
                    receiver_active,
                    torch.ones_like(receiver_active),
                )
                nlos = self._segment_cylinder_blocked(
                    self.positions[:, sender, :],
                    dest_pos,
                )
                gain_db = (
                    float(CONFIG["comm_reference_gain_db"])
                    - 10.0
                    * float(CONFIG["comm_path_loss_exponent"])
                    * torch.log10(
                        distance
                        / float(CONFIG["comm_reference_distance_m"])
                    )
                    - nlos.float()
                    * float(CONFIG["comm_nlos_additional_loss_db"])
                )
                rx_w = tx_power[:, sender] * torch.pow(
                    torch.tensor(10.0, device=self.device),
                    gain_db / 10.0,
                )
                noise_w = 10.0 ** (
                    (float(CONFIG["comm_noise_power_dbm"]) - 30.0)
                    / 10.0
                )
                snr = rx_w / noise_w
                snr_db = 10.0 * torch.log10(
                    snr.clamp_min(1e-20)
                )
                rate = float(CONFIG["comm_bandwidth_hz"]) * torch.log2(
                    1.0 + snr
                )
                rate = rate.clamp_max(
                    float(CONFIG["comm_max_link_rate_bps"])
                )
                link_ok = (
                    attempted
                    & contact
                    & (
                        snr_db
                        >= float(CONFIG["comm_snr_threshold_db"])
                    )
                )
                capacity_bytes = torch.where(
                    link_ok,
                    torch.floor(rate * self.dt / 8.0),
                    torch.zeros_like(rate),
                )
            if network_model in {"packet_approx", "uavnetsim_gpu"}:
                capacity_bytes = torch.floor(
                    torch.minimum(
                        torch.minimum(
                            capacity_bytes,
                            torch.full_like(
                                capacity_bytes,
                                queue_budget_bytes,
                            ),
                        ),
                        torch.full_like(
                            capacity_bytes,
                            float(
                                mac_service_budget_bytes
                            ),
                        ),
                    )
                    * torch.pow(
                        torch.full_like(
                            channel_contenders,
                            float(
                                CONFIG[
                                    "full_gpu_packet_contention_decay"
                                ]
                            ),
                        ),
                        (
                            channel_contenders
                            - 1.0
                        ).clamp_min(0.0),
                    )
                )

            target_oh = F.one_hot(
                target,
                T,
            ).bool()
            selected_source = torch.gather(
                self.report_source[
                    :, sender, :
                ],
                1,
                target[:, None],
            ).squeeze(1)
            selected_created = torch.gather(
                self.report_created[
                    :, sender, :
                ],
                1,
                target[:, None],
            ).squeeze(1)

            # GCS transfer: delivered-byte progress is shared by report copies
            # of the same generation.
            selected_remaining = torch.gather(
                self.report_remaining[
                    :, sender, :
                ],
                1,
                target[:, None],
            ).squeeze(1)
            gcs_actual = torch.minimum(
                capacity_bytes,
                selected_remaining,
            )
            gcs_actual = torch.where(
                is_gcs & selected_valid,
                gcs_actual,
                torch.zeros_like(gcs_actual),
            )
            gcs_new_remaining = (
                selected_remaining - gcs_actual
            ).clamp_min(0.0)
            copy_match = (
                self.report_valid
                & target_oh[:, None, :]
                & (
                    self.report_source
                    == selected_source[
                        :, None, None
                    ]
                )
                & (
                    self.report_created
                    == selected_created[
                        :, None, None
                    ]
                )
            )
            gcs_update = (
                is_gcs
                & link_ok
                & selected_valid
            )
            self.report_remaining = torch.where(
                copy_match
                & gcs_update[:, None, None],
                gcs_new_remaining[
                    :, None, None
                ],
                self.report_remaining,
            )
            gcs_complete = (
                gcs_update
                & (gcs_new_remaining <= 0.0)
            )
            delivered_target = (
                target_oh
                & gcs_complete[:, None]
            )
            new_delivery = (
                delivered_target
                & ~self.delivered
            )
            new_delivery_total += (
                new_delivery.sum(-1)
            )
            self.delivered |= delivered_target

            # Keep mission cleanup entirely tensorized: no CUDA->CPU scalar
            # synchronization is needed when no delivery completed.
            clear_target = delivered_target
            self.report_valid &= ~clear_target[
                :, None, :
            ]
            self.report_remaining = torch.where(
                self.report_valid,
                self.report_remaining,
                torch.zeros_like(
                    self.report_remaining
                ),
            )
            self.report_source = torch.where(
                self.report_valid,
                self.report_source,
                torch.full_like(
                    self.report_source,
                    -1,
                ),
            )
            self.pending_valid &= ~clear_target[
                :, None, :
            ]
            self.pending_remaining = torch.where(
                self.pending_valid,
                self.pending_remaining,
                torch.zeros_like(
                    self.pending_remaining
                ),
            )
            peer_clear = clear_target[
                :, None, None, :
            ]
            self.peer_received = torch.where(
                peer_clear,
                torch.zeros_like(
                    self.peer_received
                ),
                self.peer_received,
            )
            self.peer_source = torch.where(
                peer_clear,
                torch.full_like(
                    self.peer_source,
                    -1,
                ),
                self.peer_source,
            )

            # Peer transfer: progress is per sender-recipient-target and does
            # not consume/remove the sender's report.
            current_peer_received = torch.gather(
                peer_received,
                1,
                target[:, None],
            ).squeeze(1)
            current_peer_source = torch.gather(
                peer_source,
                1,
                target[:, None],
            ).squeeze(1)
            current_peer_created = torch.gather(
                peer_created,
                1,
                target[:, None],
            ).squeeze(1)
            generation_match = (
                (
                    current_peer_source
                    == selected_source
                )
                & (
                    current_peer_created
                    == selected_created
                )
            )
            base_peer_received = torch.where(
                generation_match,
                current_peer_received,
                torch.zeros_like(
                    current_peer_received
                ),
            )
            peer_remaining = (
                report_size - base_peer_received
            ).clamp_min(0.0)

            requested_bytes = torch.where(
                is_gcs,
                selected_remaining,
                torch.where(
                    is_peer,
                    peer_remaining,
                    torch.zeros_like(
                        selected_remaining
                    ),
                ),
            )
            max_attempt_bytes = float(
                math.floor(
                    float(
                        CONFIG[
                            "comm_max_link_rate_bps"
                        ]
                    )
                    * self.dt
                    / 8.0
                )
            )
            attempted_bytes = torch.minimum(
                requested_bytes,
                torch.full_like(
                    requested_bytes,
                    max_attempt_bytes,
                ),
            )
            tx_duration = torch.minimum(
                torch.full_like(
                    attempted_bytes,
                    self.dt,
                ),
                attempted_bytes
                * 8.0
                / float(
                    CONFIG[
                        "comm_max_link_rate_bps"
                    ]
                ),
            )
            comm_energy_all[:, sender] = torch.where(
                attempted,
                tx_power[:, sender]
                * tx_duration,
                torch.zeros_like(tx_duration),
            )

            peer_actual = torch.minimum(
                capacity_bytes,
                peer_remaining,
            )
            peer_actual = torch.where(
                is_peer
                & link_ok
                & selected_valid,
                peer_actual,
                torch.zeros_like(peer_actual),
            )
            new_peer_received = (
                base_peer_received + peer_actual
            ).clamp_max(report_size)
            receiver_oh = F.one_hot(
                receiver,
                U,
            ).bool()
            pair_mask = (
                receiver_oh[:, :, None]
                & target_oh[:, None, :]
                & (
                    is_peer
                    & selected_valid
                )[:, None, None]
            )
            peer_slice = self.peer_received[
                :, sender, :, :
            ]
            source_slice = self.peer_source[
                :, sender, :, :
            ]
            created_slice = self.peer_created[
                :, sender, :, :
            ]
            self.peer_received[
                :, sender, :, :
            ] = torch.where(
                pair_mask,
                new_peer_received[
                    :, None, None
                ],
                peer_slice,
            )
            self.peer_source[
                :, sender, :, :
            ] = torch.where(
                pair_mask,
                selected_source[
                    :, None, None
                ],
                source_slice,
            )
            self.peer_created[
                :, sender, :, :
            ] = torch.where(
                pair_mask,
                selected_created[
                    :, None, None
                ],
                created_slice,
            )
            actual_all[:, sender] = (
                gcs_actual + peer_actual
            )

        # Same-step custody commit after all radio transfers.
        self._commit_completed_peer_receipts()
        packets_tx = torch.where(
            actual_all > 0.0,
            torch.ceil(
                actual_all
                / packet_payload_bytes
            ),
            torch.zeros_like(actual_all),
        )
        return {
            "new_delivery": new_delivery_total,
            "comm_energy": comm_energy_all,
            "attempted": attempted_all,
            "bytes_tx": actual_all,
            "packets_tx": packets_tx,
            "channel_contenders": (
                channel_contenders
            ),
            "link_ok": (
                uavnetsim_radio["link_ok"]
                if uavnetsim_radio is not None
                else (actual_all > 0.0)
            ),
            "sinr_db": (
                uavnetsim_radio["sinr_db"]
                if uavnetsim_radio is not None
                else torch.full_like(actual_all, float("nan"))
            ),
            "nlos": (
                uavnetsim_radio["nlos"]
                if uavnetsim_radio is not None
                else torch.zeros_like(attempted_all)
            ),
            "interference_w": (
                uavnetsim_radio["interference_w"]
                if uavnetsim_radio is not None
                else torch.zeros_like(actual_all)
            ),
        }


    def _energy(self, motion_result, comm_energy):
        speed2 = (
            self.velocities * self.velocities
        ).sum(-1)
        realized = motion_result[
            "realized_accel"
        ]
        realized_norm = torch.linalg.vector_norm(
            realized,
            dim=-1,
            keepdim=True,
        )
        realized = realized * torch.minimum(
            torch.ones_like(realized_norm),
            torch.tensor(
                self.max_accel,
                device=self.device,
            )
            / realized_norm.clamp_min(1e-8),
        )
        accel2 = (realized * realized).sum(-1)
        commanded_accel = (
            torch.linalg.vector_norm(
                motion_result["commanded_motion"],
                dim=-1,
            )
            * self.max_accel
        )
        ground_idle = (
            (self.positions[..., 2] <= 0.0)
            & (speed2 <= 1e-12)
            & (commanded_accel <= 1e-12)
        )
        base = torch.where(
            ground_idle,
            torch.full_like(
                speed2,
                float(CONFIG["energy_idle_power_w"]),
            ),
            torch.full_like(
                speed2,
                float(CONFIG["energy_hover_power_w"]),
            ),
        )
        propulsion = (
            base
            + float(
                CONFIG["energy_speed_sq_coeff"]
            )
            * speed2
            + float(
                CONFIG["energy_accel_sq_coeff"]
            )
            * accel2
        ) * self.dt
        total = (
            propulsion + comm_energy
        ) * self.active.float()
        self.battery = (
            self.battery - total
        ).clamp_min(0.0)
        self.active &= self.battery > 0.0
        self.velocities = torch.where(
            self.active.unsqueeze(-1),
            self.velocities,
            torch.zeros_like(self.velocities),
        )
        return total


    def _expire_reports(self):
        ttl = float(CONFIG["report_ttl"])
        age = (
            self.step_count[:, None, None]
            - self.report_created
        )
        expired = (
            self.report_valid
            & (age.float() * self.dt >= ttl)
        )
        pending_age = (
            self.step_count[:, None, None]
            - self.pending_created
        )
        pending_expired = (
            self.pending_valid
            & (
                pending_age.float()
                * self.dt
                >= ttl
            )
        )
        expired_targets = (
            expired | pending_expired
        ).any(dim=1).sum(-1)

        self.report_valid &= ~expired
        self.pending_valid &= ~pending_expired
        self.report_remaining = torch.where(
            self.report_valid,
            self.report_remaining,
            torch.zeros_like(
                self.report_remaining
            ),
        )
        self.report_source = torch.where(
            self.report_valid,
            self.report_source,
            torch.full_like(
                self.report_source,
                -1,
            ),
        )
        self.pending_remaining = torch.where(
            self.pending_valid,
            self.pending_remaining,
            torch.zeros_like(
                self.pending_remaining
            ),
        )

        peer_age = (
            self.step_count[:, None, None, None]
            - self.peer_created
        )
        peer_expired = (
            (self.peer_received > 0.0)
            & (
                peer_age.float()
                * self.dt
                >= ttl
            )
        )
        self.peer_received = torch.where(
            peer_expired,
            torch.zeros_like(self.peer_received),
            self.peer_received,
        )
        self.peer_source = torch.where(
            peer_expired,
            torch.full_like(
                self.peer_source,
                -1,
            ),
            self.peer_source,
        )
        return expired_targets

    def _observation(self, entropy_map=None):
        E, U, O = self.num_envs, self.num_uavs, self.num_obstacles
        p = self.positions
        v = self.velocities
        time_frac = (self.step_count.float() / max(1, self.max_steps)).clamp(0.0, 1.0)
        self_state = torch.cat(
            (
                (p[..., :2] / self.map_size).clamp(0.0, 1.0),
                ((p[..., 2:3] - self.altitude_min) / max(1e-6, self.altitude_max - self.altitude_min)).clamp(0.0, 1.0),
                (v / self.max_speed).clamp(-1.0, 1.0),
                (self.battery / float(CONFIG["battery_j"])).unsqueeze(-1).clamp(0.0, 1.0),
                self.active.float().unsqueeze(-1),
                time_frac[:, None, None].expand(E, U, 1),
            ),
            dim=-1,
        )

        gcs_rel = (
            (self.gcs.view(1, 1, 3) - p)
            / self.xyz_scale.view(1, 1, 3)
        ).clamp(-1.0, 1.0)

        # Vectorized peer features while preserving CPU peer-ID order.
        all_relative = (
            p[:, None, :, :]
            - p[:, :, None, :]
        )
        gather_index = self.peer_index.view(
            1, U, U - 1, 1
        ).expand(E, -1, -1, 3)
        relative = torch.gather(
            all_relative,
            2,
            gather_index,
        )
        distance = torch.linalg.vector_norm(
            relative,
            dim=-1,
        )
        peer_active = torch.gather(
            self.active[:, None, :].expand(
                E, U, U
            ),
            2,
            self.peer_index.view(
                1, U, U - 1
            ).expand(E, -1, -1),
        )
        visible = (
            self.active[:, :, None]
            & peer_active
            & (
                distance
                <= float(
                    CONFIG["peer_contact_range_m"]
                )
            )
        )
        neighbors = torch.cat(
            (
                (
                    relative
                    / self.xyz_scale.view(
                        1, 1, 1, 3
                    )
                ).clamp(-1.0, 1.0),
                (
                    distance
                    / float(
                        CONFIG[
                            "peer_contact_range_m"
                        ]
                    )
                )
                .clamp(0.0, 1.0)
                .unsqueeze(-1),
                visible.float().unsqueeze(-1),
            ),
            dim=-1,
        )
        neighbors = torch.where(
            visible.unsqueeze(-1),
            neighbors,
            torch.zeros_like(neighbors),
        )

        # Nearest 3 obstacles.
        odelta = self.obstacle_xy[:, None, :, :] - p[:, :, None, :2]
        odist = torch.linalg.vector_norm(odelta, dim=-1)
        ok = min(int(CONFIG["observation_nearest_obstacles"]), O)
        _, oi = torch.topk(odist, k=ok, dim=-1, largest=False)
        oxy = torch.gather(
            self.obstacle_xy[:, None, :, :].expand(E, U, O, 2),
            2, oi.unsqueeze(-1).expand(-1, -1, -1, 2)
        )
        orad = torch.gather(self.obstacle_radius[:, None, :].expand(E, U, O), 2, oi)
        oh = torch.gather(self.obstacle_height[:, None, :].expand(E, U, O), 2, oi)
        orel = oxy - p[:, :, None, :2]
        center_distance = torch.linalg.vector_norm(orel, dim=-1)
        obstacles = torch.cat(
            (
                (orel / self.map_size).clamp(-1.0, 1.0),
                (center_distance / (math.sqrt(2.0) * self.map_size))
                .clamp(0.0, 1.0)
                .unsqueeze(-1),
                (orad / self.map_size).clamp(0.0, 1.0).unsqueeze(-1),
                (oh / max(self.altitude_max, 1.0)).clamp(0.0, 1.0).unsqueeze(-1),
            ),
            dim=-1,
        )

        # 13x13 local belief patch.
        center = torch.floor(
            p[..., :2] / self.cell_size
        ).long()
        pc = (
            center.unsqueeze(2)
            + self.belief_patch_offsets.view(
                1, 1, -1, 2
            )
        )
        pxg, pyg = pc[..., 0], pc[..., 1]
        pvalid = (pxg >= 0) & (pxg < self.grid_n) & (pyg >= 0) & (pyg < self.grid_n)
        pidx = pyg.clamp(0, self.grid_n - 1) * self.grid_n + pxg.clamp(0, self.grid_n - 1)
        patch = torch.gather(self.belief.view(E, U, -1), 2, pidx)
        patch = torch.where(pvalid, patch, torch.full_like(patch, float(CONFIG["belief_prior"])))

        coarse_n = int(CONFIG["belief_coarse_cells"])
        if entropy_map is None:
            entropy_map = self._belief_entropy()
        coarse = F.adaptive_avg_pool2d(
            entropy_map.view(E * U, 1, self.grid_n, self.grid_n),
            (coarse_n, coarse_n),
        ).view(E, U, -1).clamp(0.0, 1.0)

        valid_count = self.report_valid.sum(-1).float()
        oldest = torch.where(
            self.report_valid,
            self.report_created,
            torch.full_like(self.report_created, self.max_steps + int(CONFIG["report_ttl"]) + 1),
        ).amin(-1)
        age = torch.where(
            valid_count > 0,
            (self.step_count[:, None] - oldest).float() * self.dt / max(float(CONFIG["report_ttl"]), 1.0),
            torch.zeros_like(valid_count),
        ).clamp(0.0, 1.0)
        capacity_reports = max(
            1.0, float(CONFIG["buffer_bytes"]) / float(CONFIG["report_bytes"])
        )
        used_fraction = (valid_count / capacity_reports).clamp(0.0, 1.0)
        report_fraction = used_fraction
        pending_fraction = (
            self.pending_valid.sum(-1).float()
            / max(
                1.0,
                float(CONFIG["pending_buffer_bytes"]) / float(CONFIG["report_bytes"]),
            )
        ).clamp(0.0, 1.0)
        ordered_created = torch.where(
            self.report_valid,
            self.report_created,
            torch.full_like(
                self.report_created,
                self.max_steps + int(CONFIG["report_ttl"]) + 1,
            ),
        )
        head_target = ordered_created.argmin(-1)
        head_remaining = torch.gather(
            self.report_remaining, 2, head_target.unsqueeze(-1)
        ).squeeze(-1)
        head_delivery = torch.where(
            valid_count > 0,
            (1.0 - head_remaining / float(CONFIG["report_bytes"])).clamp(0.0, 1.0),
            torch.zeros_like(valid_count),
        )
        buffer_features = torch.stack(
            (used_fraction, report_fraction, age, pending_fraction, head_delivery),
            dim=-1,
        )

        dest_mask = self._destination_mask()
        obs = torch.cat(
            (
                self_state,
                gcs_rel,
                neighbors.reshape(E, U, -1),
                obstacles.reshape(E, U, -1),
                patch,
                coarse,
                buffer_features,
                dest_mask,
            ),
            dim=-1,
        ).float()
        if obs.shape[-1] != self.observation_dim:
            raise RuntimeError(f"GPU observation dim {obs.shape[-1]} != {self.observation_dim}")
        return obs, dest_mask

    def _report_slots(self, valid, remaining, created):
        T = valid.shape[-1]
        slot_count = int(float(CONFIG["buffer_bytes"]) // float(CONFIG["report_bytes"]))
        target_ids = torch.arange(T, device=self.device).view(1, 1, T)
        sentinel = self.max_steps + int(CONFIG["report_ttl"]) + 1
        key = torch.where(
            valid,
            created * (T + 1) + target_ids,
            torch.full_like(created, sentinel * (T + 1) + T),
        )
        order = torch.argsort(key, dim=-1)[..., :slot_count]
        chosen_valid = torch.gather(valid, 2, order)
        chosen_remaining = torch.gather(remaining, 2, order)
        chosen_created = torch.gather(created, 2, order)
        age = (
            (self.step_count[:, None, None] - chosen_created).float()
            * self.dt
            / max(float(CONFIG["report_ttl"]), 1.0)
        ).clamp(0.0, 1.0)
        delivery = (1.0 - chosen_remaining / float(CONFIG["report_bytes"])).clamp(
            0.0, 1.0
        )
        target_fraction = order.float() / max(1.0, float(T - 1))
        slots = torch.stack(
            (
                chosen_valid.float(),
                target_fraction,
                age,
                delivery,
            ),
            dim=-1,
        )
        return torch.where(chosen_valid.unsqueeze(-1), slots, torch.zeros_like(slots))

    def _state(self, obs, entropy_map=None):
        E, U = self.num_envs, self.num_uavs
        buffer_slots = self._report_slots(
            self.report_valid, self.report_remaining, self.report_created
        )
        pending_slots = self._report_slots(
            self.pending_valid, self.pending_remaining, self.pending_created
        )
        if entropy_map is None:
            entropy_map = self._belief_entropy()
        entropy = entropy_map
        summary = torch.stack(
            (
                entropy.mean(dim=(-2, -1)).clamp(0.0, 1.0),
                ((self.belief - float(CONFIG["belief_prior"])).abs() > 0.05)
                .float()
                .mean(dim=(-2, -1)),
            ),
            dim=-1,
        )
        critic_cells = int(CONFIG["critic_belief_grid_cells"])
        pooled_belief = F.adaptive_avg_pool2d(
            self.belief.reshape(E * U, 1, self.grid_n, self.grid_n),
            (critic_cells, critic_cells),
        ).reshape(E, U, -1)

        per_uav = torch.cat(
            (
                obs,
                buffer_slots.reshape(E, U, -1),
                pending_slots.reshape(E, U, -1),
                summary,
                pooled_belief,
            ),
            dim=-1,
        ).reshape(E, -1)

        targets = torch.cat(
            (
                (self.target_xy / self.map_size).clamp(0.0, 1.0),
                self.confirmed.float().unsqueeze(-1),
                self.delivered.float().unsqueeze(-1),
            ),
            dim=-1,
        ).reshape(E, -1)
        obstacles = torch.cat(
            (
                (self.obstacle_xy / self.map_size).clamp(0.0, 1.0),
                (self.obstacle_radius / self.map_size).clamp(0.0, 1.0).unsqueeze(-1),
                (self.obstacle_height / max(self.altitude_max, 1.0))
                .clamp(0.0, 1.0)
                .unsqueeze(-1),
            ),
            dim=-1,
        ).reshape(E, -1)
        time_fraction = (
            self.step_count.float() / max(1, self.max_steps)
        ).clamp(0.0, 1.0).unsqueeze(-1)
        state = torch.cat((per_uav, targets, obstacles, time_fraction), dim=-1)
        if state.shape[-1] != self.state_dim:
            raise RuntimeError(
                f"GPU global state dim {state.shape[-1]} != {self.state_dim}"
            )
        return state.float()

    @torch.no_grad()
    def step(self, continuous_action, destination_idx):
        if continuous_action.shape != (
            self.num_envs,
            self.num_uavs,
            self.continuous_dim,
        ):
            raise ValueError("continuous_action has wrong shape")
        if destination_idx.shape != (self.num_envs, self.num_uavs):
            raise ValueError("destination_idx has wrong shape")

        self.step_count += 1
        expired = self._expire_reports()
        self._flush_pending()
        old_potential = self.belief_potential
        old_confirmed = self.confirmed.clone()
        old_delivered = self.delivered.clone()

        motion = self._motion(continuous_action[..., :3])
        sensing = self._sense()
        if self.external_network is None:
            network = self._network(
                destination_idx.long(), continuous_action[..., 3:4]
            )
        else:
            network = self.external_network.step(
                destination_idx.long(), continuous_action[..., 3:4]
            )
        self._flush_pending()
        total_energy = self._energy(motion, network["comm_energy"])
        entropy_map = self._belief_entropy()
        new_potential = self._belief_potential(
            entropy_map
        )
        self.belief_potential = new_potential

        newly_confirmed = (self.confirmed & ~old_confirmed).sum(-1)
        newly_delivered = (self.delivered & ~old_delivered).sum(-1)
        success = self.delivered.all(-1)
        inactive = ~self.active.any(-1)
        horizon = self.step_count >= self.max_steps
        done = success | inactive | horizon
        next_phi = torch.where(done, torch.zeros_like(new_potential), new_potential)
        shaping = float(CONFIG["reward_info_gain"]) * (
            float(CONFIG["reward_shaping_gamma"]) * next_phi - old_potential
        )
        reward = (
            shaping
            + float(CONFIG["reward_confirmation"]) * newly_confirmed.float()
            + float(CONFIG["reward_delivery"]) * newly_delivered.float()
            - float(CONFIG["reward_false_confirmation"]) * sensing["false_confirmation"].float()
            - float(CONFIG["reward_blocked"]) * motion["blocked"].sum(-1).float()
            - float(CONFIG["reward_boundary"]) * motion["boundary"].sum(-1).float()
            - float(CONFIG["reward_expired_report"]) * expired.float()
            - float(CONFIG["reward_dropped_report"]) * sensing["dropped"].float()
            - float(CONFIG["reward_energy_per_kj"]) * total_energy.sum(-1) / 1000.0
            - float(CONFIG["reward_step_penalty"])
            + torch.where(success, torch.full_like(shaping, float(CONFIG["reward_all_delivered_bonus"])), torch.zeros_like(shaping))
        )

        obs, mask = self._observation(
            entropy_map=entropy_map
        )
        state = self._state(
            obs,
            entropy_map=entropy_map,
        )
        metrics = {
            "success": success,
            "new_confirmed": newly_confirmed,
            "new_delivered": newly_delivered,
            "blocked": motion["blocked"].sum(-1),
            "boundary": motion["boundary"].sum(-1),
            "energy_j": total_energy.sum(-1),
        }
        return obs, state, mask, reward.float(), done, metrics

    def assert_all_state_on_cuda(self):
        tensors = {
            k: v
            for k, v in self.__dict__.items()
            if isinstance(v, torch.Tensor)
        }
        bad = [k for k, v in tensors.items() if v.device.type != "cuda"]
        if bad:
            raise AssertionError(f"state tensors not on CUDA: {bad}")
        return len(tensors)


## GPU replay buffer

In [ ]:
class GpuReplayBuffer:
    """Replay buffer whose storage and sampling stay on CUDA."""

    def __init__(self, capacity, num_agents, observation_dim, state_dim, continuous_dim, discrete_dim, device, seed=44, strict_cuda=True):
        self.capacity = int(capacity)
        self.num_agents = int(num_agents)
        self.observation_dim = int(observation_dim)
        self.state_dim = int(state_dim)
        self.continuous_dim = int(continuous_dim)
        self.discrete_dim = int(discrete_dim)
        self.device = torch.device(device)
        if strict_cuda:
            _require_cuda(self.device)
        C, U, O, S, A, D = self.capacity, self.num_agents, self.observation_dim, self.state_dim, self.continuous_dim, self.discrete_dim
        self.observations = torch.empty(C, U, O, device=self.device)
        self.next_observations = torch.empty_like(self.observations)
        self.states = torch.empty(C, S, device=self.device)
        self.next_states = torch.empty_like(self.states)
        self.continuous_actions = torch.empty(C, U, A, device=self.device)
        self.destination_indices = torch.empty(C, U, device=self.device, dtype=torch.long)
        self.destination_masks = torch.empty(C, U, D, device=self.device)
        self.next_destination_masks = torch.empty_like(self.destination_masks)
        self.rewards = torch.empty(C, 1, device=self.device)
        self.terminated = torch.empty(C, 1, device=self.device)
        self.truncated = torch.zeros(C, 1, device=self.device)
        self.position = 0
        self.size = 0
        self.generator = torch.Generator(device=self.device)
        self.generator.manual_seed(int(seed))

    def __len__(self):
        return int(self.size)

    @torch.no_grad()
    def add_batch(self, observations, states, continuous_actions, destination_indices, destination_masks, rewards, next_observations, next_states, next_destination_masks, terminated):
        B = observations.shape[0]
        idx = (torch.arange(B, device=self.device) + self.position) % self.capacity
        self.observations[idx] = observations
        self.states[idx] = states
        self.continuous_actions[idx] = continuous_actions
        self.destination_indices[idx] = destination_indices
        self.destination_masks[idx] = destination_masks
        self.rewards[idx, 0] = rewards
        self.next_observations[idx] = next_observations
        self.next_states[idx] = next_states
        self.next_destination_masks[idx] = next_destination_masks
        self.terminated[idx, 0] = terminated.float()
        self.truncated[idx, 0] = 0.0
        self.position = (self.position + B) % self.capacity
        self.size = min(self.capacity, self.size + B)

    def sample(self, batch_size, device):
        device = torch.device(device)
        if device != self.device:
            raise ValueError("GpuReplayBuffer sample device must equal storage device")
        idx = torch.randint(
            0, self.size, (int(batch_size),),
            generator=self.generator, device=self.device
        )
        return {
            "observations": self.observations[idx],
            "next_observations": self.next_observations[idx],
            "states": self.states[idx],
            "next_states": self.next_states[idx],
            "continuous_actions": self.continuous_actions[idx],
            "destination_indices": self.destination_indices[idx],
            "destination_masks": self.destination_masks[idx],
            "next_destination_masks": self.next_destination_masks[idx],
            "rewards": self.rewards[idx],
            "terminated": self.terminated[idx],
            "truncated": self.truncated[idx],
        }


## Training and benchmark helpers

In [ ]:
import importlib
import os

def load_project_namespace(repo: Path):
    """Load definitions from test.ipynb without executing notebook tests/runs."""
    nb = json.loads(
        (repo / "test.ipynb").read_text()
    )
    source = "\n\n".join(
        "".join(cell["source"])
        for cell in nb["cells"]
        if cell.get("cell_type") == "code"
    )
    tree = ast.parse(
        source,
        filename="test.ipynb",
    )

    def safe_expr(node):
        if isinstance(
            node,
            (
                ast.Constant,
                ast.Name,
                ast.Attribute,
            ),
        ):
            return True
        if isinstance(
            node,
            (
                ast.Tuple,
                ast.List,
                ast.Set,
            ),
        ):
            return all(
                safe_expr(item)
                for item in node.elts
            )
        if isinstance(node, ast.Dict):
            return all(
                (
                    key is None
                    or safe_expr(key)
                )
                and safe_expr(value)
                for key, value in zip(
                    node.keys,
                    node.values,
                )
            )
        if isinstance(node, ast.UnaryOp):
            return safe_expr(node.operand)
        if isinstance(node, ast.BinOp):
            return (
                safe_expr(node.left)
                and safe_expr(node.right)
            )
        return False

    kept = []
    for node in tree.body:
        if isinstance(
            node,
            (
                ast.Import,
                ast.ImportFrom,
                ast.FunctionDef,
                ast.AsyncFunctionDef,
                ast.ClassDef,
            ),
        ):
            kept.append(node)
        elif (
            isinstance(node, ast.Assign)
            and safe_expr(node.value)
        ):
            kept.append(node)
        elif (
            isinstance(node, ast.AnnAssign)
            and node.value is not None
            and safe_expr(node.value)
        ):
            kept.append(node)

    module = ast.Module(
        body=kept,
        type_ignores=[],
    )
    ast.fix_missing_locations(module)
    module_name = "_uav_gpu_project"
    project_module = types.ModuleType(
        module_name
    )
    project_module.__file__ = "test.ipynb"
    sys.modules[module_name] = (
        project_module
    )
    ns = project_module.__dict__
    exec(
        compile(
            module,
            "test.ipynb",
            "exec",
        ),
        ns,
    )
    # A few notebook runtime globals are intentionally call-valued and are
    # therefore not part of the side-effect-free AST selection above.
    if (
        "_TORCH_SENSING_TLS"
        not in ns
        and "threading" in ns
    ):
        ns["_TORCH_SENSING_TLS"] = (
            ns["threading"].local()
        )
    return ns


def _trainer(
    repo: Path,
    algorithm: str,
    device: str,
    seed: int,
):
    """Build the existing MASAC/MATD3 learner without constructing a CPU env.

    Only lightweight Gymnasium space metadata is created on the host. No CPU
    mission reset/step/sensing/network simulation is executed by the full-GPU
    training path.
    """
    random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

    ns = load_project_namespace(repo)
    gym = ns["gym"]
    np = ns["np"]
    env_cls = ns["UAVSearchEnv"]
    dummy = object.__new__(env_cls)

    U = int(CONFIG["num_uavs"])
    T = int(CONFIG["num_targets"])
    O = int(CONFIG["num_obstacles"])
    D = U + 1
    nearest = int(CONFIG["observation_nearest_obstacles"])
    patch = int(CONFIG["belief_patch_cells"])
    coarse = int(CONFIG["belief_coarse_cells"])
    critic = int(CONFIG["critic_belief_grid_cells"])
    agent_ids = tuple(f"uav_{index}" for index in range(U))
    dummy.agent_ids = agent_ids

    agent_space = gym.spaces.Dict(
        {
            "self_state": gym.spaces.Box(-1.0, 1.0, shape=(9,), dtype=np.float32),
            "gcs_relative": gym.spaces.Box(-1.0, 1.0, shape=(3,), dtype=np.float32),
            "neighbors": gym.spaces.Box(
                -1.0, 1.0, shape=(U - 1, 5), dtype=np.float32
            ),
            "obstacles": gym.spaces.Box(
                -1.0, 1.0, shape=(nearest, 5), dtype=np.float32
            ),
            "belief_patch": gym.spaces.Box(
                0.0, 1.0, shape=(patch, patch), dtype=np.float32
            ),
            "belief_coarse": gym.spaces.Box(
                0.0, 1.0, shape=(coarse, coarse), dtype=np.float32
            ),
            "buffer": gym.spaces.Box(0.0, 1.0, shape=(5,), dtype=np.float32),
            "destination_mask": gym.spaces.Box(
                0.0, 1.0, shape=(D,), dtype=np.float32
            ),
        }
    )
    dummy.observation_space = gym.spaces.Dict(
        {agent_id: agent_space for agent_id in agent_ids}
    )
    dummy.action_space = gym.spaces.Dict(
        {
            agent_id: gym.spaces.Dict(
                {"destination": gym.spaces.Discrete(D)}
            )
            for agent_id in agent_ids
        }
    )

    observation_dim = (
        9 + 3 + 5 * (U - 1) + 5 * nearest
        + patch * patch + coarse * coarse + 5 + D
    )
    buffer_slots = int(
        float(CONFIG["buffer_bytes"]) // float(CONFIG["report_bytes"])
    )
    pending_slots = int(
        float(CONFIG["pending_buffer_bytes"]) // float(CONFIG["report_bytes"])
    )
    per_uav_state = (
        observation_dim
        + 4 * buffer_slots
        + 4 * pending_slots
        + 2
        + critic * critic
    )
    state_dim = U * per_uav_state + 4 * T + 4 * O + 1
    dummy.state_space = gym.spaces.Box(
        low=-np.inf,
        high=np.inf,
        shape=(state_dim,),
        dtype=np.float32,
    )

    if algorithm == "masac":
        trainer = ns["HybridMASAC"](
            dummy,
            device=device,
        )
    else:
        trainer = ns["HybridMATD3"](
            dummy,
            device=device,
            seed=int(seed),
        )
    trainer._project_namespace = ns
    return trainer

def make_project_gpu_replay_buffer(
    trainer,
    *,
    capacity,
    num_agents,
    observation_dim,
    state_dim,
    continuous_dim,
    discrete_dim,
    device,
    seed,
    strict_cuda=True,
):
    project_ns = getattr(
        trainer,
        "_project_namespace",
        None,
    )
    if not isinstance(project_ns, dict):
        raise RuntimeError(
            "trainer is missing project namespace"
        )
    base = project_ns[
        "HybridReplayBuffer"
    ]

    class ProjectGpuReplayBuffer(
        GpuReplayBuffer,
        base,
    ):
        pass

    return ProjectGpuReplayBuffer(
        capacity=capacity,
        num_agents=num_agents,
        observation_dim=observation_dim,
        state_dim=state_dim,
        continuous_dim=continuous_dim,
        discrete_dim=discrete_dim,
        device=device,
        seed=seed,
        strict_cuda=strict_cuda,
    )


def _matd3_actions(
    trainer,
    obs,
    mask,
    transition_steps,
    generator,
):
    with torch.no_grad():
        sampled = trainer._actor_joint_actions(
            obs,
            mask,
            straight_through=False,
            target_actor=False,
        )
        continuous = sampled["continuous"]
        destination = sampled[
            "destination_indices"
        ]
        noise = (
            torch.randn(
                continuous.shape,
                device=continuous.device,
                generator=generator,
            )
            * float(
                CONFIG[
                    "matd3_exploration_noise"
                ]
            )
        )
        continuous = (
            continuous + noise
        ).clamp(-1.0, 1.0)
        motion = continuous[..., :3]
        norm = torch.linalg.vector_norm(
            motion,
            dim=-1,
            keepdim=True,
        ).clamp_min(1.0)
        continuous = torch.cat(
            (
                motion / norm,
                continuous[..., 3:4],
            ),
            dim=-1,
        )

        start = float(
            CONFIG[
                "matd3_discrete_epsilon_start"
            ]
        )
        end = float(
            CONFIG[
                "matd3_discrete_epsilon_end"
            ]
        )
        decay = max(
            1.0,
            float(
                CONFIG[
                    "matd3_discrete_epsilon_decay_steps"
                ]
            ),
        )
        fraction = (
            transition_steps.float()
            / decay
        ).clamp(0.0, 1.0)
        epsilon = (
            start
            + fraction * (end - start)
        )
        explore = (
            torch.rand(
                destination.shape,
                device=destination.device,
                generator=generator,
            )
            < epsilon[:, None]
        )
        random_logits = torch.rand(
            mask.shape,
            device=mask.device,
            generator=generator,
        ).masked_fill(
            mask <= 0.5,
            -1.0,
        )
        random_destination = (
            random_logits.argmax(-1)
        )
        destination = torch.where(
            explore,
            random_destination,
            destination,
        )
    return continuous, destination


def _random_exploration_actions(
    env,
    mask,
):
    shape = (
        env.num_envs,
        env.num_uavs,
        3,
    )
    direction = torch.randn(
        shape,
        device=env.device,
        generator=env.generator,
    )
    direction = direction / (
        torch.linalg.vector_norm(
            direction,
            dim=-1,
            keepdim=True,
        ).clamp_min(1e-12)
    )
    radius = torch.pow(
        torch.rand(
            env.num_envs,
            env.num_uavs,
            1,
            device=env.device,
            generator=env.generator,
        ),
        1.0 / 3.0,
    )
    motion = direction * radius
    power = (
        torch.rand(
            env.num_envs,
            env.num_uavs,
            1,
            device=env.device,
            generator=env.generator,
        )
        * 2.0
        - 1.0
    )
    continuous = torch.cat(
        (motion, power),
        dim=-1,
    )
    logits = torch.rand(
        mask.shape,
        device=env.device,
        generator=env.generator,
    ).masked_fill(
        mask <= 0.5,
        -1.0,
    )
    destination = logits.argmax(-1)
    return continuous, destination


def benchmark_env(num_envs: int, steps: int, seed: int, device: str, strict_cuda: bool = True):
    env = FullGpuUAVBatchEnv(num_envs=num_envs, device=device, seed=seed, strict_cuda=strict_cuda)
    n_cuda_tensors = (
        env.assert_all_state_on_cuda()
        if env.device.type == "cuda"
        else sum(isinstance(v, torch.Tensor) for v in env.__dict__.values())
    )
    _obs, _state, mask = env.reset()
    if env.device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    transitions = 0
    for _ in range(int(steps)):
        continuous, destination = (
            _random_exploration_actions(
                env,
                mask,
            )
        )
        _obs, _state, mask, _reward, _done, _ = env.step(continuous, destination)
        transitions += num_envs
    if env.device.type == "cuda":
        torch.cuda.synchronize(env.device)
    wall = time.perf_counter() - start
    return {
        "kind": "full_gpu_env",
        "num_envs": num_envs,
        "vector_steps": int(steps),
        "transitions": transitions,
        "wall_seconds": wall,
        "transitions_per_second": transitions / wall,
        "cuda_state_tensors": n_cuda_tensors,
        "gpu_name": torch.cuda.get_device_name(env.device) if env.device.type == "cuda" else "CPU_DEBUG",
        "peak_memory_mb": torch.cuda.max_memory_allocated(env.device) / 2**20 if env.device.type == "cuda" else 0.0,
    }




def _make_hybrid_vector_env(
    project_ns,
    num_envs,
):
    gym = project_ns["gym"]
    env_fns = [
        lambda: project_ns[
            "UAVSearchEnv"
        ](backend_name="simple")
        for _ in range(int(num_envs))
    ]
    return gym.vector.AsyncVectorEnv(
        env_fns,
        context=str(
            CONFIG.get(
                "training_vector_context",
                "fork",
            )
        ),
        shared_memory=bool(
            CONFIG.get(
                "training_vector_shared_memory",
                True,
            )
        ),
        autoreset_mode=(
            gym.vector.AutoresetMode.DISABLED
        ),
    )


def benchmark_hybrid_env(
    repo: Path,
    num_envs: int,
    steps: int,
    seed: int,
):
    """CPU vectorized reference env throughput (no learner update)."""
    project_ns = load_project_namespace(
        repo
    )
    np = project_ns["np"]
    vector_env = _make_hybrid_vector_env(
        project_ns,
        num_envs,
    )
    template = project_ns[
        "UAVSearchEnv"
    ](backend_name="simple")
    rng = np.random.default_rng(
        int(seed) + 991
    )
    try:
        observations, _ = vector_env.reset(
            seed=[
                int(seed) + index
                for index in range(num_envs)
            ]
        )
        start = time.perf_counter()
        transitions = 0
        next_episode_seed = (
            int(seed) + 100_000
        )
        for _ in range(int(steps)):
            observations_by_env = [
                project_ns[
                    "_vector_observation_at"
                ](
                    observations,
                    index,
                    num_envs,
                )
                for index in range(
                    num_envs
                )
            ]
            actions_by_env = [
                project_ns[
                    "sample_valid_random_actions"
                ](
                    observations_by_env[index],
                    template.agent_ids,
                    rng,
                )
                for index in range(
                    num_envs
                )
            ]
            vector_actions = project_ns[
                "_stack_vector_actions"
            ](
                actions_by_env,
                template.agent_ids,
            )
            (
                next_observations,
                _rewards,
                terminated,
                truncated,
                _infos,
            ) = vector_env.step(
                vector_actions
            )
            transitions += num_envs
            done = np.logical_or(
                terminated,
                truncated,
            )
            observations = next_observations
            if np.any(done):
                seeds = [None] * num_envs
                for index in np.flatnonzero(
                    done
                ):
                    seeds[int(index)] = (
                        next_episode_seed
                    )
                    next_episode_seed += 1
                observations, _ = (
                    vector_env.reset(
                        seed=seeds,
                        options={
                            "reset_mask": (
                                done.astype(
                                    np.bool_
                                )
                            )
                        },
                    )
                )
        wall = time.perf_counter() - start
        return {
            "kind": "hybrid_cpu_env",
            "num_envs": int(num_envs),
            "vector_steps": int(steps),
            "transitions": int(
                transitions
            ),
            "wall_seconds": wall,
            "transitions_per_second": (
                transitions / wall
            ),
        }
    finally:
        vector_env.close()
        template.close()


def benchmark_hybrid_train(
    repo: Path,
    algorithm: str,
    num_envs: int,
    total_transitions: int,
    seed: int,
    device: str,
):
    """Best current CPU-env + CUDA-learner path, including overlap."""
    algorithm = str(
        algorithm
    ).strip().lower()
    if algorithm not in {
        "masac",
        "matd3",
    }:
        raise ValueError(
            "algorithm must be masac or matd3"
        )
    if (
        int(total_transitions)
        % int(num_envs)
        != 0
    ):
        raise ValueError(
            "total_transitions must be "
            "divisible by num_envs"
        )

    project_ns = load_project_namespace(
        repo
    )
    np = project_ns["np"]
    template = project_ns[
        "UAVSearchEnv"
    ](backend_name="simple")
    trainer = _trainer(
        repo,
        algorithm,
        device,
        seed,
    )
    replay_capacity = min(
        int(
            CONFIG[
                f"{algorithm}_replay_capacity"
            ]
        ),
        max(
            4096,
            int(total_transitions),
        ),
    )
    replay = trainer._project_namespace[
        "HybridReplayBuffer"
    ](
        capacity=replay_capacity,
        num_agents=trainer.num_agents,
        observation_dim=(
            trainer.observation_dim
        ),
        state_dim=trainer.state_dim,
        continuous_dim=(
            trainer.continuous_dim
        ),
        discrete_dim=(
            trainer.discrete_dim
        ),
        seed=int(seed) + 123,
    )
    vector_env = _make_hybrid_vector_env(
        project_ns,
        num_envs,
    )

    batch_size = int(
        CONFIG[
            f"{algorithm}_batch_size"
        ]
    )
    learning_starts = int(
        CONFIG[
            f"{algorithm}_learning_starts"
        ]
    )
    updates_per_step = int(
        CONFIG[
            f"{algorithm}_updates_per_step"
        ]
    )
    rng = np.random.default_rng(
        int(seed)
    )
    overlap = bool(
        CONFIG.get(
            "training_overlap_env_and_updates",
            True,
        )
    )
    next_episode_seed = (
        int(seed) + 100_000
    )

    try:
        observations, infos = (
            vector_env.reset(
                seed=[
                    int(seed) + index
                    for index in range(
                        num_envs
                    )
                ]
            )
        )
        states = np.asarray(
            infos["state"],
            dtype=np.float32,
        ).copy()
        global_step = 0
        pending_updates = 0
        update_count = 0
        latest_metrics = None
        env_seconds = 0.0
        wait_seconds = 0.0
        update_seconds = 0.0
        policy_seconds = 0.0

        if (
            torch.device(device).type
            == "cuda"
        ):
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()

        wall_start = time.perf_counter()
        while global_step < int(
            total_transitions
        ):
            observations_by_env = [
                project_ns[
                    "_vector_observation_at"
                ](
                    observations,
                    index,
                    num_envs,
                )
                for index in range(
                    num_envs
                )
            ]
            exploration_steps = [
                global_step + index + 1
                for index in range(
                    num_envs
                )
            ]

            policy_start = time.perf_counter()
            if all(
                step <= learning_starts
                for step in exploration_steps
            ):
                actions_by_env = [
                    project_ns[
                        "sample_valid_random_actions"
                    ](
                        observations_by_env[
                            index
                        ],
                        template.agent_ids,
                        rng,
                    )
                    for index in range(
                        num_envs
                    )
                ]
            elif all(
                step > learning_starts
                for step in exploration_steps
            ):
                actions_by_env = project_ns[
                    "_select_vector_policy_actions"
                ](
                    trainer,
                    observations_by_env,
                    algorithm,
                    exploration_steps,
                )
            else:
                actions_by_env = []
                for index in range(
                    num_envs
                ):
                    transition_step = (
                        exploration_steps[index]
                    )
                    if (
                        transition_step
                        <= learning_starts
                    ):
                        action = project_ns[
                            "sample_valid_random_actions"
                        ](
                            observations_by_env[
                                index
                            ],
                            template.agent_ids,
                            rng,
                        )
                    elif algorithm == "masac":
                        action = (
                            trainer.select_actions(
                                observations_by_env[
                                    index
                                ],
                                deterministic=False,
                            )
                        )
                    else:
                        action = (
                            trainer.select_actions(
                                observations_by_env[
                                    index
                                ],
                                deterministic=False,
                                exploration_step=(
                                    transition_step
                                ),
                            )
                        )
                    actions_by_env.append(
                        action
                    )
            policy_seconds += (
                time.perf_counter()
                - policy_start
            )

            vector_actions = project_ns[
                "_stack_vector_actions"
            ](
                actions_by_env,
                template.agent_ids,
            )
            env_start = time.perf_counter()
            if overlap:
                vector_env.step_async(
                    vector_actions
                )
            else:
                result = vector_env.step(
                    vector_actions
                )

            if (
                pending_updates > 0
                and len(replay)
                >= batch_size
            ):
                update_start = (
                    time.perf_counter()
                )
                for _ in range(
                    pending_updates
                ):
                    latest_metrics = (
                        trainer.update(
                            replay,
                            batch_size=(
                                batch_size
                            ),
                        )
                    )
                    update_count += 1
                update_seconds += (
                    time.perf_counter()
                    - update_start
                )
                pending_updates = 0

            if overlap:
                wait_start = (
                    time.perf_counter()
                )
                result = (
                    vector_env.step_wait()
                )
                wait_seconds += (
                    time.perf_counter()
                    - wait_start
                )
            env_seconds += (
                time.perf_counter()
                - env_start
            )

            (
                next_observations,
                rewards,
                terminated,
                truncated,
                next_infos,
            ) = result
            next_states = np.asarray(
                next_infos["state"],
                dtype=np.float32,
            ).copy()

            previous_global_step = (
                global_step
            )
            replay_len_before = len(
                replay
            )
            for index in range(
                num_envs
            ):
                next_env_observations = (
                    project_ns[
                        "_vector_observation_at"
                    ](
                        next_observations,
                        index,
                        num_envs,
                    )
                )
                project_ns[
                    "add_environment_transition_to_replay"
                ](
                    replay,
                    observations_by_env[
                        index
                    ],
                    states[index],
                    actions_by_env[index],
                    float(rewards[index]),
                    next_env_observations,
                    next_states[index],
                    bool(
                        terminated[index]
                    ),
                    bool(
                        truncated[index]
                    ),
                    template.agent_ids,
                )

            global_step += num_envs
            pending_updates += (
                project_ns[
                    "_count_vector_updates_due"
                ](
                    previous_global_step=(
                        previous_global_step
                    ),
                    num_envs=num_envs,
                    learning_starts=(
                        learning_starts
                    ),
                    replay_len_before=(
                        replay_len_before
                    ),
                    replay_capacity=(
                        replay.capacity
                    ),
                    batch_size=batch_size,
                    updates_per_step=(
                        updates_per_step
                    ),
                )
            )

            if (
                not overlap
                and pending_updates > 0
            ):
                update_start = (
                    time.perf_counter()
                )
                for _ in range(
                    pending_updates
                ):
                    latest_metrics = (
                        trainer.update(
                            replay,
                            batch_size=(
                                batch_size
                            ),
                        )
                    )
                    update_count += 1
                update_seconds += (
                    time.perf_counter()
                    - update_start
                )
                pending_updates = 0

            done = np.logical_or(
                terminated,
                truncated,
            )
            observations = (
                next_observations
            )
            states = next_states
            if np.any(done):
                seeds = [None] * num_envs
                for index in np.flatnonzero(
                    done
                ):
                    seeds[int(index)] = (
                        next_episode_seed
                    )
                    next_episode_seed += 1
                (
                    observations,
                    reset_infos,
                ) = vector_env.reset(
                    seed=seeds,
                    options={
                        "reset_mask": done.astype(
                            np.bool_
                        )
                    },
                )
                reset_states = np.asarray(
                    reset_infos["state"],
                    dtype=np.float32,
                )
                reset_mask = np.asarray(
                    reset_infos.get(
                        "_state",
                        done,
                    ),
                    dtype=np.bool_,
                )
                for index in np.flatnonzero(
                    reset_mask
                ):
                    states[int(index)] = (
                        reset_states[
                            int(index)
                        ]
                    )

        if pending_updates > 0:
            update_start = (
                time.perf_counter()
            )
            for _ in range(
                pending_updates
            ):
                latest_metrics = (
                    trainer.update(
                        replay,
                        batch_size=batch_size,
                    )
                )
                update_count += 1
            update_seconds += (
                time.perf_counter()
                - update_start
            )

        if (
            torch.device(device).type
            == "cuda"
        ):
            torch.cuda.synchronize()
        wall = (
            time.perf_counter()
            - wall_start
        )

        return {
            "kind": "hybrid_cpu_env_cuda_learner",
            "algorithm": algorithm,
            "num_envs": int(
                num_envs
            ),
            "transitions": int(
                global_step
            ),
            "updates": int(
                update_count
            ),
            "wall_seconds": wall,
            "transitions_per_second": (
                global_step / wall
            ),
            "updates_per_second": (
                update_count / wall
            ),
            "env_collection_seconds": (
                env_seconds
            ),
            "env_wait_seconds": (
                wait_seconds
            ),
            "policy_inference_seconds": (
                policy_seconds
            ),
            "update_seconds": (
                update_seconds
            ),
            "overlap_enabled": bool(
                overlap
            ),
            "gpu_name": (
                torch.cuda.get_device_name()
                if (
                    torch.device(
                        device
                    ).type
                    == "cuda"
                    and torch.cuda.is_available()
                )
                else "CPU"
            ),
            "peak_memory_mb": (
                torch.cuda.max_memory_allocated()
                / 2**20
                if (
                    torch.device(
                        device
                    ).type
                    == "cuda"
                    and torch.cuda.is_available()
                )
                else 0.0
            ),
        }
    finally:
        vector_env.close()
        template.close()



def _make_hybrid_vector_env(
    project_ns,
    num_envs,
):
    gym = project_ns["gym"]
    env_fns = [
        lambda: project_ns[
            "UAVSearchEnv"
        ](backend_name="simple")
        for _ in range(int(num_envs))
    ]
    return gym.vector.AsyncVectorEnv(
        env_fns,
        context=str(
            CONFIG.get(
                "training_vector_context",
                "fork",
            )
        ),
        shared_memory=bool(
            CONFIG.get(
                "training_vector_shared_memory",
                True,
            )
        ),
        autoreset_mode=(
            gym.vector.AutoresetMode.DISABLED
        ),
    )



class GpuUavNetSimBridge:
    """Packet-level UavNetSim adapter for one tensorized mission environment."""

    def __init__(self, env, project_ns, *, repo_path=None, seed=44):
        if env.num_envs != 1:
            raise ValueError(
                "GpuUavNetSimBridge currently supports exactly one mission env"
            )
        self.env = env
        self.ns = project_ns
        self.repo_path = repo_path
        self.seed = int(seed)
        self.backend = None
        self.uavs = []
        self.obstacles = []
        self.report_buffers = []
        self.pending_reports = []
        self.gcs_received_target_ids = set()
        self._last_results = []
        self._last_metrics = None
        self._timing_totals = {
            "sync_gpu_to_cpu_seconds": 0.0,
            "uavnetsim_step_seconds": 0.0,
            "sync_cpu_to_gpu_seconds": 0.0,
            "calls": 0,
        }

    @property
    def np(self):
        return self.ns["np"]

    def _tensor_numpy(self, value, dtype=None):
        array = value.detach().cpu().numpy()
        if dtype is not None:
            array = array.astype(dtype, copy=False)
        return array

    def _build_mission_objects(self):
        np = self.np
        UAV = self.ns["UAV"]
        Obstacle = self.ns["Obstacle"]
        positions = self._tensor_numpy(self.env.positions[0], np.float64)
        velocities = self._tensor_numpy(self.env.velocities[0], np.float64)
        batteries = self._tensor_numpy(self.env.battery[0], np.float64)
        active = self._tensor_numpy(self.env.active[0], bool)
        self.uavs = [
            UAV(
                id=index,
                position=positions[index].copy(),
                velocity=velocities[index].copy(),
                battery_j=float(batteries[index]),
                active=bool(active[index]),
            )
            for index in range(self.env.num_uavs)
        ]
        obstacle_xy = self._tensor_numpy(self.env.obstacle_xy[0], np.float64)
        obstacle_radius = self._tensor_numpy(self.env.obstacle_radius[0], np.float64)
        obstacle_height = self._tensor_numpy(self.env.obstacle_height[0], np.float64)
        self.obstacles = [
            Obstacle(
                position=obstacle_xy[index].copy(),
                radius=float(obstacle_radius[index]),
                height=float(obstacle_height[index]),
            )
            for index in range(self.env.num_obstacles)
        ]
        self.report_buffers = [[] for _ in range(self.env.num_uavs)]
        self.pending_reports = []
        self.gcs_received_target_ids = set()
        self._sync_gpu_to_cpu()

    def _sync_gpu_to_cpu(self):
        np = self.np
        Report = self.ns["Report"]
        positions = self._tensor_numpy(self.env.positions[0], np.float64)
        velocities = self._tensor_numpy(self.env.velocities[0], np.float64)
        batteries = self._tensor_numpy(self.env.battery[0], np.float64)
        active = self._tensor_numpy(self.env.active[0], bool)
        for index, uav in enumerate(self.uavs):
            uav.position = positions[index].copy()
            uav.velocity = velocities[index].copy()
            uav.battery_j = float(batteries[index])
            uav.active = bool(active[index])

        valid = self._tensor_numpy(self.env.report_valid[0], bool)
        remaining = self._tensor_numpy(self.env.report_remaining[0], np.float64)
        created = self._tensor_numpy(self.env.report_created[0], np.int64)
        source = self._tensor_numpy(self.env.report_source[0], np.int64)
        report_size = int(CONFIG["report_bytes"])
        ttl_s = float(CONFIG["report_ttl"])
        for holder in range(self.env.num_uavs):
            buffer = self.report_buffers[holder]
            buffer.clear()
            for target in range(self.env.num_targets):
                if not valid[holder, target]:
                    continue
                delivered_bytes = int(round(report_size - float(remaining[holder, target])))
                delivered_bytes = max(0, min(report_size, delivered_bytes))
                buffer.append(
                    Report(
                        target_id=int(target),
                        source_uav=int(source[holder, target]),
                        created_step=int(created[holder, target]),
                        size_bytes=report_size,
                        ttl_s=ttl_s,
                        delivered_bytes=delivered_bytes,
                    )
                )

        pvalid = self._tensor_numpy(self.env.pending_valid[0], bool)
        premaining = self._tensor_numpy(self.env.pending_remaining[0], np.float64)
        pcreated = self._tensor_numpy(self.env.pending_created[0], np.int64)
        self.pending_reports.clear()
        for holder in range(self.env.num_uavs):
            for target in range(self.env.num_targets):
                if not pvalid[holder, target]:
                    continue
                delivered_bytes = int(round(report_size - float(premaining[holder, target])))
                delivered_bytes = max(0, min(report_size, delivered_bytes))
                self.pending_reports.append(
                    Report(
                        target_id=int(target),
                        source_uav=int(holder),
                        created_step=int(pcreated[holder, target]),
                        size_bytes=report_size,
                        ttl_s=ttl_s,
                        delivered_bytes=delivered_bytes,
                    )
                )
        self.pending_reports.sort(
            key=lambda report: (
                int(report.created_step),
                int(report.source_uav),
                int(report.target_id),
            )
        )

        delivered = self._tensor_numpy(self.env.delivered[0], bool)
        self.gcs_received_target_ids.clear()
        self.gcs_received_target_ids.update(
            int(index) for index, flag in enumerate(delivered) if flag
        )

    def _sync_cpu_to_gpu(self):
        env = self.env
        device = env.device
        valid = torch.zeros_like(env.report_valid)
        remaining = torch.zeros_like(env.report_remaining)
        created = torch.full_like(
            env.report_created,
            env.max_steps + int(CONFIG["report_ttl"]) + 1,
        )
        source = torch.full_like(env.report_source, -1)
        for holder, buffer in enumerate(self.report_buffers):
            for report in buffer:
                target = int(report.target_id)
                valid[0, holder, target] = True
                remaining[0, holder, target] = float(
                    max(0, int(report.size_bytes) - int(report.delivered_bytes))
                )
                created[0, holder, target] = int(report.created_step)
                source[0, holder, target] = int(report.source_uav)

        pvalid = torch.zeros_like(env.pending_valid)
        premaining = torch.zeros_like(env.pending_remaining)
        pcreated = torch.full_like(
            env.pending_created,
            env.max_steps + int(CONFIG["report_ttl"]) + 1,
        )
        for report in self.pending_reports:
            holder = int(report.source_uav)
            target = int(report.target_id)
            if not (0 <= holder < env.num_uavs):
                continue
            if pvalid[0, holder, target]:
                continue
            pvalid[0, holder, target] = True
            premaining[0, holder, target] = float(
                max(0, int(report.size_bytes) - int(report.delivered_bytes))
            )
            pcreated[0, holder, target] = int(report.created_step)

        delivered = torch.zeros_like(env.delivered)
        for target in self.gcs_received_target_ids:
            if 0 <= int(target) < env.num_targets:
                delivered[0, int(target)] = True

        env.report_valid = valid.to(device)
        env.report_remaining = remaining.to(device)
        env.report_created = created.to(device)
        env.report_source = source.to(device)
        env.pending_valid = pvalid.to(device)
        env.pending_remaining = premaining.to(device)
        env.pending_created = pcreated.to(device)
        env.delivered = delivered.to(device)

    def reset_from_env(self):
        if self.backend is not None:
            self.backend.close()
        self._build_mission_objects()
        self.backend = self.ns["UavNetSimBackend"](
            repo_path=self.repo_path,
            seed=self.seed,
        )
        self.backend.reset(
            uavs=self.uavs,
            gcs_position=CONFIG["gcs_position"],
            obstacles=self.obstacles,
            report_buffers=self.report_buffers,
            pending_reports=self.pending_reports,
            gcs_received_target_ids=self.gcs_received_target_ids,
        )
        self._last_results = []
        self._last_metrics = self.backend.metrics()
        self._timing_totals = {
            "sync_gpu_to_cpu_seconds": 0.0,
            "uavnetsim_step_seconds": 0.0,
            "sync_cpu_to_gpu_seconds": 0.0,
            "calls": 0,
        }

    def step(self, destination_idx, power_action):
        if self.backend is None:
            raise RuntimeError("GpuUavNetSimBridge must be reset before step")
        timing_start = time.perf_counter()
        self._sync_gpu_to_cpu()
        after_gpu_to_cpu = time.perf_counter()
        np = self.np
        current_step = int(self.env.step_count[0].detach().cpu().item())
        destinations = self._tensor_numpy(destination_idx[0], np.int64)
        powers = self._tensor_numpy(power_action[0, :, 0], np.float64)

        HybridAction = self.ns["HybridAction"]
        build_intent = self.ns["build_transmission_intent"]
        intents = []
        sender_for_result = []
        for sender in range(self.env.num_uavs):
            choice_index = int(destinations[sender])
            if not 0 <= choice_index < self.env.discrete_dim:
                raise ValueError("destination index outside communication choices")
            destination = int(
                self.env.choices[sender, choice_index].detach().cpu().item()
            )
            if destination == -2:
                decoded_destination = None
                tx_power_w = 0.0
            else:
                decoded_destination = destination
                tx_power_w = (
                    float(CONFIG["tx_power_min_w"])
                    + (float(max(-1.0, min(1.0, powers[sender]))) + 1.0)
                    * 0.5
                    * (
                        float(CONFIG["tx_power_max_w"])
                        - float(CONFIG["tx_power_min_w"])
                    )
                )
            hybrid_action = HybridAction(
                movement=np.zeros(3, dtype=np.float64),
                destination=decoded_destination,
                tx_power_w=float(tx_power_w),
            )
            if decoded_destination is None:
                continue
            intent = build_intent(
                sender,
                hybrid_action,
                self.report_buffers[sender],
                current_step,
                peer_transfer_states=self.backend.peer_transfer_states,
            )
            if intent is not None:
                intents.append(intent)
                sender_for_result.append(sender)

        before_delivered = set(self.gcs_received_target_ids)
        self.backend.sync_positions(self.uavs)
        results = self.backend.step(
            intents,
            dt=self.env.dt,
            current_step=current_step,
        )
        after_uavnetsim = time.perf_counter()
        communication_energy = (
            self.backend.last_step_communication_energy_by_uav()
        )
        attempted = torch.zeros(
            1, self.env.num_uavs, device=self.env.device, dtype=torch.bool
        )
        bytes_tx = torch.zeros(
            1, self.env.num_uavs, device=self.env.device
        )
        for sender, result in zip(sender_for_result, results):
            attempted[0, sender] = True
            bytes_tx[0, sender] = float(result.get("tx_bytes", 0))

        self._sync_cpu_to_gpu()
        after_cpu_to_gpu = time.perf_counter()
        self._timing_totals["sync_gpu_to_cpu_seconds"] += (
            after_gpu_to_cpu - timing_start
        )
        self._timing_totals["uavnetsim_step_seconds"] += (
            after_uavnetsim - after_gpu_to_cpu
        )
        self._timing_totals["sync_cpu_to_gpu_seconds"] += (
            after_cpu_to_gpu - after_uavnetsim
        )
        self._timing_totals["calls"] += 1
        after_delivered = set(self.gcs_received_target_ids)
        new_delivery = len(after_delivered - before_delivered)
        comm_energy = torch.as_tensor(
            communication_energy,
            device=self.env.device,
            dtype=self.env.positions.dtype,
        ).view(1, self.env.num_uavs)
        self._last_results = results
        self._last_metrics = self.backend.metrics()
        return {
            "new_delivery": torch.tensor(
                [new_delivery], device=self.env.device, dtype=torch.long
            ),
            "comm_energy": comm_energy,
            "attempted": attempted,
            "bytes_tx": bytes_tx,
        }

    def metrics(self):
        if self.backend is None:
            return {}
        self._last_metrics = self.backend.metrics()
        return dict(self._last_metrics)

    def timing_metrics(self):
        result = dict(self._timing_totals)
        calls = max(1, int(result["calls"]))
        total = (
            float(result["sync_gpu_to_cpu_seconds"])
            + float(result["uavnetsim_step_seconds"])
            + float(result["sync_cpu_to_gpu_seconds"])
        )
        result["bridge_total_seconds"] = total
        result["uavnetsim_fraction"] = (
            float(result["uavnetsim_step_seconds"]) / total
            if total > 0.0
            else 0.0
        )
        result["avg_uavnetsim_step_seconds"] = (
            float(result["uavnetsim_step_seconds"]) / calls
        )
        return result

    def close(self):
        if self.backend is not None:
            self.backend.close()
            self.backend = None


def copy_cpu_reference_state_to_gpu(cpu_env, gpu_env):
    """Copy one CPU mission state into one tensor env for apples-to-apples runs."""
    if gpu_env.num_envs != 1:
        raise ValueError("copy_cpu_reference_state_to_gpu requires num_envs=1")
    device = gpu_env.device
    np = __import__("numpy")

    positions = np.stack([uav.position for uav in cpu_env.uavs])
    velocities = np.stack([uav.velocity for uav in cpu_env.uavs])
    battery = np.asarray([uav.battery_j for uav in cpu_env.uavs], dtype=np.float32)
    active = np.asarray([uav.active for uav in cpu_env.uavs], dtype=bool)
    targets = np.stack([target.position for target in cpu_env.targets])
    obstacles = np.stack([obstacle.position for obstacle in cpu_env.obstacles])
    obstacle_radius = np.asarray(
        [obstacle.radius for obstacle in cpu_env.obstacles], dtype=np.float32
    )
    obstacle_height = np.asarray(
        [obstacle.height for obstacle in cpu_env.obstacles], dtype=np.float32
    )

    gpu_env.positions = torch.as_tensor(
        positions, device=device, dtype=gpu_env.positions.dtype
    ).unsqueeze(0)
    gpu_env.velocities = torch.as_tensor(
        velocities, device=device, dtype=gpu_env.velocities.dtype
    ).unsqueeze(0)
    gpu_env.battery = torch.as_tensor(
        battery, device=device, dtype=gpu_env.battery.dtype
    ).unsqueeze(0)
    gpu_env.active = torch.as_tensor(
        active, device=device, dtype=torch.bool
    ).unsqueeze(0)
    gpu_env.target_xy = torch.as_tensor(
        targets, device=device, dtype=gpu_env.target_xy.dtype
    ).unsqueeze(0)
    gpu_env.obstacle_xy = torch.as_tensor(
        obstacles, device=device, dtype=gpu_env.obstacle_xy.dtype
    ).unsqueeze(0)
    gpu_env.obstacle_radius = torch.as_tensor(
        obstacle_radius, device=device, dtype=gpu_env.obstacle_radius.dtype
    ).unsqueeze(0)
    gpu_env.obstacle_height = torch.as_tensor(
        obstacle_height, device=device, dtype=gpu_env.obstacle_height.dtype
    ).unsqueeze(0)
    gpu_env.confirmed = torch.as_tensor(
        [[target.confirmed for target in cpu_env.targets]],
        device=device,
        dtype=torch.bool,
    )
    gpu_env.delivered.zero_()
    for target in cpu_env.gcs_received_target_ids:
        gpu_env.delivered[0, int(target)] = True
    gpu_env.belief = torch.as_tensor(
        cpu_env.belief_maps,
        device=device,
        dtype=gpu_env.belief.dtype,
    ).unsqueeze(0)
    gpu_env.step_count.fill_(int(cpu_env.current_step))
    gpu_env.report_valid.zero_()
    gpu_env.report_remaining.zero_()
    gpu_env.report_created.fill_(
        gpu_env.max_steps + int(CONFIG["report_ttl"]) + 1
    )
    gpu_env.report_source.fill_(-1)
    for holder, buffer in enumerate(cpu_env.report_buffers):
        for report in buffer:
            target = int(report.target_id)
            gpu_env.report_valid[0, holder, target] = True
            gpu_env.report_remaining[0, holder, target] = float(
                max(0, int(report.size_bytes) - int(report.delivered_bytes))
            )
            gpu_env.report_created[0, holder, target] = int(report.created_step)
            gpu_env.report_source[0, holder, target] = int(report.source_uav)
    gpu_env.pending_valid.zero_()
    gpu_env.pending_remaining.zero_()
    gpu_env.pending_created.fill_(
        gpu_env.max_steps + int(CONFIG["report_ttl"]) + 1
    )
    for report in cpu_env.pending_reports:
        holder = int(report.source_uav)
        target = int(report.target_id)
        if 0 <= holder < gpu_env.num_uavs:
            gpu_env.pending_valid[0, holder, target] = True
            gpu_env.pending_remaining[0, holder, target] = float(
                max(0, int(report.size_bytes) - int(report.delivered_bytes))
            )
            gpu_env.pending_created[0, holder, target] = int(report.created_step)
    gpu_env.peer_received.zero_()
    gpu_env.peer_source.fill_(-1)
    gpu_env.peer_created.fill_(
        gpu_env.max_steps + int(CONFIG["report_ttl"]) + 1
    )
    gpu_env.belief_potential = gpu_env._belief_potential()
    obs, mask = gpu_env._observation()
    state = gpu_env._state(obs)
    return obs, state, mask


def benchmark_uavnetsim_full_flow(
    repo: Path,
    *,
    repo_path,
    steps=20,
    seed=44,
    device="cuda:0",
    strict_cuda=True,
    communication_loaded=True,
):
    """CPU-UavNetSim vs tensor-env+UavNetSim using one scenario and action trace."""
    project_ns = load_project_namespace(repo)
    if not project_ns["uavnetsim_available"](repo_path):
        raise FileNotFoundError("UavNetSim repository is unavailable")

    CPUEnv = project_ns["UAVSearchEnv"]
    cpu_env = CPUEnv(
        backend_name="uavnetsim",
        network_backend_kwargs={"repo_path": repo_path},
    )
    cpu_env.reset(seed=int(seed))
    if communication_loaded:
        Report = project_ns["Report"]
        cpu_env.targets[0].confirmed = True
        cpu_env.report_buffers[0].clear()
        cpu_env.report_buffers[0].append(
            Report(
                target_id=0,
                source_uav=0,
                created_step=0,
                size_bytes=int(CONFIG["report_bytes"]),
                ttl_s=float(CONFIG["report_ttl"]),
                delivered_bytes=0,
            )
        )

    gpu_env = FullGpuUAVBatchEnv(
        num_envs=1,
        device=device,
        seed=int(seed),
        strict_cuda=bool(strict_cuda),
    )
    copy_cpu_reference_state_to_gpu(cpu_env, gpu_env)
    bridge = GpuUavNetSimBridge(
        gpu_env,
        project_ns,
        repo_path=repo_path,
        seed=int(seed),
    )
    gpu_env.attach_external_network(bridge)

    rng = project_ns["np"].random.default_rng(int(seed) + 99173)
    actions = rng.uniform(
        -1.0,
        1.0,
        size=(int(steps), gpu_env.num_uavs, gpu_env.continuous_dim),
    ).astype("float32")
    destinations = rng.integers(
        0,
        gpu_env.discrete_dim,
        size=(int(steps), gpu_env.num_uavs),
        endpoint=False,
        dtype="int64",
    )
    if communication_loaded:
        actions[:, 0, 3] = 1.0
        destinations[:, 0] = gpu_env.num_uavs

    def cpu_action_at(index):
        result = {}
        for sender, agent_id in enumerate(cpu_env.agent_ids):
            result[agent_id] = {
                "motion": actions[index, sender, :3].copy(),
                "destination": int(destinations[index, sender]),
                "power": project_ns["np"].array(
                    [actions[index, sender, 3]],
                    dtype=project_ns["np"].float32,
                ),
            }
        return result

    cpu_rewards = []
    cpu_start = time.perf_counter()
    cpu_steps = 0
    for index in range(int(steps)):
        out = cpu_env.step(cpu_action_at(index))
        cpu_rewards.append(float(out[1]))
        cpu_steps += 1
        if bool(out[2] or out[3]):
            break
    cpu_wall = time.perf_counter() - cpu_start
    cpu_network = cpu_env.network_backend.metrics()

    gpu_rewards = []
    if gpu_env.device.type == "cuda":
        torch.cuda.synchronize(gpu_env.device)
    gpu_start = time.perf_counter()
    gpu_steps = 0
    for index in range(int(steps)):
        continuous = torch.as_tensor(
            actions[index:index + 1],
            device=gpu_env.device,
            dtype=torch.float32,
        )
        destination = torch.as_tensor(
            destinations[index:index + 1],
            device=gpu_env.device,
            dtype=torch.long,
        )
        out = gpu_env.step(continuous, destination)
        gpu_rewards.append(float(out[3][0].detach().cpu().item()))
        gpu_steps += 1
        if bool(out[4][0].detach().cpu().item()):
            break
    if gpu_env.device.type == "cuda":
        torch.cuda.synchronize(gpu_env.device)
    gpu_wall = time.perf_counter() - gpu_start
    gpu_network = bridge.metrics()

    np = project_ns["np"]
    cpu_positions = np.stack(
        [uav.position for uav in cpu_env.uavs]
    )
    gpu_positions = (
        gpu_env.positions[0]
        .detach()
        .cpu()
        .numpy()
    )
    cpu_belief = np.asarray(
        cpu_env.belief_maps,
        dtype=np.float32,
    )
    gpu_belief = (
        gpu_env.belief[0]
        .detach()
        .cpu()
        .numpy()
    )
    common_steps = min(
        len(cpu_rewards),
        len(gpu_rewards),
    )
    reward_abs_diff = [
        abs(
            float(cpu_rewards[index])
            - float(gpu_rewards[index])
        )
        for index in range(common_steps)
    ]
    network_compare_keys = (
        "packets_injected",
        "packets_delivered",
        "packets_failed",
        "queue_limit_events",
        "payload_bytes_injected",
        "payload_bytes_delivered",
    )
    network_core_equal = all(
        cpu_network.get(key)
        == gpu_network.get(key)
        for key in network_compare_keys
    )
    final_cpu_confirmed = {
        int(target.id)
        for target in cpu_env.targets
        if target.confirmed
    }
    final_gpu_confirmed = {
        int(index)
        for index, flag in enumerate(
            gpu_env.confirmed[0]
            .detach()
            .cpu()
            .tolist()
        )
        if flag
    }
    final_cpu_delivered = set(
        int(value)
        for value in cpu_env.gcs_received_target_ids
    )
    final_gpu_delivered = {
        int(index)
        for index, flag in enumerate(
            gpu_env.delivered[0]
            .detach()
            .cpu()
            .tolist()
        )
        if flag
    }

    result = {
        "kind": "uavnetsim_full_flow_compare",
        "seed": int(seed),
        "requested_steps": int(steps),
        "same_initial_scenario": True,
        "same_action_trace": True,
        "parity": {
            "final_position_max_abs_diff_m": float(
                np.max(
                    np.abs(
                        cpu_positions
                        - gpu_positions
                    )
                )
            ),
            "final_belief_max_abs_diff": float(
                np.max(
                    np.abs(
                        cpu_belief
                        - gpu_belief
                    )
                )
            ),
            "reward_max_abs_diff": (
                float(max(reward_abs_diff))
                if reward_abs_diff
                else 0.0
            ),
            "network_core_equal": bool(
                network_core_equal
            ),
            "confirmed_equal": (
                final_cpu_confirmed
                == final_gpu_confirmed
            ),
            "delivered_equal": (
                final_cpu_delivered
                == final_gpu_delivered
            ),
            "cpu_confirmed": sorted(
                final_cpu_confirmed
            ),
            "gpu_confirmed": sorted(
                final_gpu_confirmed
            ),
            "cpu_delivered": sorted(
                final_cpu_delivered
            ),
            "gpu_delivered": sorted(
                final_gpu_delivered
            ),
        },
        "cpu": {
            "steps": cpu_steps,
            "wall_seconds": cpu_wall,
            "steps_per_second": cpu_steps / cpu_wall if cpu_wall > 0.0 else float("inf"),
            "reward_sum": float(sum(cpu_rewards)),
            "network": cpu_network,
        },
        "gpu_mission_plus_uavnetsim": {
            "steps": gpu_steps,
            "wall_seconds": gpu_wall,
            "steps_per_second": gpu_steps / gpu_wall if gpu_wall > 0.0 else float("inf"),
            "reward_sum": float(sum(gpu_rewards)),
            "network": gpu_network,
            "network_timing": bridge.timing_metrics(),
            "device": str(gpu_env.device),
        },
    }
    cpu_env.close()
    gpu_env.close()
    return result



def validate_uavnetsim_gpu_direct_link(
    repo: Path,
    *,
    repo_path=None,
    seed=44,
    steps=6,
    device="cpu",
    strict_cuda=False,
):
    """Controlled direct-to-GCS validation against the real SimPy backend.

    This is intentionally a single-link case: CSMA has no collision ambiguity,
    so queue cap, payload packetization, service budget, LoS/NLoS PHY contract,
    delivery timing, reward timing and report cleanup are directly comparable.
    """
    project_ns = load_project_namespace(repo)
    if not project_ns["uavnetsim_available"](repo_path):
        raise FileNotFoundError("UavNetSim repository is unavailable")
    np = project_ns["np"]
    CPUEnv = project_ns["UAVSearchEnv"]
    Report = project_ns["Report"]
    cpu_env = CPUEnv(
        backend_name="uavnetsim",
        network_backend_kwargs={"repo_path": repo_path},
    )
    cpu_env.reset(seed=int(seed))
    cpu_env.targets[0].confirmed = True
    cpu_env.report_buffers[0].clear()
    cpu_env.report_buffers[0].append(
        Report(
            target_id=0,
            source_uav=0,
            created_step=0,
            size_bytes=int(CONFIG["report_bytes"]),
            ttl_s=float(CONFIG["report_ttl"]),
            delivered_bytes=0,
        )
    )

    gpu_env = FullGpuUAVBatchEnv(
        num_envs=1,
        device=device,
        seed=int(seed),
        strict_cuda=bool(strict_cuda),
    )
    copy_cpu_reference_state_to_gpu(cpu_env, gpu_env)
    gpu_env.network_model = "uavnetsim_gpu"

    per_step = []
    try:
        for index in range(int(steps)):
            action = np.zeros(
                (gpu_env.num_uavs, gpu_env.continuous_dim),
                dtype=np.float32,
            )
            destination = np.zeros(
                gpu_env.num_uavs,
                dtype=np.int64,
            )
            action[0, 3] = 1.0
            destination[0] = gpu_env.num_uavs

            cpu_action = {}
            for sender, agent_id in enumerate(cpu_env.agent_ids):
                cpu_action[agent_id] = {
                    "motion": action[sender, :3].copy(),
                    "destination": int(destination[sender]),
                    "power": np.asarray(
                        [action[sender, 3]],
                        dtype=np.float32,
                    ),
                }
            cpu_out = cpu_env.step(cpu_action)
            gpu_out = gpu_env.step(
                torch.as_tensor(
                    action[None],
                    device=gpu_env.device,
                    dtype=torch.float32,
                ),
                torch.as_tensor(
                    destination[None],
                    device=gpu_env.device,
                    dtype=torch.long,
                ),
            )
            cpu_network = cpu_env.network_backend.metrics()
            gpu_delivered = bool(gpu_env.delivered[0, 0].item())
            gpu_remaining = float(
                gpu_env.report_remaining[0, 0, 0].item()
            )
            gpu_delivered_bytes = int(
                round(float(CONFIG["report_bytes"]) - gpu_remaining)
            )
            per_step.append(
                {
                    "step": index + 1,
                    "cpu_payload_bytes_delivered": int(
                        cpu_network["payload_bytes_delivered"]
                    ),
                    "gpu_payload_bytes_delivered": gpu_delivered_bytes,
                    "cpu_delivered": 0 in cpu_env.gcs_received_target_ids,
                    "gpu_delivered": gpu_delivered,
                    "cpu_reward": float(cpu_out[1]),
                    "gpu_reward": float(gpu_out[3][0].item()),
                }
            )

        byte_parity = all(
            row["cpu_payload_bytes_delivered"]
            == row["gpu_payload_bytes_delivered"]
            for row in per_step
        )
        delivery_parity = all(
            row["cpu_delivered"] == row["gpu_delivered"]
            for row in per_step
        )
        reward_max_abs_diff = max(
            (
                abs(row["cpu_reward"] - row["gpu_reward"])
                for row in per_step
            ),
            default=0.0,
        )
        return {
            "passed": bool(
                byte_parity
                and delivery_parity
                and reward_max_abs_diff < 1e-4
            ),
            "byte_parity": bool(byte_parity),
            "delivery_parity": bool(delivery_parity),
            "reward_max_abs_diff": float(reward_max_abs_diff),
            "per_step": per_step,
            "cpu_network_metrics": cpu_env.network_backend.metrics(),
        }
    finally:
        cpu_env.close()
        gpu_env.close()


def validate_uavnetsim_network_parity(
    repo: Path,
    *,
    repo_path,
    seed=44,
    device="cpu",
    strict_cuda=False,
    steps=4,
):
    """Drive two independent UavNetSim backends with identical state/intents."""
    project_ns = load_project_namespace(repo)
    CPUEnv = project_ns["UAVSearchEnv"]
    cpu_env = CPUEnv(
        backend_name="uavnetsim",
        network_backend_kwargs={"repo_path": repo_path},
    )
    cpu_env.reset(seed=int(seed))
    Report = project_ns["Report"]
    cpu_env.targets[0].confirmed = True
    cpu_env.report_buffers[0].append(
        Report(
            target_id=0,
            source_uav=0,
            created_step=0,
            size_bytes=int(CONFIG["report_bytes"]),
            ttl_s=float(CONFIG["report_ttl"]),
            delivered_bytes=0,
        )
    )

    gpu_env = FullGpuUAVBatchEnv(
        1,
        device=device,
        seed=int(seed),
        strict_cuda=bool(strict_cuda),
    )
    copy_cpu_reference_state_to_gpu(cpu_env, gpu_env)
    bridge = GpuUavNetSimBridge(
        gpu_env,
        project_ns,
        repo_path=repo_path,
        seed=int(seed),
    )
    gpu_env.attach_external_network(bridge)

    destination = torch.zeros(
        1, gpu_env.num_uavs, device=gpu_env.device, dtype=torch.long
    )
    destination[0, 0] = gpu_env.num_uavs
    power = torch.zeros(
        1, gpu_env.num_uavs, 1, device=gpu_env.device
    )
    power[0, 0, 0] = 1.0

    per_step = []
    for current_step in range(1, int(steps) + 1):
        cpu_env.current_step = current_step - 1
        gpu_env.step_count.fill_(current_step)
        action = {}
        for sender, agent_id in enumerate(cpu_env.agent_ids):
            action[agent_id] = {
                "motion": project_ns["np"].zeros(
                    3, dtype=project_ns["np"].float32
                ),
                "destination": gpu_env.num_uavs if sender == 0 else 0,
                "power": project_ns["np"].array(
                    [1.0 if sender == 0 else 0.0],
                    dtype=project_ns["np"].float32,
                ),
            }
        hybrid = cpu_env._decode_actions(cpu_env._canonical_actions(action))
        intents = []
        peer_states = cpu_env.network_backend.peer_transfer_states
        for sender, haction in enumerate(hybrid):
            intent = project_ns["build_transmission_intent"](
                sender,
                haction,
                cpu_env.report_buffers[sender],
                current_step,
                peer_transfer_states=peer_states,
            )
            if intent is not None:
                intents.append(intent)
        cpu_env.network_backend.sync_positions(cpu_env.uavs)
        cpu_results = cpu_env.network_backend.step(
            intents,
            dt=float(CONFIG["dt"]),
            current_step=current_step,
        )
        gpu_result = bridge.step(destination, power)
        cpu_metrics = cpu_env.network_backend.metrics()
        gpu_metrics = bridge.metrics()
        comparable = {
            "packets_injected": (
                cpu_metrics["packets_injected"],
                gpu_metrics["packets_injected"],
            ),
            "packets_delivered": (
                cpu_metrics["packets_delivered"],
                gpu_metrics["packets_delivered"],
            ),
            "packets_failed": (
                cpu_metrics["packets_failed"],
                gpu_metrics["packets_failed"],
            ),
            "payload_bytes_delivered": (
                cpu_metrics["payload_bytes_delivered"],
                gpu_metrics["payload_bytes_delivered"],
            ),
            "queue_limit_events": (
                cpu_metrics["queue_limit_events"],
                gpu_metrics["queue_limit_events"],
            ),
        }
        equal = all(left == right for left, right in comparable.values())
        per_step.append(
            {
                "step": current_step,
                "equal": equal,
                "metrics": comparable,
                "cpu_results": cpu_results,
                "gpu_bytes_tx": gpu_result["bytes_tx"].detach().cpu().tolist(),
            }
        )
        if not equal:
            break

    passed = all(item["equal"] for item in per_step)
    cpu_env.close()
    gpu_env.close()
    return {
        "passed": passed,
        "steps_checked": len(per_step),
        "per_step": per_step,
    }



def _rebuild_cuda_optimizers_with_fused_adam(
    trainer,
    algorithm,
):
    """Opt-in benchmark path; safe only before the first optimizer step."""
    if trainer.device.type != "cuda":
        return False
    algorithm = str(algorithm).strip().lower()
    if algorithm == "masac":
        trainer.actor_optimizer = torch.optim.Adam(
            trainer.actor.parameters(),
            lr=float(CONFIG["masac_actor_lr"]),
            fused=True,
        )
        trainer.critic_optimizer = torch.optim.Adam(
            list(trainer.critic_1.parameters())
            + list(trainer.critic_2.parameters()),
            lr=float(CONFIG["masac_critic_lr"]),
            fused=True,
        )
        trainer.alpha_continuous_optimizer = torch.optim.Adam(
            [trainer.log_alpha_continuous],
            lr=float(CONFIG["masac_alpha_lr"]),
            fused=True,
        )
        trainer.alpha_discrete_optimizer = torch.optim.Adam(
            [trainer.log_alpha_discrete],
            lr=float(CONFIG["masac_alpha_lr"]),
            fused=True,
        )
    elif algorithm == "matd3":
        trainer.actor_optimizer = torch.optim.Adam(
            trainer.actor.parameters(),
            lr=float(CONFIG["matd3_actor_lr"]),
            fused=True,
        )
        trainer.critic_optimizer = torch.optim.Adam(
            list(trainer.critic_1.parameters())
            + list(trainer.critic_2.parameters()),
            lr=float(CONFIG["matd3_critic_lr"]),
            fused=True,
        )
    else:
        raise ValueError("unsupported algorithm")
    return True


def _compile_trainer_modules(
    trainer,
    algorithm,
    mode="default",
):
    """Compile fixed-shape actor/critic modules in-place for benchmarking."""
    if trainer.device.type != "cuda":
        return []
    names = [
        "actor",
        "critic_1",
        "critic_2",
        "target_critic_1",
        "target_critic_2",
    ]
    if str(algorithm).strip().lower() == "matd3":
        names.append("target_actor")
    compiled = []
    for name in names:
        module = getattr(trainer, name, None)
        if isinstance(module, torch.nn.Module):
            module.compile(
                mode=str(mode),
                fullgraph=False,
            )
            compiled.append(name)
    return compiled


def train_full_gpu(
    repo: Path,
    algorithm: str,
    num_envs: int,
    total_transitions: int,
    seed: int,
    device: str,
    network_backend: str = "simple",
    uavnetsim_repo_path=None,
    strict_cuda: bool = True,
    compile_mode=None,
    fused_adam: bool = False,
    update_to_data_ratio=None,
    enable_wandb: bool = False,
):
    algorithm = str(algorithm).strip().lower()
    if algorithm not in {"masac", "matd3"}:
        raise ValueError(
            "algorithm must be masac or matd3"
        )
    if int(total_transitions) < 1:
        raise ValueError(
            "total_transitions must be >= 1"
        )
    network_backend = str(network_backend).strip().lower()
    if network_backend not in {
        "simple",
        "packet_approx",
        "uavnetsim_gpu",
        "uavnetsim",
    }:
        raise ValueError(
            "network_backend must be simple, packet_approx, uavnetsim_gpu, or uavnetsim"
        )
    if network_backend == "uavnetsim" and int(num_envs) != 1:
        raise ValueError(
            "UavNetSim packet-level training currently requires num_envs=1"
        )

    torch.use_deterministic_algorithms(True)
    random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

    env = FullGpuUAVBatchEnv(
        num_envs=num_envs,
        device=device,
        seed=seed,
        strict_cuda=bool(strict_cuda),
    )
    if network_backend in {
        "simple",
        "packet_approx",
        "uavnetsim_gpu",
    }:
        env.network_model = network_backend
    if env.device.type == "cuda":
        env.assert_all_state_on_cuda()
    trainer = _trainer(
        repo,
        algorithm,
        device,
        seed,
    )
    wandb_run = _init_gpu_wandb_run(
        algorithm,
        seed,
        enabled=enable_wandb,
    )
    wandb_last_log_step = -1
    fused_adam_enabled = False
    if bool(fused_adam):
        fused_adam_enabled = (
            _rebuild_cuda_optimizers_with_fused_adam(
                trainer,
                algorithm,
            )
        )
    compiled_modules = []
    if compile_mode is not None:
        compiled_modules = _compile_trainer_modules(
            trainer,
            algorithm,
            mode=compile_mode,
        )
    if network_backend == "uavnetsim":
        bridge = GpuUavNetSimBridge(
            env,
            trainer._project_namespace,
            repo_path=uavnetsim_repo_path,
            seed=seed,
        )
        env.attach_external_network(bridge)
    else:
        bridge = None
    if (
        env.observation_dim
        != trainer.observation_dim
        or env.state_dim
        != trainer.state_dim
        or env.discrete_dim
        != trainer.discrete_dim
    ):
        raise RuntimeError(
            "full-GPU environment dimensions do not match "
            "the CPU-reference learner contract"
        )

    capacity = min(
        max(
            int(
                CONFIG[
                    f"{algorithm}_replay_capacity"
                ]
            ),
            4096,
        ),
        max(
            int(total_transitions),
            4096,
        ),
    )
    replay = make_project_gpu_replay_buffer(
        trainer,
        capacity=capacity,
        num_agents=env.num_uavs,
        observation_dim=env.observation_dim,
        state_dim=env.state_dim,
        continuous_dim=env.continuous_dim,
        discrete_dim=env.discrete_dim,
        device=device,
        seed=seed + 123,
        strict_cuda=bool(strict_cuda),
    )
    batch_size = int(
        CONFIG[f"{algorithm}_batch_size"]
    )
    learning_starts = int(
        CONFIG[
            f"{algorithm}_learning_starts"
        ]
    )
    if update_to_data_ratio is None:
        update_to_data_ratio = float(
            CONFIG[
                f"{algorithm}_updates_per_step"
            ]
        )
    else:
        update_to_data_ratio = float(
            update_to_data_ratio
        )
    if update_to_data_ratio < 0.0:
        raise ValueError(
            "update_to_data_ratio must be >= 0"
        )
    update_credit = 0.0

    obs, state, mask = env.reset()
    global_step = 0
    update_count = 0
    latest_metrics = None
    completed_episodes = torch.zeros(
        num_envs,
        device=env.device,
        dtype=torch.long,
    )
    transition_offsets = torch.arange(
        num_envs,
        device=env.device,
        dtype=torch.long,
    )
    total_transitions_int = int(
        total_transitions
    )

    if env.device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(
            env.device
        )
        torch.cuda.synchronize(env.device)
    start = time.perf_counter()

    while global_step < total_transitions_int:
        transition_steps = (
            transition_offsets
            + global_step
            + 1
        )
        warmup = (
            transition_steps
            <= learning_starts
        )

        batch_first_step = global_step + 1
        batch_last_step = global_step + num_envs
        all_warmup = batch_last_step <= learning_starts
        all_policy = batch_first_step > learning_starts

        if all_warmup:
            continuous, destination = (
                _random_exploration_actions(
                    env,
                    mask,
                )
            )
        elif all_policy:
            if algorithm == "masac":
                with torch.no_grad():
                    sampled = (
                        trainer._sample_joint_policy(
                            obs,
                            mask,
                            deterministic=False,
                        )
                    )
                    continuous = sampled[
                        "continuous"
                    ]
                    destination = sampled[
                        "destination_indices"
                    ]
            else:
                continuous, destination = (
                    _matd3_actions(
                        trainer,
                        obs,
                        mask,
                        transition_steps,
                        env.generator,
                    )
                )
        else:
            # Only one vector batch can straddle learning_starts. In that
            # boundary batch both paths are needed to preserve per-transition
            # semantics exactly.
            random_continuous, random_destination = (
                _random_exploration_actions(
                    env,
                    mask,
                )
            )
            if algorithm == "masac":
                with torch.no_grad():
                    sampled = (
                        trainer._sample_joint_policy(
                            obs,
                            mask,
                            deterministic=False,
                        )
                    )
                    policy_continuous = sampled[
                        "continuous"
                    ]
                    policy_destination = sampled[
                        "destination_indices"
                    ]
            else:
                (
                    policy_continuous,
                    policy_destination,
                ) = _matd3_actions(
                    trainer,
                    obs,
                    mask,
                    transition_steps,
                    env.generator,
                )
            continuous = torch.where(
                warmup[:, None, None],
                random_continuous,
                policy_continuous,
            )
            destination = torch.where(
                warmup[:, None],
                random_destination,
                policy_destination,
            )

        (
            next_obs,
            next_state,
            next_mask,
            reward,
            done,
            _,
        ) = env.step(
            continuous,
            destination,
        )
        take = min(
            num_envs,
            int(total_transitions)
            - global_step,
        )
        replay.add_batch(
            obs[:take],
            state[:take],
            continuous[:take],
            destination[:take],
            mask[:take],
            reward[:take],
            next_obs[:take],
            next_state[:take],
            next_mask[:take],
            done[:take],
        )
        previous = global_step
        global_step += take

        if len(replay) >= batch_size:
            first_train_step = max(
                previous,
                learning_starts,
            )
            eligible = max(
                0,
                global_step - first_train_step,
            )
            update_credit += (
                eligible
                * update_to_data_ratio
            )
            updates_this_step = int(
                update_credit
            )
            update_credit -= (
                updates_this_step
            )
            for local_update in range(
                updates_this_step
            ):
                collected = trainer.update(
                    replay,
                    batch_size=batch_size,
                    collect_metrics=(
                        local_update
                        == updates_this_step - 1
                    ),
                )
                if collected is not None:
                    latest_metrics = collected
                update_count += 1

        if (
            wandb_run is not None
            and (
                global_step - wandb_last_log_step
                >= max(1, int(CONFIG.get("training_log_interval_steps", 100)))
                or global_step >= total_transitions_int
            )
        ):
            elapsed_for_log = max(
                1e-12, time.perf_counter() - start
            )
            payload = {
                "global_step": int(global_step),
                "train/transitions_per_second": (
                    global_step / elapsed_for_log
                ),
                "train/updates_per_second": (
                    update_count / elapsed_for_log
                ),
                "system/gpu_count": 1.0,
            }
            if latest_metrics is not None:
                for key, value in latest_metrics.items():
                    if isinstance(value, torch.Tensor):
                        if value.numel() != 1:
                            continue
                        value = value.detach().cpu().item()
                    if isinstance(value, (bool, int, float)):
                        payload[f"train/{key}"] = float(value)
            wandb_run.log(payload)
            wandb_last_log_step = global_step

        # A scalar host check only orchestrates rare episode reset. State
        # generation/reset and all environment math remain CUDA tensor work.
        if bool(done.any().item()):
            completed_episodes += (
                done.long()
            )
            if global_step < int(
                total_transitions
            ):
                obs, state, mask = env.reset(
                    mask=done
                )
            else:
                obs, state, mask = (
                    next_obs,
                    next_state,
                    next_mask,
                )
        else:
            obs, state, mask = (
                next_obs,
                next_state,
                next_mask,
            )

    if env.device.type == "cuda":
        torch.cuda.synchronize(env.device)
    wall = time.perf_counter() - start

    def metric_value(value):
        if isinstance(value, torch.Tensor):
            if value.numel() == 1:
                return float(
                    value.detach().cpu().item()
                )
            return None
        if isinstance(
            value,
            (int, float, bool),
        ):
            return value
        return None

    metrics = None
    if latest_metrics is not None:
        metrics = {
            key: metric_value(value)
            for key, value
            in latest_metrics.items()
            if metric_value(value)
            is not None
        }

    network_metrics = (
        bridge.metrics()
        if bridge is not None
        else None
    )
    result = {
        "kind": (
            "gpu_mission_uavnetsim_train"
            if network_backend == "uavnetsim"
            else (
                "full_gpu_uavnetsim_surrogate_train"
                if network_backend == "uavnetsim_gpu"
                else (
                    "full_gpu_packet_train"
                    if network_backend == "packet_approx"
                    else "full_gpu_train"
                )
            )
        ),
        "network_backend": network_backend,
        "algorithm": algorithm,
        "compile_mode": compile_mode,
        "compiled_modules": compiled_modules,
        "fused_adam": bool(
            fused_adam_enabled
        ),
        "num_envs": num_envs,
        "transitions": global_step,
        "updates": update_count,
        "update_to_data_ratio": float(
            update_to_data_ratio
        ),
        "episodes_completed": int(
            completed_episodes.sum()
            .detach()
            .cpu()
            .item()
        ),
        "wall_seconds": wall,
        "transitions_per_second": (
            global_step / wall
        ),
        "updates_per_second": (
            update_count / wall
        ),
        "gpu_name": (
            torch.cuda.get_device_name(
                env.device
            )
            if env.device.type == "cuda"
            else None
        ),
        "peak_memory_mb": (
            torch.cuda.max_memory_allocated(
                env.device
            )
            / 2**20
            if env.device.type == "cuda"
            else None
        ),
        "latest_update_metrics": metrics,
        "model_fingerprint_sha256": (
            _trainer_parameter_fingerprint(
                trainer
            )
        ),
        "all_env_state_cuda": (
            env.device.type == "cuda"
        ),
        "network_metrics": network_metrics,
    }
    result["wandb_run_id"] = (
        wandb_run.id if wandb_run is not None else None
    )
    result["wandb_run_name"] = (
        wandb_run.name if wandb_run is not None else None
    )
    result["wandb_run_url"] = (
        wandb_run.url if wandb_run is not None else None
    )
    if wandb_run is not None:
        wandb_run.summary["final/transitions_per_second"] = result[
            "transitions_per_second"
        ]
        wandb_run.summary["final/updates_per_second"] = result[
            "updates_per_second"
        ]
        wandb_run.summary["final/gpu_count"] = 1
        wandb_run.finish()
    env.close()
    return result






class _MASACDDPActorForward(torch.nn.Module):
    def __init__(self, actor, project_ns):
        super().__init__()
        self.actor = actor
        self._radial_squash = project_ns["radial_squash_motion_action"]
        self._categorical_sample = project_ns["straight_through_categorical_sample"]

    def forward(self, observations, destination_mask):
        mean, log_std, logits = self.actor(observations)
        std = torch.exp(log_std)
        distribution = torch.distributions.Normal(mean, std)
        pre_tanh = distribution.rsample()
        motion_action, motion_log_det = self._radial_squash(pre_tanh[..., :3])
        power_action = torch.tanh(pre_tanh[..., 3:4])
        continuous_action = torch.cat((motion_action, power_action), dim=-1)
        power_log_det = torch.log(
            1.0 - power_action.pow(2) + 1e-6
        ).sum(dim=-1, keepdim=True)
        log_probability = (
            distribution.log_prob(pre_tanh).sum(dim=-1, keepdim=True)
            - motion_log_det
            - power_log_det
        )
        discrete = self._categorical_sample(
            logits,
            destination_mask,
            temperature=CONFIG["masac_gumbel_temperature"],
            deterministic=False,
        )
        return (
            continuous_action,
            log_probability,
            discrete["action"],
            discrete["indices"],
            discrete["log_probability"],
            discrete["entropy"],
        )


class _MATD3DDPActorForward(torch.nn.Module):
    def __init__(self, actor, project_ns):
        super().__init__()
        self.actor = actor
        self._masked_argmax = project_ns["straight_through_masked_argmax"]

    def forward(self, observations, destination_mask):
        continuous, logits = self.actor(observations)
        discrete = self._masked_argmax(logits, destination_mask)
        return (
            continuous,
            discrete["action"],
            discrete["indices"],
            discrete["probabilities"],
        )


def _gpu_wandb_api_key_from_kaggle():
    if os.environ.get("WANDB_API_KEY"):
        return os.environ["WANDB_API_KEY"]
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for secret_name in CONFIG.get("kaggle_wandb_secret_names", ()):
            try:
                value = client.get_secret(str(secret_name))
            except Exception:
                continue
            if value:
                os.environ["WANDB_API_KEY"] = str(value)
                return str(value)
    except Exception:
        pass
    return None


def _init_gpu_wandb_run(algorithm, seed, enabled=None):
    if enabled is None:
        enabled = bool(CONFIG.get("training_enable_wandb", True))
    if not enabled:
        return None
    mode = str(CONFIG.get("training_wandb_mode", "online"))
    if mode == "online" and not os.environ.get("WANDB_API_KEY"):
        _gpu_wandb_api_key_from_kaggle()
    wandb = importlib.import_module("wandb")
    import secrets
    run_id = secrets.token_hex(4)
    name_template = str(
        CONFIG.get(
            "training_wandb_name_format",
            "{algorithm}-seed{seed}-{id}",
        )
    )
    run_name = name_template.format(
        algorithm=str(algorithm).upper(),
        seed=int(seed),
        id=run_id,
    )
    try:
        run = wandb.init(
            entity=CONFIG.get("training_wandb_entity"),
            project=str(CONFIG["training_wandb_project"]),
            name=run_name,
            id=run_id,
            mode=mode,
            reinit="finish_previous",
            save_code=False,
            tags=None,
            group=None,
            job_type=None,
        )
    except Exception:
        if bool(CONFIG.get("training_wandb_required", True)):
            raise
        return None
    return run


def _ddp_actor_sample_masac(trainer, actor_ddp, observations, masks):
    batch_size = int(observations.shape[0])
    flat_obs = observations.reshape(
        batch_size * trainer.num_agents,
        trainer.observation_dim,
    )
    flat_mask = masks.reshape(
        batch_size * trainer.num_agents,
        trainer.discrete_dim,
    )
    (
        continuous,
        continuous_log_probability,
        destination_one_hot,
        destination_indices,
        destination_log_probability,
        destination_entropy,
    ) = actor_ddp(flat_obs, flat_mask)
    return {
        "continuous": continuous.reshape(
            batch_size, trainer.num_agents, trainer.continuous_dim
        ),
        "continuous_log_probability": continuous_log_probability.reshape(
            batch_size, trainer.num_agents, 1
        ).sum(dim=1),
        "destination_one_hot": destination_one_hot.reshape(
            batch_size, trainer.num_agents, trainer.discrete_dim
        ),
        "destination_indices": destination_indices.reshape(
            batch_size, trainer.num_agents
        ),
        "destination_log_probability": destination_log_probability.reshape(
            batch_size, trainer.num_agents, 1
        ).sum(dim=1),
        "destination_entropy": destination_entropy.reshape(
            batch_size, trainer.num_agents, 1
        ).sum(dim=1),
    }


def _ddp_actor_actions_matd3(
    trainer, actor_ddp, observations, masks
):
    batch_size = int(observations.shape[0])
    flat_obs = observations.reshape(
        batch_size * trainer.num_agents,
        trainer.observation_dim,
    )
    flat_mask = masks.reshape(
        batch_size * trainer.num_agents,
        trainer.discrete_dim,
    )
    continuous, one_hot, indices, probabilities = actor_ddp(
        flat_obs, flat_mask
    )
    return {
        "continuous": continuous.reshape(
            batch_size, trainer.num_agents, trainer.continuous_dim
        ),
        "destination_one_hot": one_hot.reshape(
            batch_size, trainer.num_agents, trainer.discrete_dim
        ),
        "destination_indices": indices.reshape(
            batch_size, trainer.num_agents
        ),
        "destination_probabilities": probabilities.reshape(
            batch_size, trainer.num_agents, trainer.discrete_dim
        ),
    }


def _allreduce_parameter_grad(parameter):
    import torch.distributed as dist
    if parameter.grad is None:
        return
    dist.all_reduce(parameter.grad, op=dist.ReduceOp.SUM)
    parameter.grad.div_(dist.get_world_size())


def _ddp_update_masac(
    trainer,
    replay,
    batch_size,
    actor_ddp,
    critic_1_ddp,
    critic_2_ddp,
    collect_metrics=True,
):
    batch = replay.sample(batch_size, trainer.device)
    destination_one_hot = torch.nn.functional.one_hot(
        batch["destination_indices"],
        num_classes=trainer.discrete_dim,
    ).to(torch.float32)
    replay_joint_action = trainer._joint_action_tensor(
        batch["continuous_actions"],
        destination_one_hot,
    )

    with torch.no_grad(), trainer._autocast():
        next_policy = trainer._sample_joint_policy(
            batch["next_observations"],
            batch["next_destination_masks"],
            deterministic=False,
        )
        next_joint_action = trainer._joint_action_tensor(
            next_policy["continuous"],
            next_policy["destination_one_hot"],
        )
        target_q = torch.minimum(
            trainer.target_critic_1(
                batch["next_states"], next_joint_action
            ),
            trainer.target_critic_2(
                batch["next_states"], next_joint_action
            ),
        )
        entropy_adjusted_target = (
            target_q
            - trainer.alpha_continuous.detach()
            * next_policy["continuous_log_probability"]
            - trainer.alpha_discrete.detach()
            * next_policy["destination_log_probability"]
        )
        critic_target = (
            batch["rewards"]
            + trainer.gamma
            * (1.0 - batch["terminated"])
            * entropy_adjusted_target
        )

    with trainer._autocast():
        critic_1_value = critic_1_ddp(
            batch["states"], replay_joint_action
        )
        critic_2_value = critic_2_ddp(
            batch["states"], replay_joint_action
        )
        critic_loss = (
            torch.nn.functional.mse_loss(
                critic_1_value, critic_target
            )
            + torch.nn.functional.mse_loss(
                critic_2_value, critic_target
            )
        )

    trainer.critic_optimizer.zero_grad(set_to_none=True)
    if trainer.amp_enabled:
        trainer.grad_scaler.scale(critic_loss).backward()
        trainer.grad_scaler.unscale_(trainer.critic_optimizer)
    else:
        critic_loss.backward()
    critic_grad_norm = torch.nn.utils.clip_grad_norm_(
        list(trainer.critic_1.parameters())
        + list(trainer.critic_2.parameters()),
        trainer.gradient_clip_norm,
    )
    if trainer.amp_enabled:
        trainer.grad_scaler.step(trainer.critic_optimizer)
    else:
        trainer.critic_optimizer.step()

    for critic in (trainer.critic_1, trainer.critic_2):
        for parameter in critic.parameters():
            parameter.requires_grad_(False)

    with trainer._autocast():
        policy = _ddp_actor_sample_masac(
            trainer,
            actor_ddp,
            batch["observations"],
            batch["destination_masks"],
        )
        policy_joint_action = trainer._joint_action_tensor(
            policy["continuous"],
            policy["destination_one_hot"],
        )
        policy_q = torch.minimum(
            trainer.critic_1(
                batch["states"], policy_joint_action
            ),
            trainer.critic_2(
                batch["states"], policy_joint_action
            ),
        )
        actor_loss = (
            trainer.alpha_continuous.detach()
            * policy["continuous_log_probability"]
            - trainer.alpha_discrete.detach()
            * policy["destination_entropy"]
            - policy_q
        ).mean()

    trainer.actor_optimizer.zero_grad(set_to_none=True)
    if trainer.amp_enabled:
        trainer.grad_scaler.scale(actor_loss).backward()
        trainer.grad_scaler.unscale_(trainer.actor_optimizer)
    else:
        actor_loss.backward()
    actor_grad_norm = torch.nn.utils.clip_grad_norm_(
        trainer.actor.parameters(),
        trainer.gradient_clip_norm,
    )
    if trainer.amp_enabled:
        trainer.grad_scaler.step(trainer.actor_optimizer)
        trainer.grad_scaler.update()
    else:
        trainer.actor_optimizer.step()

    for critic in (trainer.critic_1, trainer.critic_2):
        for parameter in critic.parameters():
            parameter.requires_grad_(True)

    continuous_entropy = -policy[
        "continuous_log_probability"
    ].detach()
    continuous_target = torch.full_like(
        continuous_entropy,
        float(trainer.continuous_target_entropy),
    )
    valid_counts = batch["destination_masks"].sum(
        dim=-1
    ).clamp(min=1.0)
    discrete_target = (
        trainer.discrete_target_entropy_ratio
        * torch.log(valid_counts).sum(
            dim=1, keepdim=True
        )
    )
    discrete_entropy = policy["destination_entropy"].detach()
    alpha_continuous_loss = (
        trainer.log_alpha_continuous
        * (continuous_entropy - continuous_target)
    ).mean()
    alpha_discrete_loss = (
        trainer.log_alpha_discrete
        * (discrete_entropy - discrete_target)
    ).mean()

    trainer.alpha_continuous_optimizer.zero_grad(set_to_none=True)
    alpha_continuous_loss.backward()
    _allreduce_parameter_grad(trainer.log_alpha_continuous)
    trainer.alpha_continuous_optimizer.step()

    trainer.alpha_discrete_optimizer.zero_grad(set_to_none=True)
    alpha_discrete_loss.backward()
    _allreduce_parameter_grad(trainer.log_alpha_discrete)
    trainer.alpha_discrete_optimizer.step()

    trainer._soft_update_targets()
    trainer.update_count += 1

    if not collect_metrics:
        return None
    return {
        "critic_loss": float(critic_loss.detach().cpu()),
        "actor_loss": float(actor_loss.detach().cpu()),
        "alpha_continuous_loss": float(
            alpha_continuous_loss.detach().cpu()
        ),
        "alpha_discrete_loss": float(
            alpha_discrete_loss.detach().cpu()
        ),
        "alpha_continuous": float(
            trainer.alpha_continuous.detach().cpu()
        ),
        "alpha_discrete": float(
            trainer.alpha_discrete.detach().cpu()
        ),
        "continuous_entropy": float(
            continuous_entropy.mean().cpu()
        ),
        "discrete_entropy": float(
            discrete_entropy.mean().cpu()
        ),
        "target_q_mean": float(
            critic_target.mean().detach().cpu()
        ),
        "critic_grad_norm": float(
            torch.as_tensor(critic_grad_norm).detach().cpu()
        ),
        "actor_grad_norm": float(
            torch.as_tensor(actor_grad_norm).detach().cpu()
        ),
        "update_count": int(trainer.update_count),
    }


def _ddp_update_matd3(
    trainer,
    replay,
    batch_size,
    actor_ddp,
    critic_1_ddp,
    critic_2_ddp,
    collect_metrics=True,
):
    batch = replay.sample(batch_size, trainer.device)
    replay_destination_one_hot = torch.nn.functional.one_hot(
        batch["destination_indices"],
        num_classes=trainer.discrete_dim,
    ).to(torch.float32)
    replay_joint_action = trainer._joint_action_tensor(
        batch["continuous_actions"],
        replay_destination_one_hot,
    )

    with torch.no_grad(), trainer._autocast():
        target_policy = trainer._actor_joint_actions(
            batch["next_observations"],
            batch["next_destination_masks"],
            straight_through=False,
            target_actor=True,
        )
        noise = (
            torch.randn_like(target_policy["continuous"])
            * trainer.target_policy_noise
        ).clamp(
            -trainer.target_noise_clip,
            trainer.target_noise_clip,
        )
        target_continuous = (
            target_policy["continuous"] + noise
        ).clamp(-1.0, 1.0)
        target_continuous = torch.cat(
            (
                trainer._project_namespace[
                    "project_motion_action_tensor"
                ](target_continuous[..., :3]),
                target_continuous[..., 3:4],
            ),
            dim=-1,
        )
        target_joint_action = trainer._joint_action_tensor(
            target_continuous,
            target_policy["destination_one_hot"],
        )
        target_q = torch.minimum(
            trainer.target_critic_1(
                batch["next_states"], target_joint_action
            ),
            trainer.target_critic_2(
                batch["next_states"], target_joint_action
            ),
        )
        critic_target = (
            batch["rewards"]
            + trainer.gamma
            * (1.0 - batch["terminated"])
            * target_q
        )

    with trainer._autocast():
        critic_1_value = critic_1_ddp(
            batch["states"], replay_joint_action
        )
        critic_2_value = critic_2_ddp(
            batch["states"], replay_joint_action
        )
        critic_loss = (
            torch.nn.functional.mse_loss(
                critic_1_value, critic_target
            )
            + torch.nn.functional.mse_loss(
                critic_2_value, critic_target
            )
        )

    trainer.critic_optimizer.zero_grad(set_to_none=True)
    if trainer.amp_enabled:
        trainer.grad_scaler.scale(critic_loss).backward()
        trainer.grad_scaler.unscale_(trainer.critic_optimizer)
    else:
        critic_loss.backward()
    critic_grad_norm = torch.nn.utils.clip_grad_norm_(
        list(trainer.critic_1.parameters())
        + list(trainer.critic_2.parameters()),
        trainer.gradient_clip_norm,
    )
    if trainer.amp_enabled:
        trainer.grad_scaler.step(trainer.critic_optimizer)
    else:
        trainer.critic_optimizer.step()

    trainer.update_count += 1
    actor_updated = bool(
        trainer.update_count % trainer.policy_delay == 0
    )
    actor_loss = None
    actor_grad_norm = None

    if actor_updated:
        for critic in (trainer.critic_1, trainer.critic_2):
            for parameter in critic.parameters():
                parameter.requires_grad_(False)

        with trainer._autocast():
            policy = _ddp_actor_actions_matd3(
                trainer,
                actor_ddp,
                batch["observations"],
                batch["destination_masks"],
            )
            policy_joint_action = trainer._joint_action_tensor(
                policy["continuous"],
                policy["destination_one_hot"],
            )
            actor_loss = -trainer.critic_1(
                batch["states"],
                policy_joint_action,
            ).mean()

        trainer.actor_optimizer.zero_grad(set_to_none=True)
        if trainer.amp_enabled:
            trainer.grad_scaler.scale(actor_loss).backward()
            trainer.grad_scaler.unscale_(trainer.actor_optimizer)
        else:
            actor_loss.backward()
        actor_grad_norm = torch.nn.utils.clip_grad_norm_(
            trainer.actor.parameters(),
            trainer.gradient_clip_norm,
        )
        if trainer.amp_enabled:
            trainer.grad_scaler.step(trainer.actor_optimizer)
            trainer.grad_scaler.update()
        else:
            trainer.actor_optimizer.step()

        for critic in (trainer.critic_1, trainer.critic_2):
            for parameter in critic.parameters():
                parameter.requires_grad_(True)

        trainer._soft_update(
            trainer.target_actor, trainer.actor
        )
        trainer._soft_update(
            trainer.target_critic_1, trainer.critic_1
        )
        trainer._soft_update(
            trainer.target_critic_2, trainer.critic_2
        )

    if not collect_metrics:
        return None
    metrics = {
        "critic_loss": float(critic_loss.detach().cpu()),
        "critic_1_q_mean": float(
            critic_1_value.detach().mean().cpu()
        ),
        "critic_2_q_mean": float(
            critic_2_value.detach().mean().cpu()
        ),
        "target_q_mean": float(
            critic_target.detach().mean().cpu()
        ),
        "critic_grad_norm": float(
            torch.as_tensor(critic_grad_norm).detach().cpu()
        ),
        "actor_updated": bool(actor_updated),
        "update_count": int(trainer.update_count),
    }
    if actor_loss is not None:
        metrics["actor_loss"] = float(
            actor_loss.detach().cpu()
        )
        metrics["actor_grad_norm"] = float(
            torch.as_tensor(actor_grad_norm).detach().cpu()
        )
    return metrics


def _train_full_gpu_ddp_worker(
    repo,
    algorithm,
    num_envs,
    total_transitions,
    seed,
    network_backend,
    result_path,
    enable_wandb=True,
):
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel as DDP

    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    torch.cuda.set_device(local_rank)
    device = torch.device(f"cuda:{local_rank}")
    dist.init_process_group(backend="nccl")

    algorithm = str(algorithm).lower()
    if algorithm not in {"masac", "matd3"}:
        raise ValueError("algorithm must be masac or matd3")
    if world_size != 2:
        raise ValueError("current automatic DDP implementation expects exactly 2 GPUs")

    torch.use_deterministic_algorithms(True)
    random.seed(int(seed))
    torch.manual_seed(int(seed))
    torch.cuda.manual_seed(int(seed))

    shard_counts = _split_env_counts(num_envs, world_size)
    local_envs = int(shard_counts[rank])
    env = FullGpuUAVBatchEnv(
        num_envs=local_envs,
        device=str(device),
        seed=int(seed) + rank * 1_000_003,
        strict_cuda=True,
    )
    env.network_model = str(network_backend)
    env.assert_all_state_on_cuda()

    trainer = _trainer(
        Path(repo), algorithm, str(device), int(seed)
    )
    project_ns = trainer._project_namespace

    if algorithm == "masac":
        actor_forward = _MASACDDPActorForward(
            trainer.actor, project_ns
        ).to(device)
    else:
        actor_forward = _MATD3DDPActorForward(
            trainer.actor, project_ns
        ).to(device)

    actor_ddp = DDP(
        actor_forward,
        device_ids=[local_rank],
        output_device=local_rank,
        broadcast_buffers=False,
        gradient_as_bucket_view=True,
    )
    critic_1_ddp = DDP(
        trainer.critic_1,
        device_ids=[local_rank],
        output_device=local_rank,
        broadcast_buffers=False,
        gradient_as_bucket_view=True,
    )
    critic_2_ddp = DDP(
        trainer.critic_2,
        device_ids=[local_rank],
        output_device=local_rank,
        broadcast_buffers=False,
        gradient_as_bucket_view=True,
    )

    # All source networks are synchronized by DDP construction. Mirror the
    # synchronized online critics/actor into the local targets before training.
    if algorithm == "masac":
        trainer.target_critic_1.load_state_dict(
            trainer.critic_1.state_dict()
        )
        trainer.target_critic_2.load_state_dict(
            trainer.critic_2.state_dict()
        )
    else:
        trainer.target_actor.load_state_dict(
            trainer.actor.state_dict()
        )
        trainer.target_critic_1.load_state_dict(
            trainer.critic_1.state_dict()
        )
        trainer.target_critic_2.load_state_dict(
            trainer.critic_2.state_dict()
        )

    # Rank-specific data RNG after synchronized model initialization.
    random.seed(int(seed) + rank * 97_409)
    torch.cuda.manual_seed(int(seed) + rank * 97_409)

    global_capacity = min(
        max(int(CONFIG[f"{algorithm}_replay_capacity"]), 4096),
        max(int(total_transitions), 4096),
    )
    local_capacity = max(
        4096,
        int(math.ceil(global_capacity / world_size)),
    )
    replay = make_project_gpu_replay_buffer(
        trainer,
        capacity=local_capacity,
        num_agents=env.num_uavs,
        observation_dim=env.observation_dim,
        state_dim=env.state_dim,
        continuous_dim=env.continuous_dim,
        discrete_dim=env.discrete_dim,
        device=str(device),
        seed=int(seed) + 123 + rank * 10_007,
        strict_cuda=True,
    )

    global_batch_size = int(
        CONFIG[f"{algorithm}_batch_size"]
    )
    local_batch_size = max(
        1, int(math.ceil(global_batch_size / world_size))
    )
    learning_starts = int(
        CONFIG[f"{algorithm}_learning_starts"]
    )
    update_to_data_ratio = float(
        CONFIG[f"{algorithm}_updates_per_step"]
    )

    wandb_run = (
        _init_gpu_wandb_run(
            algorithm, seed, enabled=enable_wandb
        )
        if rank == 0
        else None
    )

    obs, state, mask = env.reset()
    global_step = 0
    update_count = 0
    update_credit = 0.0
    latest_metrics = None
    completed_episodes = 0
    shard_offsets = []
    running_offset = 0
    for count in shard_counts:
        shard_offsets.append(running_offset)
        running_offset += int(count)

    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize(device)
    start = time.perf_counter()
    log_interval = max(
        1, int(CONFIG.get("training_log_interval_steps", 100))
    )
    last_log_step = -1

    try:
        while global_step < int(total_transitions):
            local_offset = int(shard_offsets[rank])
            transition_steps = (
                torch.arange(
                    local_envs,
                    device=device,
                    dtype=torch.long,
                )
                + global_step
                + local_offset
                + 1
            )
            batch_first = global_step + 1
            batch_last = global_step + int(num_envs)
            all_warmup = batch_last <= learning_starts
            all_policy = batch_first > learning_starts

            if all_warmup:
                continuous, destination = (
                    _random_exploration_actions(env, mask)
                )
            elif all_policy:
                if algorithm == "masac":
                    with torch.no_grad():
                        sampled = trainer._sample_joint_policy(
                            obs,
                            mask,
                            deterministic=False,
                        )
                        continuous = sampled["continuous"]
                        destination = sampled[
                            "destination_indices"
                        ]
                else:
                    continuous, destination = _matd3_actions(
                        trainer,
                        obs,
                        mask,
                        transition_steps,
                        env.generator,
                    )
            else:
                random_continuous, random_destination = (
                    _random_exploration_actions(env, mask)
                )
                if algorithm == "masac":
                    with torch.no_grad():
                        sampled = trainer._sample_joint_policy(
                            obs,
                            mask,
                            deterministic=False,
                        )
                        policy_continuous = sampled["continuous"]
                        policy_destination = sampled[
                            "destination_indices"
                        ]
                else:
                    policy_continuous, policy_destination = (
                        _matd3_actions(
                            trainer,
                            obs,
                            mask,
                            transition_steps,
                            env.generator,
                        )
                    )
                warmup = transition_steps <= learning_starts
                continuous = torch.where(
                    warmup[:, None, None],
                    random_continuous,
                    policy_continuous,
                )
                destination = torch.where(
                    warmup[:, None],
                    random_destination,
                    policy_destination,
                )

            (
                next_obs,
                next_state,
                next_mask,
                reward,
                done,
                _,
            ) = env.step(continuous, destination)

            take_total = min(
                int(num_envs),
                int(total_transitions) - global_step,
            )
            local_take = max(
                0,
                min(
                    local_envs,
                    take_total - local_offset,
                ),
            )
            if local_take > 0:
                replay.add_batch(
                    obs[:local_take],
                    state[:local_take],
                    continuous[:local_take],
                    destination[:local_take],
                    mask[:local_take],
                    reward[:local_take],
                    next_obs[:local_take],
                    next_state[:local_take],
                    next_mask[:local_take],
                    done[:local_take],
                )

            previous = global_step
            global_step += take_total

            local_ready = torch.tensor(
                [int(len(replay) >= local_batch_size)],
                device=device,
                dtype=torch.int32,
            )
            dist.all_reduce(
                local_ready, op=dist.ReduceOp.MIN
            )
            if int(local_ready.item()) == 1:
                first_train_step = max(
                    previous, learning_starts
                )
                eligible = max(
                    0, global_step - first_train_step
                )
                update_credit += (
                    eligible * update_to_data_ratio
                )
                updates_this_step = int(update_credit)
                update_credit -= updates_this_step
                for local_update in range(
                    updates_this_step
                ):
                    collect = (
                        local_update
                        == updates_this_step - 1
                    )
                    if algorithm == "masac":
                        metrics = _ddp_update_masac(
                            trainer,
                            replay,
                            local_batch_size,
                            actor_ddp,
                            critic_1_ddp,
                            critic_2_ddp,
                            collect_metrics=collect,
                        )
                    else:
                        metrics = _ddp_update_matd3(
                            trainer,
                            replay,
                            local_batch_size,
                            actor_ddp,
                            critic_1_ddp,
                            critic_2_ddp,
                            collect_metrics=collect,
                        )
                    if metrics is not None:
                        latest_metrics = metrics
                    update_count += 1

            if bool(done.any().item()):
                completed_episodes += int(
                    done.long().sum().item()
                )
                if global_step < int(total_transitions):
                    obs, state, mask = env.reset(mask=done)
                else:
                    obs, state, mask = (
                        next_obs,
                        next_state,
                        next_mask,
                    )
            else:
                obs, state, mask = (
                    next_obs,
                    next_state,
                    next_mask,
                )

            if (
                rank == 0
                and wandb_run is not None
                and (
                    global_step - last_log_step
                    >= log_interval
                    or global_step
                    >= int(total_transitions)
                )
            ):
                elapsed = max(
                    1e-12, time.perf_counter() - start
                )
                payload = {
                    "global_step": int(global_step),
                    "train/transitions_per_second": (
                        global_step / elapsed
                    ),
                    "train/updates_per_second": (
                        update_count / elapsed
                    ),
                    "system/gpu_count": int(world_size),
                }
                if latest_metrics:
                    for key, value in latest_metrics.items():
                        if isinstance(
                            value, (bool, int, float)
                        ):
                            payload[
                                f"train/{key}"
                            ] = float(value)
                wandb_run.log(payload)
                last_log_step = global_step

        torch.cuda.synchronize(device)
        wall = time.perf_counter() - start
        dist.barrier(device_ids=[rank])

        rank_result = {
            "rank": rank,
            "device": str(device),
            "gpu_name": torch.cuda.get_device_name(device),
            "local_envs": local_envs,
            "local_replay_size": int(len(replay)),
            "peak_memory_mb": (
                torch.cuda.max_memory_allocated(device)
                / 2**20
            ),
            "completed_episodes": completed_episodes,
        }
        gathered = [None] * world_size if rank == 0 else None
        dist.gather_object(
            rank_result,
            object_gather_list=gathered,
            dst=0,
        )

        if rank == 0:
            result = {
                "kind": "full_gpu_ddp",
                "multi_gpu_strategy": "ddp",
                "algorithm": algorithm,
                "network_backend": network_backend,
                "used_gpu_count": world_size,
                "num_envs": int(num_envs),
                "envs_per_device": {
                    item["device"]: item["local_envs"]
                    for item in gathered
                },
                "devices": [
                    item["device"] for item in gathered
                ],
                "gpu_names": {
                    item["device"]: item["gpu_name"]
                    for item in gathered
                },
                "peak_memory_mb_per_device": {
                    item["device"]: item[
                        "peak_memory_mb"
                    ]
                    for item in gathered
                },
                "transitions": int(global_step),
                "updates": int(update_count),
                "episodes_completed": int(
                    sum(
                        item["completed_episodes"]
                        for item in gathered
                    )
                ),
                "wall_seconds": float(wall),
                "transitions_per_second": (
                    global_step / wall
                ),
                "updates_per_second": (
                    update_count / wall
                ),
                "latest_update_metrics": (
                    latest_metrics
                ),
                "model_fingerprint_sha256": (
                    _trainer_parameter_fingerprint(
                        trainer
                    )
                ),
                "wandb_run_id": (
                    wandb_run.id
                    if wandb_run is not None
                    else None
                ),
                "wandb_run_name": (
                    wandb_run.name
                    if wandb_run is not None
                    else None
                ),
                "wandb_run_url": (
                    wandb_run.url
                    if wandb_run is not None
                    else None
                ),
            }
            Path(result_path).write_text(
                json.dumps(
                    result,
                    indent=2,
                    default=str,
                )
            )
            if wandb_run is not None:
                wandb_run.summary[
                    "final/transitions_per_second"
                ] = result[
                    "transitions_per_second"
                ]
                wandb_run.summary[
                    "final/updates_per_second"
                ] = result[
                    "updates_per_second"
                ]
                wandb_run.summary[
                    "final/gpu_count"
                ] = world_size
                wandb_run.finish()
        dist.barrier(device_ids=[rank])
    finally:
        env.close()
        if dist.is_initialized():
            dist.destroy_process_group()


def _write_ddp_launcher_script(path):
    script = r"""import argparse
import cloudpickle

parser = argparse.ArgumentParser()
parser.add_argument("--payload", required=True)
parser.add_argument("--repo", required=True)
parser.add_argument("--algorithm", required=True)
parser.add_argument("--num-envs", type=int, required=True)
parser.add_argument("--total-transitions", type=int, required=True)
parser.add_argument("--seed", type=int, required=True)
parser.add_argument("--network-backend", required=True)
parser.add_argument("--result-path", required=True)
parser.add_argument("--enable-wandb", type=int, default=1)
args = parser.parse_args()

import sys
sys.path.insert(0, args.repo)

with open(args.payload, "rb") as handle:
    worker = cloudpickle.load(handle)

worker(
    repo=args.repo,
    algorithm=args.algorithm,
    num_envs=args.num_envs,
    total_transitions=args.total_transitions,
    seed=args.seed,
    network_backend=args.network_backend,
    result_path=args.result_path,
    enable_wandb=bool(args.enable_wandb),
)
"""
    Path(path).write_text(script)
    return Path(path)



def train_full_gpu_ddp(
    repo: Path,
    algorithm: str,
    num_envs: int,
    total_transitions: int,
    seed: int,
    network_backend: str = "uavnetsim_gpu",
    enable_wandb=None,
):
    if torch.cuda.device_count() < 2:
        raise RuntimeError(
            "DDP requires at least two visible CUDA GPUs"
        )
    if enable_wandb is None:
        enable_wandb = bool(
            CONFIG.get("training_enable_wandb", True)
        )
    # Resolve/login once in the parent process so rank 0 cannot fail during
    # W&B setup while rank 1 is already waiting in NCCL collectives. The
    # WANDB_API_KEY environment variable is inherited by torchrun workers.
    if bool(enable_wandb):
        mode = str(CONFIG.get("training_wandb_mode", "online"))
        if mode == "online" and not os.environ.get("WANDB_API_KEY"):
            project_ns = load_project_namespace(Path(repo))
            if bool(project_ns.get("running_on_kaggle", lambda: False)()):
                status = project_ns["configure_kaggle_wandb"](
                    verify_login=True
                )
                if not bool(status.get("configured", False)):
                    raise RuntimeError(
                        "W&B online logging is required but no attached Kaggle "
                        "WANDB_API_KEY/wandb_key secret could be configured: "
                        + str(status.get("error"))
                    )
            else:
                raise RuntimeError(
                    "W&B online logging is required but WANDB_API_KEY is not set"
                )
    import subprocess
    import tempfile
    import gc

    # The notebook parent may have just finished a large single-GPU run. Its
    # caching allocator can otherwise keep most T4 VRAM reserved while the two
    # torchrun workers start. Release all inactive parent allocations first.
    gc.collect()
    for device_index in range(min(2, torch.cuda.device_count())):
        with torch.cuda.device(device_index):
            torch.cuda.synchronize(device_index)
            torch.cuda.empty_cache()

    temp_dir = Path(
        tempfile.mkdtemp(prefix="uav_ddp_")
    )
    import cloudpickle
    payload_path = temp_dir / "worker.cloudpickle"
    with payload_path.open("wb") as payload_file:
        cloudpickle.dump(
            _train_full_gpu_ddp_worker,
            payload_file,
        )
    launcher = _write_ddp_launcher_script(
        temp_dir / "uav_ddp_runner.py",
    )
    result_path = temp_dir / "result.json"
    command = [
        sys.executable,
        "-m",
        "torch.distributed.run",
        "--standalone",
        "--nnodes=1",
        "--nproc-per-node=2",
        str(launcher),
        "--payload",
        str(payload_path),
        "--repo",
        str(Path(repo).resolve()),
        "--algorithm",
        str(algorithm),
        "--num-envs",
        str(int(num_envs)),
        "--total-transitions",
        str(int(total_transitions)),
        "--seed",
        str(int(seed)),
        "--network-backend",
        str(network_backend),
        "--result-path",
        str(result_path),
        "--enable-wandb",
        "1" if enable_wandb else "0",
    ]
    stdout_path = temp_dir / "torchrun.stdout.log"
    stderr_path = temp_dir / "torchrun.stderr.log"
    with stdout_path.open("w") as stdout_file, stderr_path.open("w") as stderr_file:
        child_env = os.environ.copy()
        child_env.setdefault(
            "PYTORCH_CUDA_ALLOC_CONF",
            "expandable_segments:True",
        )
        completed = subprocess.run(
            command,
            cwd=str(Path(repo).resolve()),
            check=False,
            text=True,
            stdout=stdout_file,
            stderr=stderr_file,
            env=child_env,
        )
    stdout_text = stdout_path.read_text(errors="replace")
    stderr_text = stderr_path.read_text(errors="replace")
    if completed.returncode != 0:
        raise RuntimeError(
            "DDP training failed.\nSTDOUT:\n"
            + stdout_text[-12000:]
            + "\nSTDERR:\n"
            + stderr_text[-12000:]
        )
    if not result_path.is_file():
        raise RuntimeError(
            "DDP training completed without result.json"
        )
    result = json.loads(result_path.read_text())
    result["ddp_stdout_tail"] = stdout_text[-4000:]
    result["ddp_stderr_tail"] = stderr_text[-4000:]
    return result



def resolve_full_execution_mode(execution_mode=None):
    """Resolve cpu/gpu/auto while keeping one tensor implementation."""
    mode = (
        CONFIG.get("full_gpu_execution_mode", "auto")
        if execution_mode is None
        else execution_mode
    )
    mode = str(mode).strip().lower()
    if mode not in {"cpu", "gpu", "auto"}:
        raise ValueError("execution_mode must be cpu, gpu, or auto")
    if mode == "auto":
        mode = "gpu" if torch.cuda.is_available() else "cpu"
    if mode == "gpu" and not torch.cuda.is_available():
        raise RuntimeError(
            "execution_mode='gpu' requires an available CUDA GPU"
        )
    return mode


def resolve_full_gpu_devices(max_gpus=None):
    """Return the CUDA devices used by automatic training, capped at two."""
    if not torch.cuda.is_available():
        return []
    visible = int(torch.cuda.device_count())
    if max_gpus is None:
        max_gpus = int(CONFIG.get("full_gpu_max_gpus", 2))
    limit = max(1, min(int(max_gpus), 2))
    return [torch.device(f"cuda:{index}") for index in range(min(visible, limit))]


def _split_env_counts(total_envs, num_shards):
    total_envs = int(total_envs)
    num_shards = int(num_shards)
    if total_envs < 1:
        raise ValueError("total_envs must be >= 1")
    if num_shards < 1:
        raise ValueError("num_shards must be >= 1")
    num_shards = min(num_shards, total_envs)
    base, remainder = divmod(total_envs, num_shards)
    return [base + (1 if index < remainder else 0) for index in range(num_shards)]


def train_full_gpu_sharded(
    repo: Path,
    algorithm: str,
    num_envs: int,
    total_transitions: int,
    seed: int,
    devices=None,
    network_backend: str = "uavnetsim_gpu",
    compile_mode=None,
    fused_adam: bool = False,
    update_to_data_ratio=None,
    enable_wandb=None,
):
    """Use two CUDA devices by sharding environments and keeping one learner.

    GPU0 owns the replay buffer and learner/backward/optimizer. Environment
    shards are split across GPU0/GPU1 and stepped concurrently. Policy
    inference is centralized on GPU0 so there remains exactly one model state.
    This avoids two independently diverging MARL learners while still moving
    the dominant batched environment/network simulation onto both GPUs.
    """
    algorithm = str(algorithm).strip().lower()
    if algorithm not in {"masac", "matd3"}:
        raise ValueError("algorithm must be masac or matd3")
    network_backend = str(network_backend).strip().lower()
    if network_backend not in {"simple", "packet_approx", "uavnetsim_gpu"}:
        raise ValueError(
            "multi-GPU sharding supports simple, packet_approx, or uavnetsim_gpu"
        )
    if devices is None:
        devices = resolve_full_gpu_devices()
    devices = [torch.device(device) for device in devices]
    if len(devices) < 2:
        raise ValueError("train_full_gpu_sharded requires at least two CUDA devices")
    devices = devices[:2]
    for device in devices:
        if device.type != "cuda":
            raise ValueError("all sharded devices must be CUDA devices")
    if len({device.index for device in devices}) != len(devices):
        raise ValueError("sharded CUDA devices must be distinct")

    torch.use_deterministic_algorithms(True)
    random.seed(int(seed))
    torch.manual_seed(int(seed))
    torch.cuda.manual_seed_all(int(seed))

    shard_counts = _split_env_counts(num_envs, len(devices))
    primary = devices[0]
    trainer = _trainer(repo, algorithm, str(primary), seed)
    if enable_wandb is None:
        enable_wandb = bool(CONFIG.get("training_enable_wandb", True))
    wandb_run = _init_gpu_wandb_run(
        algorithm, seed, enabled=bool(enable_wandb)
    )
    wandb_last_log_step = -1
    fused_adam_enabled = False
    if bool(fused_adam):
        fused_adam_enabled = _rebuild_cuda_optimizers_with_fused_adam(
            trainer, algorithm
        )
    compiled_modules = []
    if compile_mode is not None:
        compiled_modules = _compile_trainer_modules(
            trainer, algorithm, mode=compile_mode
        )

    envs = []
    shard_states = []
    offsets = []
    running_offset = 0
    for rank, (device, count) in enumerate(zip(devices, shard_counts)):
        env_seed = int(seed) + rank * 1_000_003
        env = FullGpuUAVBatchEnv(
            num_envs=int(count),
            device=str(device),
            seed=env_seed,
            strict_cuda=True,
        )
        env.network_model = network_backend
        env.assert_all_state_on_cuda()
        if (
            env.observation_dim != trainer.observation_dim
            or env.state_dim != trainer.state_dim
            or env.discrete_dim != trainer.discrete_dim
        ):
            raise RuntimeError("multi-GPU environment dimensions do not match learner")
        obs, state, mask = env.reset()
        envs.append(env)
        shard_states.append([obs, state, mask])
        offsets.append(running_offset)
        running_offset += int(count)

    capacity = min(
        max(int(CONFIG[f"{algorithm}_replay_capacity"]), 4096),
        max(int(total_transitions), 4096),
    )
    reference_env = envs[0]
    replay = make_project_gpu_replay_buffer(
        trainer,
        capacity=capacity,
        num_agents=reference_env.num_uavs,
        observation_dim=reference_env.observation_dim,
        state_dim=reference_env.state_dim,
        continuous_dim=reference_env.continuous_dim,
        discrete_dim=reference_env.discrete_dim,
        device=str(primary),
        seed=int(seed) + 123,
        strict_cuda=True,
    )
    batch_size = int(CONFIG[f"{algorithm}_batch_size"])
    learning_starts = int(CONFIG[f"{algorithm}_learning_starts"])
    if update_to_data_ratio is None:
        update_to_data_ratio = float(CONFIG[f"{algorithm}_updates_per_step"])
    else:
        update_to_data_ratio = float(update_to_data_ratio)
    if update_to_data_ratio < 0.0:
        raise ValueError("update_to_data_ratio must be >= 0")

    policy_generator = torch.Generator(device=primary)
    policy_generator.manual_seed(int(seed) + 77_777)
    completed_episodes = 0
    global_step = 0
    update_count = 0
    update_credit = 0.0
    latest_metrics = None
    total_transitions_int = int(total_transitions)
    per_device_env_seconds = [0.0 for _ in devices]
    transfer_seconds = 0.0
    policy_seconds = 0.0
    update_seconds = 0.0

    for device in devices:
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize(device)

    import concurrent.futures

    def move_to_primary(tensor):
        nonlocal transfer_seconds
        if tensor.device == primary:
            return tensor
        started = time.perf_counter()
        result = tensor.to(primary, non_blocking=True)
        transfer_seconds += time.perf_counter() - started
        return result

    def policy_actions_for_all(transition_steps_all):
        nonlocal policy_seconds, transfer_seconds
        started = time.perf_counter()
        obs_parts = []
        mask_parts = []
        for obs, _state, mask in shard_states:
            transfer_start = time.perf_counter()
            obs_parts.append(obs if obs.device == primary else obs.to(primary, non_blocking=True))
            mask_parts.append(mask if mask.device == primary else mask.to(primary, non_blocking=True))
            transfer_seconds += time.perf_counter() - transfer_start
        obs_all = torch.cat(obs_parts, dim=0)
        mask_all = torch.cat(mask_parts, dim=0)
        if algorithm == "masac":
            with torch.no_grad():
                sampled = trainer._sample_joint_policy(
                    obs_all, mask_all, deterministic=False
                )
                continuous_all = sampled["continuous"]
                destination_all = sampled["destination_indices"]
        else:
            continuous_all, destination_all = _matd3_actions(
                trainer,
                obs_all,
                mask_all,
                transition_steps_all,
                policy_generator,
            )
        continuous_parts = []
        destination_parts = []
        start_index = 0
        for device, count in zip(devices, shard_counts):
            stop_index = start_index + int(count)
            c = continuous_all[start_index:stop_index]
            d = destination_all[start_index:stop_index]
            transfer_start = time.perf_counter()
            if device != primary:
                c = c.to(device, non_blocking=True)
                d = d.to(device, non_blocking=True)
            transfer_seconds += time.perf_counter() - transfer_start
            continuous_parts.append(c)
            destination_parts.append(d)
            start_index = stop_index
        policy_seconds += time.perf_counter() - started
        return continuous_parts, destination_parts

    wall_start = time.perf_counter()
    executor = concurrent.futures.ThreadPoolExecutor(max_workers=len(envs))
    try:
        while global_step < total_transitions_int:
            transition_steps_all = (
                torch.arange(int(num_envs), device=primary, dtype=torch.long)
                + global_step
                + 1
            )
            batch_first_step = global_step + 1
            batch_last_step = global_step + int(num_envs)
            all_warmup = batch_last_step <= learning_starts
            all_policy = batch_first_step > learning_starts

            if all_warmup:
                action_parts = [
                    _random_exploration_actions(env, shard_states[index][2])
                    for index, env in enumerate(envs)
                ]
                continuous_parts = [item[0] for item in action_parts]
                destination_parts = [item[1] for item in action_parts]
            elif all_policy:
                continuous_parts, destination_parts = policy_actions_for_all(
                    transition_steps_all
                )
            else:
                policy_continuous, policy_destination = policy_actions_for_all(
                    transition_steps_all
                )
                continuous_parts = []
                destination_parts = []
                for index, env in enumerate(envs):
                    random_continuous, random_destination = _random_exploration_actions(
                        env, shard_states[index][2]
                    )
                    local_start = offsets[index]
                    local_stop = local_start + shard_counts[index]
                    warmup_local = (
                        transition_steps_all[local_start:local_stop]
                        <= learning_starts
                    ).to(env.device)
                    continuous_parts.append(
                        torch.where(
                            warmup_local[:, None, None],
                            random_continuous,
                            policy_continuous[index],
                        )
                    )
                    destination_parts.append(
                        torch.where(
                            warmup_local[:, None],
                            random_destination,
                            policy_destination[index],
                        )
                    )

            def step_one(index):
                env = envs[index]
                with torch.cuda.device(env.device):
                    started = time.perf_counter()
                    result = env.step(
                        continuous_parts[index], destination_parts[index]
                    )
                    elapsed = time.perf_counter() - started
                return index, result, elapsed

            futures = [executor.submit(step_one, index) for index in range(len(envs))]
            step_results = [None] * len(envs)
            for future in futures:
                index, result, elapsed = future.result()
                step_results[index] = result
                per_device_env_seconds[index] += elapsed

            take_total = min(
                int(num_envs), total_transitions_int - global_step
            )
            remaining_take = take_total
            for index, env in enumerate(envs):
                (
                    next_obs,
                    next_state,
                    next_mask,
                    reward,
                    done,
                    _metrics,
                ) = step_results[index]
                obs, state, mask = shard_states[index]
                take = min(int(shard_counts[index]), remaining_take)
                if take > 0:
                    replay.add_batch(
                        move_to_primary(obs[:take]),
                        move_to_primary(state[:take]),
                        move_to_primary(continuous_parts[index][:take]),
                        move_to_primary(destination_parts[index][:take]),
                        move_to_primary(mask[:take]),
                        move_to_primary(reward[:take]),
                        move_to_primary(next_obs[:take]),
                        move_to_primary(next_state[:take]),
                        move_to_primary(next_mask[:take]),
                        move_to_primary(done[:take]),
                    )
                    remaining_take -= take
                shard_states[index] = [next_obs, next_state, next_mask]

            previous = global_step
            global_step += take_total
            if len(replay) >= batch_size:
                first_train_step = max(previous, learning_starts)
                eligible = max(0, global_step - first_train_step)
                update_credit += eligible * update_to_data_ratio
                updates_this_step = int(update_credit)
                update_credit -= updates_this_step
                update_start = time.perf_counter()
                for local_update in range(updates_this_step):
                    collected = trainer.update(
                        replay,
                        batch_size=batch_size,
                        collect_metrics=(local_update == updates_this_step - 1),
                    )
                    if collected is not None:
                        latest_metrics = collected
                    update_count += 1
                update_seconds += time.perf_counter() - update_start

            if (
                wandb_run is not None
                and (
                    global_step - wandb_last_log_step
                    >= max(1, int(CONFIG.get("training_log_interval_steps", 100)))
                    or global_step >= total_transitions_int
                )
            ):
                elapsed_for_log = max(
                    1e-12, time.perf_counter() - wall_start
                )
                payload = {
                    "global_step": int(global_step),
                    "train/transitions_per_second": global_step / elapsed_for_log,
                    "train/updates_per_second": update_count / elapsed_for_log,
                    "system/gpu_count": len(devices),
                }
                if latest_metrics:
                    for key, value in latest_metrics.items():
                        if isinstance(value, torch.Tensor):
                            if value.numel() != 1:
                                continue
                            value = value.detach().cpu().item()
                        if isinstance(value, (bool, int, float)):
                            payload[f"train/{key}"] = float(value)
                wandb_run.log(payload)
                wandb_last_log_step = global_step

            for index, env in enumerate(envs):
                done = step_results[index][4]
                if bool(done.any().item()):
                    completed_episodes += int(done.long().sum().item())
                    if global_step < total_transitions_int:
                        obs, state, mask = env.reset(mask=done)
                        shard_states[index] = [obs, state, mask]

        for device in devices:
            torch.cuda.synchronize(device)
        wall = time.perf_counter() - wall_start
    finally:
        executor.shutdown(wait=True)

    def scalarize(metrics):
        if metrics is None:
            return None
        result = {}
        for key, value in metrics.items():
            if isinstance(value, torch.Tensor):
                if value.numel() == 1:
                    result[key] = float(value.detach().cpu().item())
            elif isinstance(value, (int, float, bool)):
                result[key] = value
        return result

    peak_memory = {
        str(device): torch.cuda.max_memory_allocated(device) / 2**20
        for device in devices
    }
    gpu_names = {
        str(device): torch.cuda.get_device_name(device)
        for device in devices
    }
    result = {
        "kind": "full_gpu_auto_multi",
        "multi_gpu_strategy": "env_shard_single_learner",
        "algorithm": algorithm,
        "network_backend": network_backend,
        "num_envs": int(num_envs),
        "envs_per_device": {
            str(device): int(count)
            for device, count in zip(devices, shard_counts)
        },
        "used_gpu_count": len(devices),
        "devices": [str(device) for device in devices],
        "learner_device": str(primary),
        "gpu_names": gpu_names,
        "transitions": int(global_step),
        "updates": int(update_count),
        "update_to_data_ratio": float(update_to_data_ratio),
        "episodes_completed": int(completed_episodes),
        "wall_seconds": float(wall),
        "transitions_per_second": global_step / wall,
        "updates_per_second": update_count / wall,
        "policy_seconds": float(policy_seconds),
        "update_seconds": float(update_seconds),
        "env_seconds_per_device": {
            str(device): float(seconds)
            for device, seconds in zip(devices, per_device_env_seconds)
        },
        "transfer_enqueue_seconds": float(transfer_seconds),
        "peak_memory_mb_per_device": peak_memory,
        "compile_mode": compile_mode,
        "compiled_modules": compiled_modules,
        "fused_adam": bool(fused_adam_enabled),
        "latest_update_metrics": scalarize(latest_metrics),
        "model_fingerprint_sha256": _trainer_parameter_fingerprint(trainer),
        "wandb_run_id": (
            wandb_run.id if wandb_run is not None else None
        ),
        "wandb_run_name": (
            wandb_run.name if wandb_run is not None else None
        ),
        "wandb_run_url": (
            wandb_run.url if wandb_run is not None else None
        ),
    }
    if wandb_run is not None:
        wandb_run.summary["final/transitions_per_second"] = result[
            "transitions_per_second"
        ]
        wandb_run.summary["final/updates_per_second"] = result[
            "updates_per_second"
        ]
        wandb_run.summary["final/gpu_count"] = len(devices)
        wandb_run.finish()
    for env in envs:
        env.close()
    return result


def train_full_gpu_auto(
    repo: Path,
    algorithm: str,
    num_envs: int,
    total_transitions: int,
    seed: int,
    network_backend: str = "uavnetsim_gpu",
    max_gpus=None,
    auto_multi_gpu=None,
    compile_mode=None,
    fused_adam: bool = False,
    update_to_data_ratio=None,
    enable_wandb=None,
    execution_mode=None,
):
    """Run the same tensor training path in cpu/gpu/auto mode.

    cpu: force CPU for logic/regression testing.
    gpu: require CUDA and auto-select one GPU or 2-GPU DDP.
    auto: use GPU when CUDA is available, otherwise CPU.
    """
    if auto_multi_gpu is None:
        auto_multi_gpu = bool(
            CONFIG.get("full_gpu_auto_multi_gpu", True)
        )
    resolved_mode = resolve_full_execution_mode(
        execution_mode
    )
    if enable_wandb is None:
        enable_wandb = (
            False
            if resolved_mode == "cpu"
            else bool(CONFIG.get("training_enable_wandb", True))
        )

    if resolved_mode == "cpu":
        result = train_full_gpu(
            repo=repo,
            algorithm=algorithm,
            num_envs=int(num_envs),
            total_transitions=int(total_transitions),
            seed=int(seed),
            device="cpu",
            network_backend=network_backend,
            strict_cuda=False,
            compile_mode=compile_mode,
            fused_adam=False,
            update_to_data_ratio=update_to_data_ratio,
            enable_wandb=bool(enable_wandb),
        )
        result["execution_mode"] = "cpu"
        result["used_gpu_count"] = 0
        result["devices"] = ["cpu"]
        result["multi_gpu_strategy"] = "cpu"
        result["learner_device"] = "cpu"
        return result

    devices = resolve_full_gpu_devices(max_gpus=max_gpus)
    if not devices:
        raise RuntimeError(
            "GPU mode resolved but no CUDA device is available"
        )

    strategy = str(
        CONFIG.get("full_gpu_multi_gpu_strategy", "ddp")
    ).strip().lower()

    if (
        bool(auto_multi_gpu)
        and len(devices) >= 2
        and int(num_envs) >= 2
    ):
        if strategy == "ddp":
            if compile_mode is not None:
                raise ValueError(
                    "compile_mode is not enabled in the DDP path yet; "
                    "use compile_mode=None for deterministic DDP training"
                )
            if fused_adam:
                raise ValueError(
                    "fused_adam is not enabled in the DDP path yet; "
                    "use fused_adam=False until separately benchmarked"
                )
            if update_to_data_ratio is not None:
                configured = float(
                    CONFIG[
                        f"{str(algorithm).lower()}_updates_per_step"
                    ]
                )
                if float(update_to_data_ratio) != configured:
                    raise ValueError(
                        "custom update_to_data_ratio is not enabled in "
                        "the DDP launcher yet"
                    )
            result = train_full_gpu_ddp(
                repo=repo,
                algorithm=algorithm,
                num_envs=int(num_envs),
                total_transitions=int(total_transitions),
                seed=int(seed),
                network_backend=network_backend,
                enable_wandb=enable_wandb,
            )
            result["execution_mode"] = "gpu"
            return result
        if strategy == "env_shard_single_learner":
            result = train_full_gpu_sharded(
                repo=repo,
                algorithm=algorithm,
                num_envs=int(num_envs),
                total_transitions=int(total_transitions),
                seed=int(seed),
                devices=devices[:2],
                network_backend=network_backend,
                compile_mode=compile_mode,
                fused_adam=bool(fused_adam),
                update_to_data_ratio=update_to_data_ratio,
                enable_wandb=bool(enable_wandb),
            )
            result["execution_mode"] = "gpu"
            return result
        raise ValueError(
            "full_gpu_multi_gpu_strategy must be ddp or "
            "env_shard_single_learner"
        )

    result = train_full_gpu(
        repo=repo,
        algorithm=algorithm,
        num_envs=int(num_envs),
        total_transitions=int(total_transitions),
        seed=int(seed),
        device=str(devices[0]),
        network_backend=network_backend,
        strict_cuda=True,
        compile_mode=compile_mode,
        fused_adam=bool(fused_adam),
        update_to_data_ratio=update_to_data_ratio,
        enable_wandb=bool(enable_wandb),
    )
    result["execution_mode"] = "gpu"
    result["used_gpu_count"] = 1
    result["devices"] = [str(devices[0])]
    result["multi_gpu_strategy"] = "single_gpu"
    result["learner_device"] = str(devices[0])
    return result


def benchmark_full_gpu_training_optimizations(
    repo: Path,
    *,
    algorithms=("masac", "matd3"),
    env_counts=(256, 512, 1024),
    total_transitions=8192,
    seed=44,
    device="cuda:0",
    network_backend="uavnetsim_gpu",
    test_compile=True,
    test_fused_adam=True,
    test_amp=True,
    update_to_data_ratios=(1.0,),
):
    """Short end-to-end optimization sweep without changing production defaults.

    Timing-only fields are expected to vary. Deterministic metrics/fingerprints
    are returned so any speed optimization can be rejected if it changes the
    numerical training path unexpectedly.
    """
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for optimization benchmarking")

    previous_amp = bool(CONFIG.get("training_amp_enabled", False))
    cases = [
        {
            "name": "baseline",
            "compile_mode": None,
            "fused_adam": False,
            "amp": False,
        }
    ]
    if test_fused_adam:
        cases.append(
            {
                "name": "fused_adam",
                "compile_mode": None,
                "fused_adam": True,
                "amp": False,
            }
        )
    if test_compile:
        cases.append(
            {
                "name": "compile_default_fused",
                "compile_mode": "default",
                "fused_adam": bool(test_fused_adam),
                "amp": False,
            }
        )
    if test_amp:
        cases.append(
            {
                "name": "amp_fused",
                "compile_mode": None,
                "fused_adam": bool(test_fused_adam),
                "amp": True,
            }
        )
    if test_amp and test_compile:
        cases.append(
            {
                "name": "compile_amp_fused",
                "compile_mode": "default",
                "fused_adam": bool(test_fused_adam),
                "amp": True,
            }
        )

    results = []
    try:
        for algorithm in algorithms:
            for env_count in env_counts:
                for utd in update_to_data_ratios:
                    for case in cases:
                        CONFIG["training_amp_enabled"] = bool(
                            case["amp"]
                        )
                        torch.cuda.empty_cache()
                        torch.cuda.synchronize()
                        try:
                            item = train_full_gpu(
                                repo=repo,
                                algorithm=algorithm,
                                num_envs=int(env_count),
                                total_transitions=int(total_transitions),
                                seed=int(seed),
                                device=device,
                                network_backend=network_backend,
                                strict_cuda=True,
                                compile_mode=case["compile_mode"],
                                fused_adam=bool(case["fused_adam"]),
                                update_to_data_ratio=float(utd),
                            )
                            item["optimization_case"] = case["name"]
                            item["amp_enabled"] = bool(case["amp"])
                            results.append(item)
                        except torch.cuda.OutOfMemoryError as exc:
                            torch.cuda.empty_cache()
                            results.append(
                                {
                                    "algorithm": algorithm,
                                    "num_envs": int(env_count),
                                    "optimization_case": case["name"],
                                    "amp_enabled": bool(case["amp"]),
                                    "update_to_data_ratio": float(utd),
                                    "error": "CUDA OOM",
                                    "detail": str(exc),
                                }
                            )
        return results
    finally:
        CONFIG["training_amp_enabled"] = previous_amp


def profile_full_gpu_step(
    repo: Path,
    *,
    algorithm="masac",
    num_envs=256,
    seed=44,
    device="cuda:0",
    network_backend="uavnetsim_gpu",
    warmup_steps=3,
    active_steps=5,
    output_trace=None,
):
    """Profile actor inference + environment/network step on CUDA.

    This intentionally excludes optimizer updates; use the end-to-end sweep for
    learner throughput. The profiler table makes environment/network hotspots
    visible before attempting more invasive rewrites.
    """
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for profiling")
    algorithm = str(algorithm).strip().lower()
    env = FullGpuUAVBatchEnv(
        num_envs=int(num_envs),
        device=device,
        seed=int(seed),
        strict_cuda=True,
    )
    env.network_model = str(network_backend)
    trainer = _trainer(repo, algorithm, device, seed)
    obs, state, mask = env.reset()

    def one_step():
        nonlocal obs, state, mask
        if algorithm == "masac":
            with torch.no_grad():
                sample = trainer._sample_joint_policy(
                    obs,
                    mask,
                    deterministic=False,
                )
                continuous = sample["continuous"]
                destination = sample["destination_indices"]
        elif algorithm == "matd3":
            transition_steps = torch.ones(
                int(num_envs),
                dtype=torch.long,
                device=env.device,
            )
            continuous, destination = _matd3_actions(
                trainer,
                obs,
                mask,
                transition_steps,
                env.generator,
            )
        else:
            raise ValueError("algorithm must be masac or matd3")
        obs, state, mask, _, done, _ = env.step(
            continuous,
            destination,
        )
        if bool(done.any().item()):
            obs, state, mask = env.reset(mask=done)

    for _ in range(int(warmup_steps)):
        one_step()
    torch.cuda.synchronize(env.device)

    activities = [torch.profiler.ProfilerActivity.CPU]
    activities.append(torch.profiler.ProfilerActivity.CUDA)
    with torch.profiler.profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        for _ in range(int(active_steps)):
            one_step()
    torch.cuda.synchronize(env.device)
    table = prof.key_averages().table(
        sort_by="self_cuda_time_total",
        row_limit=30,
    )
    if output_trace is not None:
        prof.export_chrome_trace(str(output_trace))
    env.close()
    return table



def benchmark_gpu_network_models(
    *,
    network_models=("packet_approx", "uavnetsim_gpu"),
    env_counts=(256, 512, 1024),
    steps=40,
    warmup_steps=4,
    seed=44,
    device="cuda:0",
):
    """Measure the cost of richer network fidelity on the same GPU env."""
    previous = CONFIG.get("full_gpu_network_model", "simple")
    results = []
    try:
        for model in network_models:
            CONFIG["full_gpu_network_model"] = str(model)
            for count in env_counts:
                torch.cuda.empty_cache()
                if int(warmup_steps) > 0:
                    benchmark_env(
                        int(count),
                        int(warmup_steps),
                        int(seed),
                        device,
                        True,
                    )
                torch.cuda.empty_cache()
                item = benchmark_env(
                    int(count),
                    int(steps),
                    int(seed),
                    device,
                    True,
                )
                item["network_model"] = str(model)
                results.append(item)
        return results
    finally:
        CONFIG["full_gpu_network_model"] = previous


def benchmark_simple_scaling(
    repo: Path,
    *,
    cpu_env_counts=(1, 2, 4, 8, 16),
    gpu_env_counts=(1, 2, 4, 8, 16, 32, 64, 128, 256),
    vector_steps=40,
    warmup_steps=4,
    seed=44,
    device="cuda:0",
):
    results = {"cpu": [], "gpu": []}

    for count in cpu_env_counts:
        if int(warmup_steps) > 0:
            benchmark_hybrid_env(
                repo,
                num_envs=int(count),
                steps=int(warmup_steps),
                seed=int(seed),
            )
        item = benchmark_hybrid_env(
            repo,
            num_envs=int(count),
            steps=int(vector_steps),
            seed=int(seed),
        )
        results["cpu"].append(item)

    for count in gpu_env_counts:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(torch.device(device))
        if int(warmup_steps) > 0:
            benchmark_env(
                num_envs=int(count),
                steps=int(warmup_steps),
                seed=int(seed),
                device=device,
                strict_cuda=True,
            )
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(torch.device(device))
        item = benchmark_env(
            num_envs=int(count),
            steps=int(vector_steps),
            seed=int(seed),
            device=device,
            strict_cuda=True,
        )
        results["gpu"].append(item)

    cpu_by_count = {
        int(item["num_envs"]): item
        for item in results["cpu"]
    }
    gpu_by_count = {
        int(item["num_envs"]): item
        for item in results["gpu"]
    }
    same_count = []
    for count in sorted(set(cpu_by_count) & set(gpu_by_count)):
        cpu_rate = float(cpu_by_count[count]["transitions_per_second"])
        gpu_rate = float(gpu_by_count[count]["transitions_per_second"])
        same_count.append(
            {
                "num_envs": int(count),
                "cpu_transitions_per_second": cpu_rate,
                "gpu_transitions_per_second": gpu_rate,
                "gpu_over_cpu_speedup": (
                    gpu_rate / cpu_rate
                    if cpu_rate > 0.0
                    else float("inf")
                ),
            }
        )

    best_cpu = max(
        results["cpu"],
        key=lambda item: item["transitions_per_second"],
    )
    best_gpu = max(
        results["gpu"],
        key=lambda item: item["transitions_per_second"],
    )
    results["same_count_comparison"] = same_count
    results["best_cpu"] = best_cpu
    results["best_gpu"] = best_gpu
    results["best_gpu_over_best_cpu"] = (
        float(best_gpu["transitions_per_second"])
        / float(best_cpu["transitions_per_second"])
    )
    results["same_count_crossover_envs"] = next(
        (
            item["num_envs"]
            for item in same_count
            if item["gpu_over_cpu_speedup"] > 1.0
        ),
        None,
    )
    return results


def _uavnetsim_cpu_worker(payload):
    repo_text, seed, steps, communication_loaded = payload
    repo = Path(repo_text)
    project_ns = load_project_namespace(repo)
    np = project_ns["np"]
    CPUEnv = project_ns["UAVSearchEnv"]
    env = CPUEnv(
        backend_name="uavnetsim",
        network_backend_kwargs={"repo_path": None},
    )
    try:
        env.reset(seed=int(seed))
        if communication_loaded:
            Report = project_ns["Report"]
            env.targets[0].confirmed = True
            env.report_buffers[0].clear()
            env.report_buffers[0].append(
                Report(
                    target_id=0,
                    source_uav=0,
                    created_step=0,
                    size_bytes=int(CONFIG["report_bytes"]),
                    ttl_s=float(CONFIG["report_ttl"]),
                    delivered_bytes=0,
                )
            )
        rng = np.random.default_rng(int(seed) + 99173)
        start = time.perf_counter()
        completed = 0
        for _ in range(int(steps)):
            action = {}
            for sender, agent_id in enumerate(env.agent_ids):
                continuous = rng.uniform(
                    -1.0, 1.0, size=4
                ).astype(np.float32)
                discrete_dim = int(
                    env.action_space[
                        agent_id
                    ]["destination"].n
                )
                destination = int(
                    rng.integers(0, discrete_dim)
                )
                if communication_loaded and sender == 0:
                    continuous[3] = 1.0
                    destination = discrete_dim - 1
                action[agent_id] = {
                    "motion": continuous[:3],
                    "destination": destination,
                    "power": np.asarray(
                        [continuous[3]],
                        dtype=np.float32,
                    ),
                }
            out = env.step(action)
            completed += 1
            if bool(out[2] or out[3]):
                break
        wall = time.perf_counter() - start
        return {
            "steps": int(completed),
            "wall_seconds": float(wall),
            "network": env.network_backend.metrics(),
        }
    finally:
        env.close()


def benchmark_uavnetsim_cpu_worker_scaling(
    repo: Path,
    *,
    env_counts=(1, 2, 4, 8),
    steps=4,
    seed=44,
    communication_loaded=True,
):
    import multiprocessing as mp

    results = []
    context = mp.get_context("fork")

    def worker_entry(payload, queue):
        try:
            queue.put(
                ("ok", _uavnetsim_cpu_worker(payload))
            )
        except BaseException as exc:
            import traceback
            queue.put(
                (
                    "error",
                    (
                        type(exc).__name__,
                        str(exc),
                        traceback.format_exc(),
                    ),
                )
            )

    for count in env_counts:
        payloads = [
            (
                str(repo),
                int(seed) + index,
                int(steps),
                bool(communication_loaded),
            )
            for index in range(int(count))
        ]
        queue = context.Queue()
        processes = [
            context.Process(
                target=worker_entry,
                args=(payload, queue),
            )
            for payload in payloads
        ]
        start = time.perf_counter()
        for process in processes:
            process.start()
        messages = [
            queue.get()
            for _ in processes
        ]
        for process in processes:
            process.join()
        wall = time.perf_counter() - start
        errors = [
            payload
            for status, payload in messages
            if status != "ok"
        ]
        if errors:
            raise RuntimeError(
                "UavNetSim worker failed: "
                + repr(errors[0])
            )
        worker_results = [
            payload
            for status, payload in messages
            if status == "ok"
        ]
        transitions = sum(
            int(item["steps"])
            for item in worker_results
        )
        results.append(
            {
                "num_envs": int(count),
                "requested_steps_per_env": int(steps),
                "transitions": int(transitions),
                "wall_seconds": float(wall),
                "transitions_per_second": (
                    transitions / wall
                    if wall > 0.0
                    else float("inf")
                ),
                "worker_wall_seconds": [
                    float(item["wall_seconds"])
                    for item in worker_results
                ],
                "network_metrics": [
                    item["network"]
                    for item in worker_results
                ],
            }
        )
    return results


def _scalarize_metrics(metrics):
    if metrics is None:
        return None
    result = {}
    for key, value in metrics.items():
        if isinstance(value, torch.Tensor):
            if value.numel() == 1:
                result[str(key)] = float(
                    value.detach().cpu().item()
                )
        elif isinstance(value, (int, float, bool)):
            result[str(key)] = value
    return result


def _trainer_parameter_fingerprint(trainer):
    """SHA256 over all nn.Module parameters/buffers, independent of device."""
    import hashlib

    digest = hashlib.sha256()
    module_names = sorted(
        name
        for name, value in trainer.__dict__.items()
        if isinstance(value, torch.nn.Module)
    )
    for module_name in module_names:
        module = getattr(trainer, module_name)
        digest.update(module_name.encode("utf-8"))
        for key, tensor in sorted(
            module.state_dict().items()
        ):
            digest.update(key.encode("utf-8"))
            value = (
                tensor.detach()
                .cpu()
                .contiguous()
            )
            digest.update(
                str(value.dtype).encode("utf-8")
            )
            digest.update(
                str(tuple(value.shape)).encode(
                    "utf-8"
                )
            )
            digest.update(value.numpy().tobytes())
    return digest.hexdigest()


def benchmark_uavnetsim_training_compare(
    repo: Path,
    *,
    repo_path,
    algorithm="masac",
    total_steps=8,
    seed=44,
    device="cuda:0",
    benchmark_learning_starts=4,
    benchmark_batch_size=4,
    benchmark_updates_per_step=1,
):
    """End-to-end CPU-mission vs GPU-mission training with UavNetSim on both.

    The benchmark overrides learning-start/batch settings only inside this
    function so a short run actually exercises replay sampling and gradients.
    Production CONFIG values are restored before returning.
    """
    algorithm = str(algorithm).strip().lower()
    if algorithm not in {"masac", "matd3"}:
        raise ValueError(
            "algorithm must be masac or matd3"
        )
    if not torch.cuda.is_available():
        raise RuntimeError(
            "training comparison requires CUDA"
        )
    total_steps = int(total_steps)
    if total_steps < 1:
        raise ValueError(
            "total_steps must be >= 1"
        )

    project_ns = load_project_namespace(repo)
    if not project_ns["uavnetsim_available"](
        repo_path
    ):
        raise FileNotFoundError(
            "UavNetSim repository is unavailable"
        )

    prefix = algorithm
    keys = {
        "learning": (
            f"{prefix}_learning_starts"
        ),
        "batch": f"{prefix}_batch_size",
        "updates": (
            f"{prefix}_updates_per_step"
        ),
    }
    previous = {
        key: CONFIG[name]
        for key, name in keys.items()
    }
    CONFIG[keys["learning"]] = int(
        benchmark_learning_starts
    )
    CONFIG[keys["batch"]] = int(
        benchmark_batch_size
    )
    CONFIG[keys["updates"]] = int(
        benchmark_updates_per_step
    )

    cpu_env = None
    try:
        CPUEnv = project_ns["UAVSearchEnv"]
        cpu_env = CPUEnv(
            backend_name="uavnetsim",
            network_backend_kwargs={
                "repo_path": repo_path,
            },
        )
        project_ns["set_seed"](int(seed))
        if algorithm == "masac":
            cpu_trainer = project_ns[
                "HybridMASAC"
            ](
                cpu_env,
                device=device,
            )
            train_fn = project_ns[
                "train_masac"
            ]
        else:
            cpu_trainer = project_ns[
                "HybridMATD3"
            ](
                cpu_env,
                device=device,
                seed=int(seed),
            )
            train_fn = project_ns[
                "train_matd3"
            ]

        torch.cuda.synchronize()
        cpu_start = time.perf_counter()
        cpu_result = train_fn(
            cpu_env,
            total_steps=total_steps,
            seed=int(seed),
            trainer=cpu_trainer,
        )
        torch.cuda.synchronize()
        cpu_wall = (
            time.perf_counter()
            - cpu_start
        )
        cpu_network = (
            cpu_env.network_backend.metrics()
        )

        torch.cuda.synchronize()
        gpu_result = train_full_gpu(
            repo=repo,
            algorithm=algorithm,
            num_envs=1,
            total_transitions=total_steps,
            seed=int(seed),
            device=device,
            network_backend="uavnetsim",
            uavnetsim_repo_path=repo_path,
        )

        return {
            "kind": (
                "uavnetsim_training_compare"
            ),
            "algorithm": algorithm,
            "seed": int(seed),
            "total_steps": total_steps,
            "benchmark_overrides": {
                keys["learning"]: int(
                    benchmark_learning_starts
                ),
                keys["batch"]: int(
                    benchmark_batch_size
                ),
                keys["updates"]: int(
                    benchmark_updates_per_step
                ),
            },
            "cpu_mission_gpu_learner": {
                "wall_seconds": cpu_wall,
                "steps_per_second": (
                    total_steps / cpu_wall
                ),
                "completed_episodes": len(
                    cpu_result[
                        "completed_episodes"
                    ]
                ),
                "completed_episode_metrics": (
                    cpu_result[
                        "completed_episodes"
                    ]
                ),
                "latest_update_metrics": (
                    _scalarize_metrics(
                        cpu_result[
                            "latest_update_metrics"
                        ]
                    )
                ),
                "model_fingerprint_sha256": (
                    _trainer_parameter_fingerprint(
                        cpu_result["trainer"]
                    )
                ),
                "network_metrics": (
                    cpu_network
                ),
            },
            "gpu_mission_uavnetsim": (
                gpu_result
            ),
        }
    finally:
        if cpu_env is not None:
            cpu_env.close()
        for key, name in keys.items():
            CONFIG[name] = previous[key]



## Usage

CUDA Simple benchmark: benchmark_env(num_envs=64, steps=100, seed=CONFIG["seed"], device="cuda:0")

Simple full-GPU training: train_full_gpu(repo=Path.cwd(), algorithm="masac", num_envs=64, total_transitions=10000, seed=CONFIG["seed"], device="cuda:0")

Packet-level UavNetSim training (mission math on GPU, UavNetSim on CPU/SimPy): train_full_gpu(repo=Path.cwd(), algorithm="masac", num_envs=1, total_transitions=10000, seed=CONFIG["seed"], device="cuda:0", network_backend="uavnetsim", uavnetsim_repo_path="/path/to/UavNetSim")

CPU-UavNetSim vs GPU-mission+UavNetSim full-flow comparison: benchmark_uavnetsim_full_flow(Path.cwd(), repo_path="/path/to/UavNetSim", steps=20, seed=CONFIG["seed"], device="cuda:0")

Network-only packet-level parity check: validate_uavnetsim_network_parity(Path.cwd(), repo_path="/path/to/UavNetSim").

Use algorithm="matd3" for MATD3. CPU debug is allowed for shape/integration checking via strict_cuda=False. UavNetSim is not CUDA-native, so any UavNetSim path necessarily includes a CPU/SimPy networking stage.

End-to-end short training comparison with UavNetSim on both paths: benchmark_uavnetsim_training_compare(Path.cwd(), repo_path="/path/to/UavNetSim", algorithm="masac", total_steps=8, device="cuda:0").

Scaling benchmark:
benchmark_simple_scaling(Path.cwd(), cpu_env_counts=(1,2,4,8,16), gpu_env_counts=(1,2,4,8,16,32,64,128,256), vector_steps=40, warmup_steps=4, seed=CONFIG["seed"], device="cuda:0")

Parallel packet-level CPU scaling:
benchmark_uavnetsim_cpu_worker_scaling(Path.cwd(), env_counts=(1,2,4,8), steps=4, seed=CONFIG["seed"])

benchmark_uavnetsim_full_flow now reports GPU-to-CPU sync time, UavNetSim step time, CPU-to-GPU sync time, and the UavNetSim fraction.


### GPU network modes

CONFIG["full_gpu_network_model"] = "simple" keeps the original tensor link model.

CONFIG["full_gpu_network_model"] = "packet_approx" keeps the mission/network state fully tensorized on CUDA and additionally models the UavNetSim-relevant packet payload/queue limit, 802.11b-style CSMA/CA airtime overhead, shared-channel contention, packet counts, LoS/NLoS, path loss/SNR/rate, report TTL/buffers, peer relay, GCS delivery, and communication energy. This is a calibrated training surrogate, not a bit-for-bit GPU port of SimPy UavNetSim.

Use the real UavNetSimBackend/GpuUavNetSimBridge for packet-level validation (routing/MAC event scheduling, collisions/interference, ACK/retry timing, delay/PDR/throughput).

### Optional learner speed experiments

train_full_gpu(..., fused_adam=True) benchmarks fused CUDA Adam for actor/critic optimizers.

train_full_gpu(..., compile_mode="default") benchmarks torch.compile without forcing the CUDA-graph-heavy reduce-overhead mode.

CONFIG["training_amp_enabled"] = True enables CUDA AMP (autocast + GradScaler) in MASAC/MATD3 learner updates; keep it disabled until same-seed reproducibility and numerical stability pass on the target GPU.

train_full_gpu(..., update_to_data_ratio=0.5) supports UTD experiments. Ratios below 1 change the learning schedule and therefore are hyperparameter experiments, not semantics-preserving code optimizations.

GPU-native packet-aware training:
train_full_gpu(repo=Path.cwd(), algorithm="masac", num_envs=256, total_transitions=10000, seed=CONFIG["seed"], device="cuda:0", network_backend="packet_approx")

This packet_approx path is CUDA-batched and mirrors the CURRENT project UavNetSim configuration (forced MARL next hop, 1024-byte payloads, queue cap, IEEE-802.11b CSMA/CA airtime/backoff/ACK cost, shared contention, LoS/NLoS A2A channel, relay/GCS delivery and communication energy). It is a training surrogate, not the entire upstream UavNetSim protocol library.

GPU-native UavNetSim surrogate:
`network_backend="uavnetsim_gpu"` keeps the current mission networking on CUDA: UavNetSim-v2 A2A LoS/NLoS path gain, simultaneous-link SINR/interference, packet queue cap, expected CSMA/CA backoff + ACK/SIFS/DIFS airtime, relay custody, GCS delivery, report TTL/buffers and communication energy. It is intentionally a deterministic time-slotted training surrogate; the official SimPy UavNetSim backend remains the packet-event validator.

Optimization sweep:
`benchmark_full_gpu_training_optimizations(Path.cwd(), algorithms=("masac","matd3"), env_counts=(256,512,1024), total_transitions=8192, network_backend="uavnetsim_gpu")` compares baseline, fused Adam, torch.compile, AMP, and combined paths without changing production defaults.

Profiler:
`profile_full_gpu_step(Path.cwd(), algorithm="masac", num_envs=256, network_backend="uavnetsim_gpu", output_trace="/tmp/uav_trace.json")` profiles actor + environment/network CUDA hotspots.

Direct CPU-UavNetSim vs GPU-surrogate validation: `validate_uavnetsim_gpu_direct_link(Path.cwd(), seed=CONFIG["seed"], steps=6, device="cuda:0", strict_cuda=True)`.

Network-fidelity scaling: `benchmark_gpu_network_models(network_models=("packet_approx","uavnetsim_gpu"), env_counts=(256,512,1024), steps=40, warmup_steps=4, seed=CONFIG["seed"], device="cuda:0")`.

Automatic GPU selection:
`train_full_gpu_auto(Path.cwd(), algorithm="masac", num_envs=2048, total_transitions=..., seed=CONFIG["seed"], network_backend="uavnetsim_gpu")` uses `cuda:0` only when one GPU is visible, and shards environments across `cuda:0` + `cuda:1` when two or more GPUs are visible. The cap is controlled by `full_gpu_max_gpus` and is currently 2. The learner/backward stays on `cuda:0`; GPU1 executes its environment/network shard in parallel.

Production automatic training:
train_full_gpu_auto(...) is the single entry point. It uses one GPU when only one is visible and switches to 2-process NCCL DDP when two or more GPUs are visible. Only rank 0 creates one W&B run, whose display name is exactly <ALGORITHM>-seed<SEED>-<RUN_ID>.
